# **🎯 INTEGRATED COX REGRESSION PIPELINE**
## **Purpose**: Complete pipeline from raw snapshot data to Cox regression analysis with OER performance metrics
### **Pipeline Structure**:
- **PHASE 1**: Data Loading & Basic Preparation
- **PHASE 2**: OER Integration & Performance Metrics
- **PHASE 3**: Variable Creation (Time-Varying & Static)
- **PHASE 4**: Advanced Filtering
- **PHASE 5**: Cox Analysis Preparation & Execution
### **Key Features**:
- Progressive data saves at each stage
- Boolean cell flags for selective execution
- Standard naming pattern: `df_pipeline_XX_stage`
- Handles multiple officer appointments for year group calculation
- Real OER data with accurate rating pools (no inference needed)
---


# **🔧 CELL 0: LIBRARY IMPORTS & GLOBAL SETTINGS**
##### **Environment Setup & Database Connection**
- **Library Imports**: All required packages for analysis
- **Global Settings**: Database connection and configuration
- **Utility Functions**: Helper functions for data processing
- **Environment Variables**: Paths, settings, and constants
##### **Focus**: Initialize the analysis environment


In [1]:
# === CELL 0: LIBRARY IMPORTS & GLOBAL SETTINGS ===
# Imports all necessary libraries and sets up the analysis environment
# Configures plotting styles and database connections
# === 0.1. LIBRARY IMPORTS ===
from functionsG import *
# Import necessary libraries
import modin.pandas as pd
import ray
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from typing import Optional, List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

# Import scikit-survival for competing risks analysis
from sksurv.nonparametric import kaplan_meier_estimator, nelson_aalen_estimator
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

# Import plot helper functions
from cox_plot_helpers import *
import sys
import importlib  # For reloading modules
from IPython.display import IFrame, Image
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

# === 0.2. DATABASE CONNECTION ===
# === Ray/Modin Environment Variables ===
from init_ray_cluster_working import init_ray_cluster
# init_ray_cluster()
# Clean out the tmp folder
!bash ./ray_tmp_clean_working.sh <<< 'y'

# === 0.3. PLOTTING CONFIGURATION ===
# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

# === 0.3.1. PANDAS DISPLAY OPTIONS ===
# Set pandas display options for better visibility
pd.options.display.max_rows = 100  # Display up to 100 rows (default is 60)
pd.options.display.max_columns = None  # Display all columns
pd.options.display.width = None  # Auto-detect terminal width
pd.options.display.max_colwidth = 50  # Limit column width for readability

# === 0.4. IMPORT PIPELINE CONFIGURATION ===
# Import all configuration settings from pipeline_config.py (or an override)
# This includes: cell execution flags, global settings, analysis config, 
# pool settings, filtering params, column config, and plot config
# Set pip_config_file to override which config module is used
pip_config_file = 'pipeline_config_div_name'
# Example overrides: 'pipeline_config_15_1', 'pipeline_config_16_1', 'pipeline_config_div_name'
# Note: module name only (no .py extension)
module_name = pip_config_file[:-3] if pip_config_file.endswith('.py') else pip_config_file
print(module_name)
# Always reload base config first
base_config = importlib.import_module('pipeline_config')
importlib.reload(base_config)

# If using an override, reload it on top of base
if module_name != 'pipeline_config':
    pipeline_config = importlib.import_module(module_name)
    importlib.reload(pipeline_config)
else:
    pipeline_config = base_config
    
# Re-import all symbols from the selected module
globals().update({k: v for k, v in pipeline_config.__dict__.items() if not k.startswith('_')})
# === 0.4.0. OPTIONAL: RUN 503 PREP ===
# If enabled, (re)build prestige UIC lists via py_503_hierarchies.py
if RUN_503:
    print_v("\n🔧 RUN_503 enabled: building prestige UIC lists...")
    try:
        import py_503_hierarchies as hier
        hier.get_prestige_uics_by_fy(save=True, var_dir=var_dir)
        print_v(f"✅ Saved prestige_uics_by_fy (with backfill if enabled)")
    except Exception as e:
        print(f"⚠️ RUN_503 failed: {e}")
        
# === 0.4.1. RELOAD PIPELINE CONFIGURATION FUNCTION ===
# Helper function to reload pipeline_config.py without restarting the kernel
# Call this at the beginning of any cell to pick up changes made to pipeline_config.py
def reload_pipeline_config():
    """
    to pick up any changes made to these files.
    
    This allows you to edit these files and have changes take effect immediately
    without having to restart the Jupyter kernel and rerun all previous cells.
    
    Usage:
    ------
        # At the beginning of any cell:
        reload_pipeline_config()
        
        # Now all configuration variables (CELL*, PLOT_CONFIG, etc.) reflect the latest changes
        # And all functions from the reloaded modules are updated
    """
    # Reload pipeline_config (or override moule)
    module_name = globals().get('pip_config_file', 'pipeline_config')
    module_name = module_name[:-3] if module_name.endswith('.py') else module_name

    # Always reload base config first
    base_config = importlib.import_module('pipeline_config')
    importlib.reload(base_config)
    
    # If using an override, reload it on top of base
    if module_name != 'pipeline_config':
        pipeline_config = importlib.import_module(module_name)
        importlib.reload(pipeline_config)
    else:
        pipeline_config = base_config
    
    # Re-import all symbols from the reloaded module
    # This updates all the variables in the current namespace
    globals().update({k: v for k, v in pipeline_config.__dict__.items() 
                      if not k.startswith('_')})
    globals()['pip_config_file'] = module_name
    
    # Reload add_cum_oer_metrics_mod_working
    try:
        import add_cum_oer_metrics_mod_working
        importlib.reload(add_cum_oer_metrics_mod_working)
    except ImportError:
        pass  # Module not imported yet, will be imported when needed
     
    # Reload cox_plot_helpers
    try:
        import cox_plot_helpers
        importlib.reload(cox_plot_helpers)
    except ImportError:
        pass  # Module not imported yet, will be imported when needed
    
    # Reload NATO_lvl
    try:
        import NATO_lvl
        importlib.reload(NATO_lvl)
    except ImportError:
        pass  # Module not imported yet, will be imported when needed
        
    # Reload py_503_hierarchies
    try:
        import py_503_hierarchies
        importlib.reload(py_503_hierarchies)
    except ImportError:
        pass  # Module not imported yet, will be imported when needed
        
    print_v(f"✅ Reloaded modules - all configuration settings updated")
    print_v(f"   • {module_name}.py")
    print_v(f"   • add_cum_oer_metrics_mod_working.py")
    print_v(f"   • cox_plot_helpers.py")
    print_v(f"   • NATO_lvl.py")
    print_v(f"   • py_503_hierarchies.py")
    print_v(f"\n   • Current CELL5_POOL_MEANS:  {CELL5_POOL_MEANS}")
    print_v(f"   • Current CELL6_POOL_RANKS:  {CELL6_POOL_RANKS}")
    print_v(f"   • Current CELL10:  {CELL10}")
    print_v(f"   • Current CELL10_5:  {CELL10_5}")
    print_v(f"   • Current CELL11:  {CELL11}")
    print_v(f"   • Current CELL12:  {CELL12}")
    # print_v(f"   • Filtering Parameters:  {filtering_params}")
    print_v(f"   • STANDARDIZE_CONFIG['cols']:  {STANDARDIZE_CONFIG['cols']}")
    print_v(f"   • COLUMN_CONFIG['model_static_cols']:  {COLUMN_CONFIG['model_static_cols']}")
    print_v(f"   • COLUMN_CONFIG['model_time_varying_cols']:  {COLUMN_CONFIG['model_time_varying_cols']}")
    print_v(f"   • filtering_params['filter_job_codes']['include_specific']:  {filtering_params['filter_job_codes']['include_specific']}")

# Note: The following are imported from pipeline_config.py:
# - All CELL* execution flags (CELL0, CELL1, CELL2, ..., CELL12_8)
# - Global settings (var_dir, uic_dir, verbose, null_reports, cox_var_dir)
# - Analysis configuration (SOURCE_RANK, TARGET_RANK)
# - Pool settings (POOL_MIN_SIZE, POOL_EXCLUDE_SELF)
# - Filtering parameters (filtering_params)
# - Column configuration (COLUMN_CONFIG)
# - Plot configuration (PLOT_CONFIG)
# - Helper function (print_v)

# Additional local variable (not in config file)
nest = 2  # Nesting level for time_start/time_stop functions

# === 0.5. NOTEBOOK CELL LOGGING FUNCTION ===
# Context manager for logging all output from notebook cells to files
# Useful when AWS AppStream session times out - you can check the log files to see progress
# Note: This function uses CELL6_ENABLE_LOGGING and CELL6_LOG_DIR from pipeline_config
import os
import sys
from contextlib import contextmanager
from datetime import datetime
@contextmanager
def notebook_cell_logging(cell_name):
    """
    Context manager to log all output from a notebook cell to both .log and .md files.
    
    Logs all stdout/stderr output to:
    - Full log file: cell_{cell_name}_{timestamp}.log (complete output)
    - Progress summary: cell_{cell_name}_progress.md (markdown summary with timestamps)
    
    Useful when AWS AppStream session times out - you can check the log files to see progress.
    Logging can be enabled/disabled via CELL6_ENABLE_LOGGING in pipeline_config.py.
    
    Parameters:
    -----------
    cell_name : str
        Name/identifier for the cell (e.g., '1', '6', '11', '12')
    
    Usage:
    ------
        with notebook_cell_logging('4'):
            # All print statements here will be logged
            print_v("Starting computation...")
            # Cell code here
    """
    if not CELL6_ENABLE_LOGGING:
        # Logging disabled - just pass through
        yield
        return
    
    # Create log directory if it doesn't exist
    os.makedirs(CELL6_LOG_DIR, exist_ok=True)
    
    # Create log file names with timestamp
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_file = os.path.join(CELL6_LOG_DIR, f'cell_{cell_name}_{timestamp}.log')
    md_file = os.path.join(CELL6_LOG_DIR, f'cell_{cell_name}_progress.md')
    
    # Open log files
    log_fp = open(log_file, 'w', encoding='utf-8')
    md_fp = open(md_file, 'w', encoding='utf-8')
    
    # Write header to markdown file
    md_fp.write(f"# CELL {cell_name} Progress Log\n\n")
    md_fp.write(f"**Started**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    md_fp.write("---\n\n")
    md_fp.flush()
    
    # Announce logging is active (print to console and write to log files)
    log_dir_abs = os.path.abspath(CELL6_LOG_DIR)
    announcement = f"\n📝 Logging enabled for CELL {cell_name}:\n"
    announcement += f"   • Log directory: {log_dir_abs}\n"
    announcement += f"   • Full log: {os.path.abspath(log_file)}\n"
    announcement += f"   • Progress summary: {os.path.abspath(md_file)}\n"
    announcement += f"   • All output is being written in real-time\n"
    
    # Print to console (before redirecting stdout)
    print(announcement)
    
    # Also write to log files
    log_fp.write(announcement)
    md_fp.write(announcement)
    log_fp.flush()
    md_fp.flush()
        
    # Save original stdout/stderr
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    
    class TeeOutput:
        """Class that writes to both console and log files"""
        def __init__(self, *files):
            self.files = files
        
        def write(self, text):
            # Write to all files (console + log + md)
            for f in self.files:
                f.write(text)
                f.flush()  # Immediate flush so output is visible even if session times out
        
        def flush(self):
            for f in self.files:
                f.flush()
    
        def isatty(self):
            # Delegate to original stdout's isatty() method (needed by colorful library used by Ray)
            # The first file in self.files is the original stdout/stderr
            if self.files and hasattr(self.files[0], 'isatty'):
                return self.files[0].isatty()
            return False
    # Create tee output that writes to console, log, and md
    tee_stdout = TeeOutput(original_stdout, log_fp, md_fp)
    tee_stderr = TeeOutput(original_stderr, log_fp, md_fp)
    
    # Track if there was an exception
    exception_occurred = False
    exception_info = None
       
    try:
        # Redirect stdout and stderr
        sys.stdout = tee_stdout
        sys.stderr = tee_stderr
        
        yield
        
    except Exception as e:
        # Mark that an exception occurred
        exception_occurred = True
        exception_info = str(e)
        # Re-raise so the exception propagates normally
        raise
      
        
    finally:
        # Restore original stdout/stderr
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        
        # Write footer to markdown file (mark as ERROR if exception occurred)
        try:
            if exception_occurred:
                md_fp.write(f"\n---\n\n**⚠️ ERROR/CRASH**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                if exception_info:
                    md_fp.write(f"**Exception**: {exception_info}\n")
                md_fp.write("\n**Note**: Check the full log file for complete error traceback.\n")
            else:
                md_fp.write(f"\n---\n\n**✅ Completed**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            md_fp.flush()
        except Exception as e:
            # If writing footer fails, at least try to log it
            print(f"⚠️ Warning: Could not write footer to markdown file: {e}")
        
        # Close files (with error handling to ensure they're always closed)
        try:
            log_fp.close()
        except Exception as e:
            print(f"⚠️ Warning: Error closing log file: {e}")
        
        try:
            md_fp.close()
        except Exception as e:
            print(f"⚠️ Warning: Error closing markdown file: {e}")
        
        # Print summary
        try:
            if exception_occurred:
                print(f"\n📝 Logging saved for CELL {cell_name} (ERROR occurred):")
            else:
                print(f"\n📝 Logging complete for CELL {cell_name}:")
            print(f"   - Full log: {os.path.abspath(log_file)}")
            print(f"   - Progress summary: {os.path.abspath(md_file)}")
        except Exception as e:
            # If printing summary fails, at least try to show basic info
            print(f"\n📝 Logging files saved in: {log_dir_abs}")
            
# if CELL0:
#     lu_table_dict = load_lu_tables(0)
def df_null_report(df_in, below_percent=50):
    df = df_in.copy(); cols = df.columns.tolist()
    data_dict = {'Column':[],'Unique':[],'Null Values':[],'Percent Non-null':[]}
    for col in cols:
        null_count = df[col].isnull().sum()
        unique_count = df[col].nunique()
        percent_non_null = (df[col].notnull().sum() / len(df)) * 100
        data_dict['Column'].append(col)
        data_dict['Unique'].append(unique_count)
        data_dict['Null Values'].append(null_count)
        data_dict['Percent Non-null'].append(percent_non_null)
    df_null_report = pd.DataFrame(data_dict)
    below_percent_df = df_null_report[df_null_report['Percent Non-null'] < below_percent]
    head1 = "\033[1m" + '===  DF_NULL_REPORT ===\n' + "\033[0m"
    head2 = "\033[1m" + f"\n\n=== BELOW_{below_percent}_PERCENT_DF ===\n" + "\033[0m"
    print(head1,df_null_report,head2,below_percent_df)


Current size of /tmp/ray:
0	/tmp/ray
Deleting contents of /tmp/ray...
Done.
pipeline_config_div_name
Saving ./running_vars//oer_cols_keep
Saving ./running_vars//snap_cols_clean
Saving ./running_vars//oer_cols_keep
Saving ./running_vars//snap_cols_clean


In [2]:
globals().get('pip_config_file')

'pipeline_config_div_name'

# **📦 CELL 1: LOAD RAW SNAPSHOT DATA**
##### **Load master snapshot data from database or feather file**
- **Source Options**:
  - **502 output** (recommended): `df_502_base.feather` - already filtered, has yg/dor_cpt/dor_maj
  - **Database table**: `master_pde_ref_culld`
  - **Existing feather file**: `df_pipeline_01_raw.feather`
  - **512 output**: `df_master_pde_joined.feather`
- **Output**: `df_pipeline_01_raw` (all snapshot data with all columns)
##### **Focus**: Get raw data ready for initial processing. If loading from 502, filtering and yg/dor calculations are already done.


In [3]:
# === CELL 1: LOAD RAW SNAPSHOT DATA ===
# Load master snapshot data from database or feather file
# Option to load from 502 output (df_502_base.feather) which already has filtering and yg/dor calculations
with notebook_cell_logging('1'):
    
    burrito_act= f" 🌟🌟🌟🌟  Running the WHOLE BIG BURRITO Pipeline...  🌟🌟🌟🌟"
    burrito_timer = time_start(burrito_act,nest=0)
    if CELL1:
        print_v("\n\n=== PHASE 1: DATA LOADING & BASIC PREPARATION ===\n")
        cell1_act = "CELL 1: Loading raw snapshot data"
        cell1 = time_start(cell1_act, nest=0)
        
        # === LOADING OPTIONS ===
        # Option 1: Load from 502 output (recommended - already filtered and has yg/dor_cpt/dor_maj)
        # Option 2: Load from database table 'master_pde_ref_culld'
        # Option 3: Load from existing feather file 'df_pipeline_01_raw'
        # Option 4: Load from 512 output 'df_master_pde_joined'
        
        load_from_502 = True   # Set to True to load from 502 output (recommended)
        load_from_db = False   # Set to True to load from database
        load_from_512 = False  # Set to True to load from 512 output
        
        if load_from_502:
            # Load from 502 output (already has filtering, yg, dor_cpt, dor_maj)
            try:
                load_act = "Loading from 502 output (df_502_base.feather)"
                load_timer = time_start(load_act, nest=1)
                df_raw = load_feather('df_502_base')
                time_stop(load_timer, action=load_act, nest=1)
                print_v(f"✅ Loaded from 502 output: {len(df_raw):,} rows with {df_raw.pid_pde.nunique():,} officers")
                print_v(f"📊 502 output already includes:")
                print_v(f"   • Commissioned officers filtered")
                print_v(f"   • Data quality checks completed")
                if 'yg' in df_raw.columns:
                    print_v(f"   • Year group (yg) calculated")
                if 'dor_cpt' in df_raw.columns:
                    print_v(f"   • Date of rank CPT (dor_cpt) calculated")
                if 'dor_maj' in df_raw.columns:
                    print_v(f"   • Date of rank MAJ (dor_maj) calculated")
            except FileNotFoundError:
                print_v("⚠️ df_502_base.feather not found. Falling back to database...")
                load_from_502 = False
                load_from_db = True
        
        if not load_from_502:
            if load_from_db:
                load_act = "Loading from database table 'master_pde_ref_culld'"
                load_timer = time_start(load_act, nest=1)
                df_raw = table_to_df('master_pde_ref_culld', show=True)
                time_stop(load_timer, action=load_act, nest=1)
                print_v(f"📊 Loaded {len(df_raw):,} rows with {df_raw.pid_pde.nunique():,} officers")
            elif load_from_512:
                try:
                    df_raw = load_feather('df_master_pde_joined')
                    print_v(f"✅ Loaded from 512 output: {len(df_raw):,} rows")
                except FileNotFoundError:
                    print_v("⚠️ df_master_pde_joined.feather not found. Loading from database...")
                    load_act = "Loading from database table 'master_pde_ref_culld'"
                    load_timer = time_start(load_act, nest=1)
                    df_raw = table_to_df('master_pde_ref_culld', show=True)
                    time_stop(load_timer, action=load_act, nest=1)
            else:
                # Try to load from existing feather file
                try:
                    df_raw = load_feather('df_pipeline_01_raw')
                    print_v(f"✅ Loaded from feather file: {len(df_raw):,} rows")
                except FileNotFoundError:
                    print_v("⚠️ Feather file not found, loading from database...")
                    load_act = "Loading from database table 'master_pde_ref_culld'"
                    load_timer = time_start(load_act, nest=1)
                    df_raw = table_to_df('master_pde_ref_culld', show=True)
                    time_stop(load_timer, action=load_act, nest=1)
        
        # Save raw data (for consistency, even if from 502)
        save_act = "Saving df_pipeline_01_raw"
        save_timer = time_start(save_act, nest=1)
        store_feather(df_raw, 'df_pipeline_01_raw')
        time_stop(save_timer, action=save_act, nest=1)
        print_v(f"📊 Date range: {df_raw.snpsht_dt.min()} to {df_raw.snpsht_dt.max()}")
        
        time_stop(cell1, action=cell1_act, nest=0)
    else:
        print_v("By-passing CELL 1...")
        if CELL2:
            df_raw = load_feather('df_pipeline_01_raw')
    tyme()



📝 Logging enabled for CELL 1:
   • Log directory: /data/TALENT_NET/home/AN_LEVINEC/Network_1P_shell/cell6_logs
   • Full log: /data/TALENT_NET/home/AN_LEVINEC/Network_1P_shell/cell6_logs/cell_1_20260325_080209.log
   • Progress summary: /data/TALENT_NET/home/AN_LEVINEC/Network_1P_shell/cell6_logs/cell_1_progress.md
   • All output is being written in real-time

Start  🌟🌟🌟🌟  Running the WHOLE BIG BURRITO Pipeline...  🌟🌟🌟🌟 at 11:02:09 (EST) Wed, 25 Mar 2026
By-passing CELL 1...

📝 Logging complete for CELL 1:
   - Full log: /data/TALENT_NET/home/AN_LEVINEC/Network_1P_shell/cell6_logs/cell_1_20260325_080209.log
   - Progress summary: /data/TALENT_NET/home/AN_LEVINEC/Network_1P_shell/cell6_logs/cell_1_progress.md


In [4]:
if null_reports:
    df_null_report(df_raw)


# **🔍 CELL 2: BASIC FILTERING**
##### **Filter to commissioned officers and remove irregular MOS**
- **Filter**: Commissioned officers only (remove cadets, enlisted)
- **Remove**: Officers with irregular MOS/job codes (MC, DC, VC, SP, AN, JA, CH, MS)
- **Output**: `df_pipeline_02_base` (clean base dataset)
##### **Focus**: Create clean base dataset for analysis


In [5]:
# === CELL 2: BASIC FILTERING ===
# Filter to commissioned officers and remove irregular MOS
# NOTE: If loading from 502 output, this filtering is already done - cell will skip
# Reload pipeline_config to pick up any changes made 
# to configuration and other .py scripts
reload_pipeline_config()
with notebook_cell_logging('2'):
    if CELL2:
        print_v("\n🔍 CELL 2: Applying basic filtering...")
        cell2_act = "CELL 2: Basic filtering"
        cell2 = time_start(cell2_act, nest=0)
        
        # Load raw data if not already loaded
        if 'df_raw' not in locals():
            df_raw = load_feather('df_pipeline_01_raw')
        
        # Filter currently unnecessary columns
        if CLEAN_SNAP_COLS:
            snap_cols_clean = load_json('snap_cols_clean', var_dir)
            df_raw = df_raw[[col for col in df_raw.columns if col not in snap_cols_clean]]
            print(f"\n   • Cleaned snapshot DF (df_pipeline_01_raw) columns using {var_dir}/snap_cols_clean.json")     
            print(f"   • Columns cleaned: {snap_cols_clean}\n")     
                
        # Check if data already has yg column (indicates it came from 502)
        if 'yg' in df_raw.columns and 'dor_cpt' in df_raw.columns:
            print_v("✅ Data appears to be from 502 output (has yg and dor_cpt columns)")
            print_v("⏭️ Skipping filtering - already done in 502")
            df_base = df_raw.copy()
        else:
            # Perform filtering (original logic)
            df_base = df_raw.copy()
            initial_count = len(df_base)
            initial_officers = df_base.pid_pde.nunique()
            print_v(f"📊 Initial row count: {initial_count:,}")
            print_v(f"📊 Initial officers: {initial_officers:,}")  
            
        # Filter to commissioned officers only
            filter_act = "Filtering to commissioned officers"
            filter_timer = time_start(filter_act, nest=1)
            commissioned_ranks = ['CPT', 'MAJ', 'LTC', 'COL', 'BG', 'MG', 'LTG', 'GEN', 'O03', 'O04', 'O05', 'O06', 'O07', 'O08', 'O09', 'O10']
            df_base = df_base[df_base['rank_pde'].isin(commissioned_ranks)].copy()
            time_stop(filter_timer, action=filter_act, nest=1)
            after_rank_count = len(df_base)
            after_rank_officers = df_base.pid_pde.nunique()
            removed_rows = initial_count - after_rank_count
            removed_officers = initial_officers - after_rank_officers
            print_v(f"✅ After commissioned officer filter: {after_rank_count:,} rows ({after_rank_officers:,} officers)")
            print_v(f"   • Removed: {removed_rows:,} rows ({removed_rows/initial_count*100:.2f}% of rows)")
            print_v(f"   • Removed: {removed_officers:,} officers ({removed_officers/initial_officers*100:.2f}% of officers)")
            
            # Remove officers with irregular MOS/job codes
            problematic_job_codes = ['MC', 'DC', 'VC', 'SP', 'AN', 'JA', 'CH', 'MS']
            print_v(f"🔍 Removing officers with problematic job codes: {problematic_job_codes}")
            
            if 'occ_crer_grp_cd' in df_base.columns:
                remove_act = "Removing officers with problematic job codes"
                remove_timer = time_start(remove_act, nest=1)
                problematic_officers = df_base[df_base['occ_crer_grp_cd'].isin(problematic_job_codes)]['pid_pde'].unique()
                print_v(f"📊 Found {len(problematic_officers):,} officers with problematic job codes")
                df_base = df_base[~df_base['pid_pde'].isin(problematic_officers)].copy()
                time_stop(remove_timer, action=remove_act, nest=1)
                after_job_count = len(df_base)
                after_job_officers = df_base.pid_pde.nunique()
                removed_job_rows = after_rank_count - after_job_count
                removed_job_officers = len(problematic_officers)
                print_v(f"✅ After removing problematic job codes: {after_job_count:,} rows ({after_job_officers:,} officers)")
                print_v(f"   • Removed: {removed_job_rows:,} rows ({removed_job_rows/after_rank_count*100:.2f}% of remaining rows)")
                print_v(f"   • Removed: {removed_job_officers:,} officers ({removed_job_officers/after_rank_officers*100:.2f}% of remaining officers)")
            else:
                print_v("⚠️ Column 'occ_crer_grp_cd' not found, skipping job code filter")
                after_job_count = after_rank_count
                after_job_officers = after_rank_officers
            
            # Summary
            final_count = len(df_base)
            final_officers = df_base.pid_pde.nunique()
            total_removed_rows = initial_count - final_count
            total_removed_officers = initial_officers - final_officers
            print_v(f"\n📊 Filtering Summary:")
            print_v(f"  Initial: {initial_count:,} rows ({initial_officers:,} officers)")
            print_v(f"  Final: {final_count:,} rows ({final_officers:,} officers)")
            print_v(f"  Total Removed: {total_removed_rows:,} rows ({total_removed_rows/initial_count*100:.2f}% of initial rows)")
            print_v(f"  Total Removed: {total_removed_officers:,} officers ({total_removed_officers/initial_officers*100:.2f}% of initial officers)")
        
        # Save filtered base data
        save_act = "Saving df_pipeline_02_base"
        save_timer = time_start(save_act, nest=1)
        store_feather(df_base, 'df_pipeline_02_base')
        time_stop(save_timer, action=save_act, nest=1)
        
        time_stop(cell2, action=cell2_act, nest=0)
    else:
        print_v("By-passing CELL 2...")
        if CELL3:
            df_base = load_feather('df_pipeline_02_base')
        
    if null_reports:
        df_null_report(df_base)
    
    tyme()


Saving ./running_vars//oer_cols_keep
Saving ./running_vars//snap_cols_clean
✅ Reloaded modules - all configuration settings updated
   • pipeline_config_div_name.py
   • add_cum_oer_metrics_mod_working.py
   • cox_plot_helpers.py
   • NATO_lvl.py
   • py_503_hierarchies.py

   • Current CELL5_POOL_MEANS:  False
   • Current CELL6_POOL_RANKS:  False
   • Current CELL10:  True
   • Current CELL10_5:  True
   • Current CELL11:  True
   • Current CELL12:  True
   • STANDARDIZE_CONFIG['cols']:  ['tb_ratio_fwd_snr', 'pool_minus_mean_snr_fwd']
   • COLUMN_CONFIG['model_static_cols']:  []
   • COLUMN_CONFIG['model_time_varying_cols']:  ['z_tb_ratio_fwd_snr', 'z_pool_minus_mean_snr_fwd', 'z_tb_ratio_fwd_snr_sq', 'z_pool_minus_mean_snr_fwd_sq', 'star_pool_interaction', 'div_name']
   • filtering_params['filter_job_codes']['include_specific']:  ['IN', 'AR', 'EN', 'AV', 'FA', 'SF', 'AD', 'CM', 'CA', 'MI', 'MP', 'SC', 'AG', 'FI', 'OD', 'QM', 'TC', 'LG']

📝 Logging enabled for CELL 2:
   • Log direc

# **📅 CELL 3: CALCULATE YEAR GROUP (yg)**
##### **Calculate year group from ofcr_apnt_dt using fiscal year**
- **Source**: `ofcr_apnt_dt` column
- **Method**: Fiscal year calculation (get_fy function)
- **Multiple Appointments**: Exclude officers with multiple distinct yg values
- **Output**: `df_pipeline_03_base` with `yg` column added
##### **Focus**: Create year group variable for cohort analysis


In [6]:
# === CELL 3: CALCULATE YEAR GROUP (yg) ===
# Calculate year group from ofcr_apnt_dt using fiscal year
# NOTE: If loading from 502 output, yg is already calculated - cell will skip
with notebook_cell_logging('3'):
    if CELL3:
        print_v("\n📅 CELL 3: Calculating year group (yg)...")
        cell3_act = "CELL 3: Calculate year group"
        cell3 = time_start(cell3_act, nest=0)
        
        # Load base data if not already loaded
        if 'df_base' not in locals():
            df_base = load_feather('df_pipeline_02_base')
        
        # Import 501-style year group builder (thorough handling of duplicates)
        from build_yg_dict_501 import build_yg_dict_501, apply_yg_dict
        
        # Check if yg already exists (from 502)
        if 'yg' in df_base.columns:
            print_v("✅ Year group (yg) column already exists (likely from 502 output)")
            print_v("⏭️ Skipping yg calculation - already done in 502")
            df_with_yg = df_base.copy()
            # Ensure yg is integer type (convert from float if necessary)
            if df_with_yg['yg'].dtype != 'int64' and df_with_yg['yg'].dtype != 'Int64':
                convert_act = "Converting existing yg column to integer"
                convert_timer = time_start(convert_act, nest=1)
                # Use Int64 (nullable integer) if there are NaNs, otherwise regular int
                if df_with_yg['yg'].isna().any():
                    df_with_yg['yg'] = df_with_yg['yg'].astype('Int64')
                    print_v("✅ Converted yg to Int64 (nullable integer) to handle NaNs")
                else:
                    df_with_yg['yg'] = df_with_yg['yg'].astype('int64')
                    print_v("✅ Converted yg to int64")
                time_stop(convert_timer, action=convert_act, nest=1)
        else:
            # Perform yg calculation using 501 logic
            df_with_yg = df_base.copy()
            initial_officers = df_with_yg.pid_pde.nunique()
            initial_rows = len(df_with_yg)
            print_v(f"📊 Initial officers: {initial_officers:,}")
            print_v(f"📊 Initial rows: {initial_rows:,}")
            
            # Check if ofcr_apnt_dt exists
            if 'ofcr_apnt_dt' not in df_with_yg.columns:
                print_v("⚠️ ERROR: Column 'ofcr_apnt_dt' not found!")
                print_v("Available columns:", list(df_with_yg.columns)[:20])
            else:
                build_act = "Building yg dictionary (501 logic)"
                build_timer = time_start(build_act, nest=1)
                rank_for_yg = SOURCE_RANK if 'SOURCE_RANK' in globals() else 'CPT'
                yg_dict = build_yg_dict_501(
                    df_with_yg,
                    rank=rank_for_yg,
                    salvage_from_rank='MAJ',
                    compo_filter='R',
                    verbose=True
                )
                time_stop(build_timer, action=build_act, nest=1)
                
                map_act = "Mapping yg to all snapshots (501 logic)"
                map_timer = time_start(map_act, nest=1)
                df_with_yg = apply_yg_dict(df_with_yg, yg_dict, dtype='Int64')
                time_stop(map_timer, action=map_act, nest=1)
                
                # Move yg column after ofcr_apnt_dt if both exist
                if 'ofcr_apnt_dt' in df_with_yg.columns:
                    cols = df_with_yg.columns.tolist()
                    if 'yg' in cols and 'ofcr_apnt_dt' in cols:
                        cols.remove('yg')
                        ofcr_idx = cols.index('ofcr_apnt_dt')
                        cols.insert(ofcr_idx + 1, 'yg')
                        df_with_yg = df_with_yg[cols]
                
                # Summary
                final_officers = df_with_yg.pid_pde.nunique()
                final_rows = len(df_with_yg)
                snapshots_with_yg = df_with_yg['yg'].notna().sum()
                snapshots_with_yg_pct = (snapshots_with_yg / len(df_with_yg) * 100) if len(df_with_yg) > 0 else 0
                missing_yg = df_with_yg['yg'].isna().sum()
                print_v(f"\n📊 Year Group Summary (501 logic):")
                print_v(f"  Initial officers: {initial_officers:,}")
                print_v(f"  Final officers: {final_officers:,}")
                print_v(f"  Initial rows: {initial_rows:,}")
                print_v(f"  Final rows: {final_rows:,}")
                print_v(f"  Snapshots with yg: {snapshots_with_yg:,} / {len(df_with_yg):,} snapshots ({snapshots_with_yg_pct:.1f}%)")
                print_v(f"  Snapshots missing yg: {missing_yg:,}")
                print_v(f"  Year groups: {sorted(df_with_yg['yg'].dropna().unique())}")
        
        # Save updated base data
        save_act = "Saving df_pipeline_03_base with yg"
        save_timer = time_start(save_act, nest=1)
        store_feather(df_with_yg, 'df_pipeline_03_base')
        time_stop(save_timer, action=save_act, nest=1)
        df_base = df_with_yg.copy()
        
        time_stop(cell3, action=cell3_act, nest=0)
    else:
        print_v("By-passing CELL 3...")
        if CELL4:
            df_base = load_feather('df_pipeline_03_base')
        
    tyme()



📝 Logging enabled for CELL 3:
   • Log directory: /data/TALENT_NET/home/AN_LEVINEC/Network_1P_shell/cell6_logs
   • Full log: /data/TALENT_NET/home/AN_LEVINEC/Network_1P_shell/cell6_logs/cell_3_20260325_080209.log
   • Progress summary: /data/TALENT_NET/home/AN_LEVINEC/Network_1P_shell/cell6_logs/cell_3_progress.md
   • All output is being written in real-time

By-passing CELL 3...

📝 Logging complete for CELL 3:
   - Full log: /data/TALENT_NET/home/AN_LEVINEC/Network_1P_shell/cell6_logs/cell_3_20260325_080209.log
   - Progress summary: /data/TALENT_NET/home/AN_LEVINEC/Network_1P_shell/cell6_logs/cell_3_progress.md


In [7]:
if null_reports:
    df_null_report(df_base)


# **📊 PHASE 2: OER INTEGRATION & PERFORMANCE METRICS**
## **CELL 4: Merge OER to Snapshots**
## **CELL 5: Pool Means / Minus-Means / Size**
## **CELL 6: Pool Ranks / Percentiles / Z-Scores**


# **🔗 CELL 4: MERGE OER TO SNAPSHOTS**
##### **Build individual forward/backward metrics**
- **Function**: `assign_oer_to_snapshots_fast()`
- **Output**: `df_pipeline_04a_basic_metrics` (individual fwd/bwd metrics)
##### **Focus**: Create the base OER metrics used by downstream pool steps


In [8]:
# del df_base

In [9]:
# === CELL 4: MERGE OER TO SNAPSHOTS ===
# Build individual fwd/bwd OER metrics (no pool metrics here)
# Reload pipeline_config to pick up any changes made
reload_pipeline_config()
with notebook_cell_logging('4'):
    if CELL4:
        print_v("\n🔗 CELL 4: Building individual fwd/bwd OER metrics...")
        cell4_act = "CELL 4: Build individual fwd/bwd metrics"
        cell4 = time_start(cell4_act, nest=nest)
        # Load base data if not already loaded
        if 'df_base' not in locals():
            df_base = load_feather('df_pipeline_03_base')
        # Load OER data if not already loaded
        if 'df_oer_enriched' not in locals():
            try:
                df_oer_enriched = load_feather('df_oer_enriched')
                print_v(f"✅ Loaded df_oer_enriched: {len(df_oer_enriched):,} rows")
                print_v(f"📊 Columns: {len(df_oer_enriched.columns)} columns")
            except FileNotFoundError:
                error_msg = (
                    "\n" + "="*80 + "\n"
                    "❌ ERROR: df_oer_enriched.feather not found!\n\n"
                    "This file must be created by running 512_oer_int_working.ipynb first.\n\n"
                    "REQUIRED STEPS:\n"
                    "  1. Open 512_oer_int_working.ipynb\n"
                    "  2. Run CELL 0 (library imports and setup)\n"
                    "  3. Run CELL 1 (load raw OER data)\n"
                    "  4. Run CELL 2 (clean and prepare OER data)\n"
                    "  5. Run CELL 3 (add rater/snr_rater UICs - creates df_oer_enriched.feather)\n\n"
                    "The 512 notebook performs:\n"
                    "  - Loads and cleans raw OER data (df_oer.feather)\n"
                    "  - Adds rater and senior rater UICs using backward snapshot matching\n"
                    "  - Creates df_oer_enriched.feather (required for 520 CELL 4)\n\n"
                    "After running 512 CELLS 0-3, return to this notebook and run CELL 4 again.\n"
                    "="*80
                )
                print_v(error_msg)
                raise FileNotFoundError(
                    "df_oer_enriched.feather not found. Please run 512_oer_int_working.ipynb CELLS 0-3 first."
                )
        from add_cum_oer_metrics_mod_working import assign_oer_to_snapshots_fast
        merge_act = "OER assignment (forward/backward)"
        merge_timer = time_start(merge_act, nest=nest+1)
        df_with_metrics = assign_oer_to_snapshots_fast(
            df_snaps=df_base,
            df_oers=df_oer_enriched,
            pid_col=CELL6_COLUMN_MAPPING['pid_col'],
            snapshot_date_col=CELL6_COLUMN_MAPPING['snapshot_date_col'],
            rated_officer_col=CELL6_COLUMN_MAPPING['rated_officer_col'],
            eval_strt_col=CELL6_COLUMN_MAPPING['start_date_col'],
            eval_thru_col=CELL6_COLUMN_MAPPING['thru_date_col'],
            rater_col=CELL6_COLUMN_MAPPING['rater_col'],
            snr_rater_col=CELL6_COLUMN_MAPPING['snr_rater_col'],
            rtr_box_col=CELL6_COLUMN_MAPPING['rater_box_col'],
            snr_box_col=CELL6_COLUMN_MAPPING['snr_rater_box_col'],
            clean_oer_cols=CLEAN_OER_COLS,
            tb_value=NEW_OER_TB_VALUE,
            use_tqdm=NEW_OER_USE_TQDM,
            create_legacy_bwd_diag=True,
        )
        time_stop(merge_timer, action=merge_act, nest=nest+1)
        print_v(f"✅ OER assignment complete: {len(df_with_metrics):,} rows")
        # Save for downstream cells
        save_act = "Saving df_pipeline_04a_basic_metrics"
        save_timer = time_start(save_act, nest=nest+1)
        store_feather(df_with_metrics, 'df_pipeline_04a_basic_metrics')
        time_stop(save_timer, action=save_act, nest=nest+1)
        df_base = df_with_metrics.copy()
        time_stop(cell4, action=cell4_act, nest=nest)
    else:
        print_v("By-passing CELL 4...")
        if CELL5_POOL_MEANS:
            df_base = load_feather('df_pipeline_04a_basic_metrics')
    tyme()


Saving ./running_vars//oer_cols_keep
Saving ./running_vars//snap_cols_clean
✅ Reloaded modules - all configuration settings updated
   • pipeline_config_div_name.py
   • add_cum_oer_metrics_mod_working.py
   • cox_plot_helpers.py
   • NATO_lvl.py
   • py_503_hierarchies.py

   • Current CELL5_POOL_MEANS:  False
   • Current CELL6_POOL_RANKS:  False
   • Current CELL10:  True
   • Current CELL10_5:  True
   • Current CELL11:  True
   • Current CELL12:  True
   • STANDARDIZE_CONFIG['cols']:  ['tb_ratio_fwd_snr', 'pool_minus_mean_snr_fwd']
   • COLUMN_CONFIG['model_static_cols']:  []
   • COLUMN_CONFIG['model_time_varying_cols']:  ['z_tb_ratio_fwd_snr', 'z_pool_minus_mean_snr_fwd', 'z_tb_ratio_fwd_snr_sq', 'z_pool_minus_mean_snr_fwd_sq', 'star_pool_interaction', 'div_name']
   • filtering_params['filter_job_codes']['include_specific']:  ['IN', 'AR', 'EN', 'AV', 'FA', 'SF', 'AD', 'CM', 'CA', 'MI', 'MP', 'SC', 'AG', 'FI', 'OD', 'QM', 'TC', 'LG']

📝 Logging enabled for CELL 4:
   • Log direc

In [10]:
df_cell4 = load_feather('df_pipeline_04a_basic_metrics')
vars_to_check = [
    'tb_ratio_fwd_snr',
    'tb_ratio_bwd_snr',
    'tb_ratio_bwd_snr_legacy',
]

for v in vars_to_check:
    if v in df_cell4.columns:
        s = df_cell4[v]
        print(f"\n{v}")
        print("    non-null:", s.notna().mean())
        print("    zeroes:", (s==0).mean())
        print("    mean:", s.mean())
        print("    std:", s.std())
        print("    non-null:", s.notna().mean())
        print("    min/max:", s.min(), s.max())

 df_pipeline_04a_basic_metrics Loaded!!  - (05.32 seconds and 2,149.103 MB of memory)

tb_ratio_fwd_snr
    non-null: 0.7724672593852959
    zeroes: 0.1523903643623034
    mean: 0.4574048938224075
    std: 0.3203700971826269
    non-null: 0.7724672593852959
    min/max: 0.0 1.0

tb_ratio_bwd_snr
    non-null: 0.7838163943661098
    zeroes: 0.1676814598318389
    mean: 0.445271435246594
    std: 0.32383339258415283
    non-null: 0.7838163943661098
    min/max: 0.0 1.0

tb_ratio_bwd_snr_legacy
    non-null: 0.7724672593852959
    zeroes: 0.1523903643623034
    mean: 0.4574048938224075
    std: 0.3203700971826269
    non-null: 0.7724672593852959
    min/max: 0.0 1.0


In [11]:
if null_reports:
    df_null_report(df_base)

# **📈 CELL 5: POOL METRICS (FWD/BWD)**
##### **Prerequisite: CELL 5 already created individual fwd/bwd metrics**
- `assign_oer_to_snapshots_fast()` creates mode-specific cumulative metrics and ratios
- Input: `df_pipeline_04a_basic_metrics`
##### **This cell adds pool metrics in two steps with checkpoints**
- Step 1: pool means, minus-means, and pool sizes (fwd + bwd)
- Step 2: pool ranks, percentiles, and z-scores (fwd + bwd)


# **CELL 5: POOL MEANS / MINUS-MEANS / SIZE (FWD/BWD)**
##### **Inputs**: `df_pipeline_04a_basic_metrics`
##### **Output**: `df_pipeline_05_pool_means`
##### **Focus**: Add pool mean/minus-mean and pool size for rater and senior rater pools


In [12]:
# === CELL 5: POOL MEANS / MINUS-MEANS / SIZE (FWD/BWD) ===
# Adds pool means, minus-means, and pool sizes for rater/senior rater pools.
# Reload pipeline_config to pick up any changes made
reload_pipeline_config()
with notebook_cell_logging('5'):
    if CELL5_POOL_MEANS:
        print_v("\nCELL 5: Pool means/minus-mean/size (fwd & bwd)...")
        cell5_act = "CELL 5: Pool means/minus-mean/size"
        cell5 = time_start(cell5_act, nest=nest)
        try:
            df_base = load_feather('df_pipeline_04a_basic_metrics')
        except FileNotFoundError:
            print_v("❌ ERROR: df_pipeline_04a_basic_metrics not found. Run CELL 4 first.")
            raise
        from add_cum_oer_metrics_mod_working import add_pool_means_and_sizes
        df_pool_means = add_pool_means_and_sizes(
            df_in=df_base,
            snapshot_date_col=CELL6_COLUMN_MAPPING['snapshot_date_col'],
            rtr_col='rtr_rater_bwd',
            snr_col='snr_rater_bwd',
            ratio_rtr_fwd_col=TB_RATIO_FWD_RTR_COL,
            ratio_snr_fwd_col=TB_RATIO_FWD_SNR_COL,
            ratio_rtr_bwd_col=TB_RATIO_BWD_RTR_COL,
            ratio_snr_bwd_col=TB_RATIO_BWD_SNR_COL,
            pool_min_size=POOL_MIN_SIZE,
            exclude_self=POOL_EXCLUDE_SELF,
        )
        save_act = "Saving df_pipeline_05_pool_means"
        save_timer = time_start(save_act, nest=nest+1)
        store_feather(df_pool_means, 'df_pipeline_05_pool_means')
        time_stop(save_timer, action=save_act, nest=nest+1)
        df_base = df_pool_means.copy()
        time_stop(cell5, action=cell5_act, nest=nest)
    else:
        print_v("By-passing CELL 5...")
        if CELL6_POOL_RANKS:
            try:
                df_base = load_feather('df_pipeline_05_pool_means')
            except FileNotFoundError:
                print_v("⚠️ df_pipeline_05_pool_means not found - run CELL 5")
                raise
    tyme()


Saving ./running_vars//oer_cols_keep
Saving ./running_vars//snap_cols_clean
✅ Reloaded modules - all configuration settings updated
   • pipeline_config_div_name.py
   • add_cum_oer_metrics_mod_working.py
   • cox_plot_helpers.py
   • NATO_lvl.py
   • py_503_hierarchies.py

   • Current CELL5_POOL_MEANS:  False
   • Current CELL6_POOL_RANKS:  False
   • Current CELL10:  True
   • Current CELL10_5:  True
   • Current CELL11:  True
   • Current CELL12:  True
   • STANDARDIZE_CONFIG['cols']:  ['tb_ratio_fwd_snr', 'pool_minus_mean_snr_fwd']
   • COLUMN_CONFIG['model_static_cols']:  []
   • COLUMN_CONFIG['model_time_varying_cols']:  ['z_tb_ratio_fwd_snr', 'z_pool_minus_mean_snr_fwd', 'z_tb_ratio_fwd_snr_sq', 'z_pool_minus_mean_snr_fwd_sq', 'star_pool_interaction', 'div_name']
   • filtering_params['filter_job_codes']['include_specific']:  ['IN', 'AR', 'EN', 'AV', 'FA', 'SF', 'AD', 'CM', 'CA', 'MI', 'MP', 'SC', 'AG', 'FI', 'OD', 'QM', 'TC', 'LG']

📝 Logging enabled for CELL 5:
   • Log direc

In [13]:
if null_reports:
    df_debug = load_feather('df_pipeline_05_pool_means')
    df_null_report(df_debug)


# **CELL 6: POOL RANKS / PERCENTILES / Z-SCORES (FWD/BWD)**
##### **Inputs**: `df_pipeline_05_pool_means`
##### **Output**: `df_pipeline_06_pool_ranks`
##### **Focus**: Add rank, percentile, and z-score pool metrics for rater and senior rater pools


In [14]:
# === CELL 6: POOL RANKS / PERCENTILES / Z-SCORES (FWD/BWD) ===
# Adds pool ranks, rank percentiles, and pool z-scores for rater/senior rater pools.
# Reload pipeline_config to pick up any changes made
reload_pipeline_config()
with notebook_cell_logging('6'):
    if CELL6_POOL_RANKS:
        print_v("\nCELL 6: Pool ranks/percentiles/z-scores (fwd & bwd)...")
        cell6_act = "CELL 6: Pool ranks/percentiles/z-scores"
        cell6 = time_start(cell6_act, nest=nest)
        try:
            df_base = load_feather('df_pipeline_05_pool_means')
        except FileNotFoundError:
            print_v("❌ ERROR: df_pipeline_05_pool_means not found. Run CELL 5 first.")
            raise
        from add_cum_oer_metrics_mod_working import add_pool_ranks_pct_zscores
        df_pool_ranks = add_pool_ranks_pct_zscores(
            df_in=df_base,
            snapshot_date_col=CELL6_COLUMN_MAPPING['snapshot_date_col'],
            rtr_col='rtr_rater_bwd',
            snr_col='snr_rater_bwd',
            ratio_rtr_fwd_col=TB_RATIO_FWD_RTR_COL,
            ratio_snr_fwd_col=TB_RATIO_FWD_SNR_COL,
            ratio_rtr_bwd_col=TB_RATIO_BWD_RTR_COL,
            ratio_snr_bwd_col=TB_RATIO_BWD_SNR_COL,
            pool_min_size=POOL_MIN_SIZE,
            rank_ascending=True,
        )
        save_act = "Saving df_pipeline_06_pool_ranks"
        save_timer = time_start(save_act, nest=nest+1)
        store_feather(df_pool_ranks, 'df_pipeline_06_pool_ranks')
        time_stop(save_timer, action=save_act, nest=nest+1)
        df_base = df_pool_ranks.copy()
        time_stop(cell6, action=cell6_act, nest=nest)
    else:
        print_v("By-passing CELL 6...")
        if CELL7:
            try:
                df_base = load_feather('df_pipeline_06_pool_ranks')
            except FileNotFoundError:
                print_v("⚠️ df_pipeline_06_pool_ranks not found - run CELL 6")
                raise
    tyme()


Saving ./running_vars//oer_cols_keep
Saving ./running_vars//snap_cols_clean
✅ Reloaded modules - all configuration settings updated
   • pipeline_config_div_name.py
   • add_cum_oer_metrics_mod_working.py
   • cox_plot_helpers.py
   • NATO_lvl.py
   • py_503_hierarchies.py

   • Current CELL5_POOL_MEANS:  False
   • Current CELL6_POOL_RANKS:  False
   • Current CELL10:  True
   • Current CELL10_5:  True
   • Current CELL11:  True
   • Current CELL12:  True
   • STANDARDIZE_CONFIG['cols']:  ['tb_ratio_fwd_snr', 'pool_minus_mean_snr_fwd']
   • COLUMN_CONFIG['model_static_cols']:  []
   • COLUMN_CONFIG['model_time_varying_cols']:  ['z_tb_ratio_fwd_snr', 'z_pool_minus_mean_snr_fwd', 'z_tb_ratio_fwd_snr_sq', 'z_pool_minus_mean_snr_fwd_sq', 'star_pool_interaction', 'div_name']
   • filtering_params['filter_job_codes']['include_specific']:  ['IN', 'AR', 'EN', 'AV', 'FA', 'SF', 'AD', 'CM', 'CA', 'MI', 'MP', 'SC', 'AG', 'FI', 'OD', 'QM', 'TC', 'LG']

📝 Logging enabled for CELL 6:
   • Log direc

In [15]:
if null_reports:
    df_debug = load_feather('df_pipeline_06_pool_ranks')
    df_null_report(df_debug)


# **📊 PHASE 3: VARIABLE CREATION**
## **CELL 7: Create Time-Varying Variables**
## **CELL 8: Create Static Variables**


# **⏰ CELL 7: CREATE TIME-VARYING VARIABLES**
##### **Create variables that change over time**
- **Unit Assignments**: Prestige units, GTW units, division assignments
- **Cumulative Metrics**: Prestige sum/mean, division eigenvalues
- **Job Codes**: Current job code, job code changes
- **Output**: `df_pipeline_07_time_varying`
##### **Focus**: Create dynamic variables for each snapshot


In [23]:
del df_base


In [24]:
df_test = load_feather("df_uic_div_lookup_pde")
assert {'asg_uic_pde','fy','div_name'}.issubset(df_test.columns)
assert df_test[['asg_uic_pde','fy']].drop_duplicates().shape[0] == len(df_test), "non-unique key rows"
print("div non-null rate:", df_test['div_name'].notna().mean())

 df_uic_div_lookup_pde Loaded!!  - (00.01 seconds and 3.243 MB of memory)
div non-null rate: 0.4922920548873454


In [25]:
# === CELL 7: CREATE TIME-VARYING VARIABLES ===
# Create variables that change over time (prestige, divisions, job codes, etc.)
# NOTE: If loading from 502, prestige_unit and prestige_sum are already included
# Reload pipeline_config to pick up any changes made 
# to configuration and other .py scripts
reload_pipeline_config()
with notebook_cell_logging('7'):
    if CELL7:
        print_v("\n⏰ CELL 7: Creating time-varying variables...")
        cell7_act = "CELL 7: Create time-varying variables"
        cell7 = time_start(cell7_act, nest=0)
        
        # Load data with metrics if not already loaded
        if 'df_base' not in locals():
            df_base = load_feather('df_pipeline_06_pool_ranks')
        
        df_time_varying = df_base.copy()
        total_snapshots = len(df_time_varying)
        
        # Add fiscal year if not present
        if 'fy' not in df_time_varying.columns:
            fy_act = "Adding fiscal year column"
            fy_timer = time_start(fy_act, nest=1)
            df_time_varying = add_fy_col(df_time_varying, date_column='snpsht_dt', show=True)
            time_stop(fy_timer, action=fy_act, nest=1)
        
        # === 7.1. PRESTIGE VARIABLES (Config-driven; delegated to py_503_hierarchies) ===
        print_v("\n📊 Checking unit classification variables...")
        print_v("   (Uses PRESTIGE_CONFIG in pipeline_config.py)")
        try:
            from py_503_hierarchies import ensure_prestige_metrics
            prestige_act = "Applying prestige metrics (503_hierarchies)"
            prestige_timer = time_start(prestige_act, nest=1)
            df_time_varying, prestige_info = ensure_prestige_metrics(
                df_time_varying,
                prestige_config=PRESTIGE_CONFIG,
                fy_col='fy',
                uic_col='asg_uic_pde',
                pid_col='pid_pde',
                date_col='snpsht_dt',
                unit_col='prestige_unit',
                sum_col='prestige_sum',
                mean_col='prestige_mean',
                logger=print_v,
            )
            time_stop(prestige_timer, action=prestige_act, nest=1)

            if not prestige_info.get('enabled', True):
                print_v("⏭️ Prestige processing disabled by PRESTIGE_CONFIG['enable_prestige_processing']")
            else:
                if prestige_info.get('created_unit', False):
                    print_v("✅ Created prestige_unit")
                elif 'prestige_unit' in df_time_varying.columns:
                    print_v("✅ prestige_unit already exists (likely from 502 output)")

                if prestige_info.get('created_sum', False) and prestige_info.get('created_mean', False):
                    print_v("✅ Created cumulative prestige metrics (prestige_sum, prestige_mean)")
                elif prestige_info.get('created_sum', False):
                    print_v("✅ Created cumulative prestige metric (prestige_sum)")
                elif prestige_info.get('created_mean', False):
                    print_v("✅ Created cumulative prestige metric (prestige_mean)")
                elif 'prestige_sum' in df_time_varying.columns:
                    print_v("✅ prestige_sum already exists (likely from 502 output)")

            # Report statistics (using clear terminology: snapshots, not assignments)
            if 'prestige_unit' in df_time_varying.columns:
                prestige_snpshts = df_time_varying[df_time_varying['prestige_unit'] == 1]
                prestige_snpsht_count = len(prestige_snpshts)
                officers_with_prestige = prestige_snpshts['pid_pde'].nunique()
                prestige_pct = (prestige_snpsht_count / total_snapshots * 100) if total_snapshots > 0 else 0
                print_v(f"  • Total snapshots in prestige units: {prestige_snpsht_count:,} / {total_snapshots:,} snapshots ({prestige_pct:.1f}%)")
                print_v(f"  • Officers with prestige service: {officers_with_prestige:,}")
        except Exception as e:
            print_v(f"⚠️ Prestige step failed: {e}")
            if 'prestige_unit' not in df_time_varying.columns:
                df_time_varying['prestige_unit'] = 0
        
        # === 7.3. CREATE DIVISION NAME + METRICS (Optional) ===
        if DIVISION_CONFIG.get('enabled', False):
            try:
                from py_503_hierarchies import apply_division_name, add_division_metrics; print("IMPORT is GOOD")
                df_time_varying = apply_division_name(df_time_varying, DIVISION_CONFIG); print("DIVISION_NAME is GOOD")
                df_time_varying = add_division_metrics(df_time_varying, DIVISION_CONFIG)
                print_v("✅ Division name/metrics updated")
                # DIV DEBUG: immediate post-merge coverage checks
                if 'div_name' in df_time_varying.columns:
                    print_v(f"[DIV_DEBUG] div_name non-null rate after Cell 7 merge: {df_time_varying['div_name'].notna().mean():.2%}")
                    miss_mask = df_time_varying['div_name'].isna()
                    if miss_mask.any():
                        print_v(f"[DIV_DEBUG] Missing div_name rows:{int(miss_mask.sum()):,}/{len(df_time_varying):,})")
                        if 'asg_uic_pde' in df_time_varying.columns:
                            print(df_time_varying.loc[miss_mask, 'asg_uic_pde'].astype(str).value_counts().head(15))
                        if 'fy' in df_time_varying.columns:
                            print(df_time_varying.loc[miss_mask, 'fy'].value_counts(dropna=False).head(15))                
            except Exception as e:
                print_v(f"⚠️ Division name/metrics step failed: {e}")
        
        # === 7.4. CREATE JOB CODE VARIABLE ===
        if 'job_code' not in df_time_varying.columns and 'occ_crer_grp_cd' in df_time_varying.columns:
            job_act = "Creating job_code from occ_crer_grp_cd"
            job_timer = time_start(job_act, nest=1)
            df_time_varying['job_code'] = df_time_varying['occ_crer_grp_cd'].astype('string')
            time_stop(job_timer, action=job_act, nest=1)
            print_v("✅ Created job_code from occ_crer_grp_cd")
        
        print_v(f"\n✅ Time-varying variables created: {df_time_varying.shape}")
        print_v(f"  • Columns: {len(df_time_varying.columns)}")
        if 'prestige_unit' in df_time_varying.columns:
            prestige_snpsht_count = df_time_varying['prestige_unit'].sum()
            prestige_pct = (prestige_snpsht_count / len(df_time_varying) * 100) if len(df_time_varying) > 0 else 0
            print_v(f"  • Snapshots in prestige units: {prestige_snpsht_count:,} / {len(df_time_varying):,} snapshots ({prestige_pct:.1f}%)")
        
        # Save
        save_act = "Saving df_pipeline_07_time_varying"
        save_timer = time_start(save_act, nest=1)
        store_feather(df_time_varying, 'df_pipeline_07_time_varying')
        time_stop(save_timer, action=save_act, nest=1)
        df_base = df_time_varying.copy()
        
        time_stop(cell7, action=cell7_act, nest=0)
    else:
        print_v("By-passing CELL 7...")
        if CELL8:
            df_base = load_feather('df_pipeline_07_time_varying')
    tyme()


Saving ./running_vars//oer_cols_keep
Saving ./running_vars//snap_cols_clean
✅ Reloaded modules - all configuration settings updated
   • pipeline_config_div_name.py
   • add_cum_oer_metrics_mod_working.py
   • cox_plot_helpers.py
   • NATO_lvl.py
   • py_503_hierarchies.py

   • Current CELL5_POOL_MEANS:  False
   • Current CELL6_POOL_RANKS:  False
   • Current CELL10:  True
   • Current CELL10_5:  True
   • Current CELL11:  True
   • Current CELL12:  True
   • STANDARDIZE_CONFIG['cols']:  ['tb_ratio_fwd_snr', 'pool_minus_mean_snr_fwd']
   • COLUMN_CONFIG['model_static_cols']:  []
   • COLUMN_CONFIG['model_time_varying_cols']:  ['z_tb_ratio_fwd_snr', 'z_pool_minus_mean_snr_fwd', 'z_tb_ratio_fwd_snr_sq', 'z_pool_minus_mean_snr_fwd_sq', 'star_pool_interaction', 'div_name']
   • filtering_params['filter_job_codes']['include_specific']:  ['IN', 'AR', 'EN', 'AV', 'FA', 'SF', 'AD', 'CM', 'CA', 'MI', 'MP', 'SC', 'AG', 'FI', 'OD', 'QM', 'TC', 'LG']

📝 Logging enabled for CELL 7:
   • Log direc

In [19]:
df_ul = load_feather('df_uic_div_lookup')

 df_uic_div_lookup Loaded!!  - (00.01 seconds and 3.627 MB of memory)


In [20]:
df_ul.fy.isna().sum()

0

In [21]:
if null_reports:
    df_debug =load_feather('df_pipeline_07_time_varying')
    df_null_report(df_debug)

df_debug =load_feather('df_pipeline_07_time_varying')
'div_name' in df_debug.columns


 df_pipeline_07_time_varying Loaded!!  - (04.46 seconds and 3,187.046 MB of memory)


True

# **🔒 CELL 8: CREATE STATIC VARIABLES**
##### **Create variables that don't change over time**
- **Demographics**: `sex` from `pn_sex_cd`, `age_cpt`, `age_maj`
- **Career**: `job_code_changed`, `final_job_code`
- **Marital Status**: `married` from `mrtl_stat_cd`
- **Output**: `df_pipeline_08_combined` (time-varying + static merged)
##### **Focus**: Create static variables and merge with time-varying


In [22]:
# === CELL 8: CREATE STATIC VARIABLES ===
# Create variables that don't change over time and merge with time-varying
# Reload pipeline_config to pick up any changes made 
# to configuration and other .py scripts
reload_pipeline_config()
with notebook_cell_logging('8'):
    if CELL8:
        print_v("\n🔒 CELL 8: Creating static variables...")
        cell8_act = "CELL 8: Create static variables"
        cell8 = time_start(cell8_act, nest=0)
        
        # Load time-varying data if not already loaded
        if 'df_base' not in locals():
            df_base = load_feather('df_pipeline_07_time_varying')
        
        df_time_varying = df_base.copy()
        
        # Create sex column from pn_sex_cd (convert M/F to 1/0)
        # Also handle case where sex exists but is still in string format
        if 'sex' not in df_time_varying.columns:
            if 'pn_sex_cd' in df_time_varying.columns:
                sex_act = "Creating sex column from pn_sex_cd (M=1, F=0)"
                sex_timer = time_start(sex_act, nest=1)
                # Convert M to 1, F (or anything else) to 0, same logic as married column
                df_time_varying['sex'] = (df_time_varying['pn_sex_cd'] == 'M').astype(int)
                time_stop(sex_timer, action=sex_act, nest=1)
                print_v("✅ Created sex column from pn_sex_cd (M=1, F=0)")
            else:
                print_v("⚠️ pn_sex_cd not found, cannot create sex column")
        elif 'sex' in df_time_varying.columns and df_time_varying['sex'].dtype == 'object':
            # Sex column exists but is still string format, convert it
            sex_act = "Converting existing sex column from string to numeric (M=1, F=0)"
            sex_timer = time_start(sex_act, nest=1)
            df_time_varying['sex'] = (df_time_varying['sex'] == 'M').astype(int)
            time_stop(sex_timer, action=sex_act, nest=1)
            print_v("✅ Converted existing sex column to numeric (M=1, F=0)")
        
        # Create age_cpt (age at Captain promotion)
        if 'age_cpt' not in df_time_varying.columns:
            if 'dor_cpt' in df_time_varying.columns and 'date_birth_pde' in df_time_varying.columns:
                age_act = "Creating age_cpt (age at Captain promotion)"
                age_timer = time_start(age_act, nest=1)
                # Calculate age at promotion for each officer
                df_age = df_time_varying[['pid_pde', 'dor_cpt', 'date_birth_pde']].drop_duplicates().dropna()
                if len(df_age) > 0:
                    # Calculate age difference in years
                    df_age['age_cpt'] = (df_age['dor_cpt'] - df_age['date_birth_pde']).dt.days / 365.25
                    age_cpt_dict = dict(zip(df_age['pid_pde'], df_age['age_cpt']))
                    df_time_varying['age_cpt'] = df_time_varying['pid_pde'].map(age_cpt_dict)
                time_stop(age_timer, action=age_act, nest=1)
                print_v("✅ Created age_cpt column")
            else:
                print_v("⚠️ dor_cpt or date_birth_pde not found, cannot create age_cpt")
        
        # Create married status
        if 'married' not in df_time_varying.columns:
            if 'mrtl_stat_cd' in df_time_varying.columns:
                married_act = "Creating married column from mrtl_stat_cd"
                married_timer = time_start(married_act, nest=1)
                # Married = 1, Single/Other = 0 (adjust based on your coding)
                df_time_varying['married'] = (df_time_varying['mrtl_stat_cd'] == 'M').astype(int)
                time_stop(married_timer, action=married_act, nest=1)
                print_v("✅ Created married column from mrtl_stat_cd")
            else:
                print_v("⚠️ mrtl_stat_cd not found, cannot create married column")
        
        # Create job_code_changed and final_job_code
        if 'job_code' in df_time_varying.columns:
            job_act = "Creating job code variable(s)"
            job_timer = time_start(job_act, nest=1)
            # Sort by pid_pde and snpsht_dt
            df_time_varying = df_time_varying.sort_values(by=['pid_pde', 'snpsht_dt'])
        
            # Job code changed: 1 if changed BEFORE OR DURING SOURCE_RANK period (ignoring NaNs)
            if CREATE_JOB_CODE_CHANGED:
                changed_act = "Working job_code_changed"
                changed_timer = time_start(changed_act, nest=3)
                last_src_rnk_date = (df_time_varying[df_time_varying['rank_pde'] == SOURCE_RANK]
                                     .groupby('pid_pde')['snpsht_dt'].last())
        
                # Check if job_code changed from first snapshot through last SOURCE_RANK snapshot
                def job_code_changed_before_or_during_src_rnk(pid_group):
                    pid = pid_group.name
                    cutoff = last_src_rnk_date.get(pid, pid_group['snpsht_dt'].max())
                    subset = pid_group[pid_group['snpsht_dt'] <= cutoff]
                    # Filter out NaN values before checking for changes
                    job_codes = subset['job_code'].dropna()
                    return (job_codes != job_codes.iloc[0]).any() if len(job_codes) > 1 else 0
                job_code_changed_dict = df_time_varying.groupby('pid_pde').apply(job_code_changed_before_or_during_src_rnk).astype(int).to_dict()
                df_time_varying['job_code_changed'] = df_time_varying['pid_pde'].map(job_code_changed_dict).fillna(0)
                time_stop(changed_timer,action=changed_act,nest=3)
            else:
                df_time_varying['job_code_changed'] = 0
           
            # Final job code (last SOURCE_RANK snapshot job code - last job_code before promotion, censoring or attrition)
            final_act = "Working final_job_code"
            final_timer = time_start(final_act, nest=2)
            final_job_code_dict = (df_time_varying[df_time_varying['rank_pde'] == SOURCE_RANK]
                .sort_values(by=['pid_pde','snpsht_dt'])
                .groupby('pid_pde')['job_code'].last().to_dict())
            df_time_varying['final_job_code'] = df_time_varying['pid_pde'].map(final_job_code_dict)
            print_v(f"✅ Calculated final_job_code from last {SOURCE_RANK} snapshot for each officer")
            time_stop(final_timer,action=final_act,nest=2)
            
            time_stop(job_timer, action=job_act, nest=1)
            print_v("✅ Created final_job_code" + (" and job_code_changed" if CREATE_JOB_CODE_CHANGED else " (job_code_changed skipped)"))
        
        print_v(f"✅ Static variables created: {df_time_varying.shape}")
        
        # Verify key columns exist
        key_static_vars = ['sex', 'yg', 'age_cpt']
        missing_vars = [var for var in key_static_vars if var not in df_time_varying.columns]
        if missing_vars:
            print_v(f"⚠️ Missing static variables: {missing_vars}")
        else:
            print_v(f"✅ All key static variables present: {key_static_vars}")
        
        # Save combined data
        save_act = "Saving df_pipeline_08_combined"
        save_timer = time_start(save_act, nest=1)
        store_feather(df_time_varying, 'df_pipeline_08_combined')
        time_stop(save_timer, action=save_act, nest=1)
        df_base = df_time_varying.copy()
        
        time_stop(cell8, action=cell8_act, nest=0)
    else:
        print_v("By-passing CELL 8...")
        if CELL9:
            df_base = load_feather('df_pipeline_08_combined')
    tyme()


Saving ./running_vars//oer_cols_keep
Saving ./running_vars//snap_cols_clean
✅ Reloaded modules - all configuration settings updated
   • pipeline_config_div_name.py
   • add_cum_oer_metrics_mod_working.py
   • cox_plot_helpers.py
   • NATO_lvl.py
   • py_503_hierarchies.py

   • Current CELL5_POOL_MEANS:  False
   • Current CELL6_POOL_RANKS:  False
   • Current CELL10:  True
   • Current CELL10_5:  True
   • Current CELL11:  True
   • Current CELL12:  True
   • STANDARDIZE_CONFIG['cols']:  ['tb_ratio_fwd_snr', 'pool_minus_mean_snr_fwd']
   • COLUMN_CONFIG['model_static_cols']:  []
   • COLUMN_CONFIG['model_time_varying_cols']:  ['z_tb_ratio_fwd_snr', 'z_pool_minus_mean_snr_fwd', 'z_tb_ratio_fwd_snr_sq', 'z_pool_minus_mean_snr_fwd_sq', 'star_pool_interaction', 'div_name']
   • filtering_params['filter_job_codes']['include_specific']:  ['IN', 'AR', 'EN', 'AV', 'FA', 'SF', 'AD', 'CM', 'CA', 'MI', 'MP', 'SC', 'AG', 'FI', 'OD', 'QM', 'TC', 'LG']

📝 Logging enabled for CELL 8:
   • Log direc

KeyboardInterrupt: 

In [ ]:
if null_reports:
    df_debug =load_feather('df_pipeline_08_combined')
    df_null_report(df_debug)
df_debug =load_feather('df_pipeline_08_combined')
'div_name' in df_debug.columns

# **🔍 PHASE 4: ADVANCED FILTERING**
## **CELL 9: Advanced Filtering**


# **🔍 CELL 9: ADVANCED FILTERING**
##### **Apply analysis-specific filters**
- **Date Ranges**: Filter snapshots by date
- **Job Codes**: Include/exclude specific job codes
- **Gender**: Filter by gender
- **Year Groups**: Filter by yg ranges
- **Divisions**: GTW, prestige, specific divisions
- **Output**: `df_pipeline_09_filtered` (ready for Cox analysis)
##### **Focus**: Create filtered dataset for specific analysis


In [ ]:
del df_base

In [ ]:
# === CELL 9: ADVANCED FILTERING ===
# Apply analysis-specific filters (dates, job codes, divisions, etc.)
# Reload pipeline_config to pick up any changes made 
# to configuration and other .py scripts
reload_pipeline_config()
with notebook_cell_logging('9'):
    if CELL9:
        print_v("\n🔍 CELL 9: Applying advanced filtering...")
        cell9_act = "CELL 9: Advanced filtering"
        cell9 = time_start(cell9_act, nest=0)
        print("FILTERING PARAMETERS:",filtering_params)
        # Load combined data if not already loaded
        if 'df_base' not in locals():
            df_base = load_feather('df_pipeline_08_combined')
        
        df_filtered = df_base.copy()
        initial_count = len(df_filtered)
        initial_officers = df_filtered.pid_pde.nunique()
        print_v(f"📊 Initial row count: {initial_count:,}")
        print_v(f"📊 Initial officers: {initial_officers:,}")
        
        # Track cumulative removals
        current_count = initial_count
        current_officers = initial_officers
        
        # === FILTERING PARAMETERS ===
        # filtering_params is imported from pipeline_config.py
        # Edit pipeline_config.py to modify filtering settings
        # The filtering_params dictionary contains:
        #   - filter_dates: Date range filtering
        #   - filter_job_codes: Job code inclusion/exclusion
        #   - filter_gender: Gender filtering options
        #   - filter_ygs: Year group filtering
        #   - filter_divisions: Division/unit filtering
        
        # === DATE FILTERING ===
        if filtering_params['filter_dates']['enabled']:
            filter_act = "Date filtering"
            filter_timer = time_start(filter_act, nest=1)
            start_date = pd.to_datetime(filtering_params['filter_dates']['start_date'])
            end_date = pd.to_datetime(filtering_params['filter_dates']['end_date'])
            before_count = len(df_filtered)
            before_officers = df_filtered.pid_pde.nunique()
            df_filtered = df_filtered[
                (df_filtered['snpsht_dt'] >= start_date) & 
                (df_filtered['snpsht_dt'] <= end_date)
            ].copy()
            after_count = len(df_filtered)
            after_officers = df_filtered.pid_pde.nunique()
            removed_rows = before_count - after_count
            removed_officers = before_officers - after_officers
            time_stop(filter_timer, action=filter_act, nest=1)
            print_v(f"✅ Date filtering: {after_count:,} rows ({after_officers:,} officers)")
            print_v(f"   • Removed: {removed_rows:,} rows ({removed_rows/before_count*100:.2f}% of rows)")
            print_v(f"   • Removed: {removed_officers:,} officers ({removed_officers/before_officers*100:.2f}% of officers)")
            current_count = after_count
            current_officers = after_officers
        
        # === JOB CODE FILTERING ===
        if filtering_params['filter_job_codes']['enabled']:
            if filtering_params['filter_job_codes']['exclude_problematic']:
                filter_act = "Excluding problematic job codes"
                filter_timer = time_start(filter_act, nest=1)
                problematic = filtering_params['filter_job_codes']['problematic_codes']
                if 'occ_crer_grp_cd' in df_filtered.columns:
                    before_count = len(df_filtered)
                    before_officers = df_filtered.pid_pde.nunique()
                    problematic_officers = df_filtered[df_filtered['occ_crer_grp_cd'].isin(problematic)]['pid_pde'].unique()
                    df_filtered = df_filtered[~df_filtered['pid_pde'].isin(problematic_officers)].copy()
                    after_count = len(df_filtered)
                    after_officers = df_filtered.pid_pde.nunique()
                    removed_rows = before_count - after_count
                    removed_officers = len(problematic_officers)
                    time_stop(filter_timer, action=filter_act, nest=1)
                    print_v(f"✅ Excluded {removed_officers:,} officers with problematic job codes")
                    print_v(f"   • Excluded job codes: {problematic}")
                    print_v(f"   • Removed: {removed_rows:,} rows ({removed_rows/before_count*100:.2f}% of rows)")
                    print_v(f"   • Removed: {removed_officers:,} officers ({removed_officers/before_officers*100:.2f}% of officers)")
                    current_count = after_count
                    current_officers = after_officers
            
            if filtering_params['filter_job_codes']['include_specific']:
                filter_act = "Including specific job codes"
                filter_timer = time_start(filter_act, nest=1)
                include_codes = filtering_params['filter_job_codes']['include_specific']
                exclude_other_during_target_rank = filtering_params['filter_job_codes'].get('exclude_other_jobs_during_target_rank', filtering_params['filter_job_codes'].get('include_only_specific', True))
                if 'occ_crer_grp_cd' in df_filtered.columns:
                    before_count = len(df_filtered)
                    before_officers = df_filtered.pid_pde.nunique()
                    include_officers = df_filtered[df_filtered['occ_crer_grp_cd'].isin(include_codes)]['pid_pde'].unique()
                    df_filtered = df_filtered[df_filtered['pid_pde'].isin(include_officers)].copy()
                    if exclude_other_during_target_rank:
                        df_other_jobs = df_filtered[(~df_filtered.job_code.isin(include_codes)) & (df_filtered.rank_pde == SOURCE_RANK)].copy()
                        other_jobs_officers = df_other_jobs['pid_pde'].unique()
                        df_filtered = df_filtered[~df_filtered['pid_pde'].isin(other_jobs_officers)]
                    after_count = len(df_filtered)
                    after_officers = df_filtered.pid_pde.nunique()
                    removed_rows = before_count - after_count
                    removed_officers = before_officers - after_officers
                    time_stop(filter_timer, action=filter_act, nest=1)
                    print_v(f"✅ Included {len(include_officers):,} officers with specific job codes")
                    print_v(f"   • Included job codes: {include_codes}")
                    if exclude_other_during_target_rank:
                        print_v(f"   • Removed officers with any other job_codes during their {SOURCE_RANK} time")
                    print_v(f"   • Removed: {removed_rows:,} rows ({removed_rows/before_count*100:.2f}% of rows)")
                    print_v(f"   • Removed: {removed_officers:,} officers ({removed_officers/before_officers*100:.2f}% of officers)")
                    current_count = after_count
                    current_officers = after_officers
        
        # === GENDER FILTERING ===
        if filtering_params['filter_gender']['enabled']:
            if 'sex' in df_filtered.columns:
                # Step 1: Exclude dual-gender officers (if enabled)
                if filtering_params['filter_gender']['exclude_dual_gender']:
                    filter_act = "Excluding dual gender officers"
                    filter_timer = time_start(filter_act, nest=1)
                    before_count = len(df_filtered)
                    before_officers = df_filtered.pid_pde.nunique()
                    gender_counts = df_filtered.groupby('pid_pde')['sex'].nunique()
                    dual_gender_officers = gender_counts[gender_counts > 1].index
                    df_filtered = df_filtered[~df_filtered['pid_pde'].isin(dual_gender_officers)].copy()
                    after_count = len(df_filtered)
                    after_officers = df_filtered.pid_pde.nunique()
                    removed_rows = before_count - after_count
                    removed_officers = len(dual_gender_officers)
                    time_stop(filter_timer, action=filter_act, nest=1)
                    print_v(f"✅ Excluded {removed_officers:,} officers with multiple gender values")
                    print_v(f"   • Removed: {removed_rows:,} rows ({removed_rows/before_count*100:.2f}% of rows)")
                    print_v(f"   • Removed: {removed_officers:,} officers ({removed_officers/before_officers*100:.2f}% of officers)")
                    current_count = after_count
                    current_officers = after_officers
                
                # Step 2: Filter to specific gender (if specified)
                include_gender = filtering_params['filter_gender'].get('include_gender', None)
                if include_gender and include_gender.lower() in ['male', 'female', 'm', 'f']:
                    filter_act = f"Filtering to {include_gender.lower()} officers only"
                    filter_timer = time_start(filter_act, nest=1)
                    before_count = len(df_filtered)
                    before_officers = df_filtered.pid_pde.nunique()
                    
                    # Convert gender string to numeric (sex: 1=Male, 0=Female)
                    if include_gender.lower() in ['male', 'm']:
                        gender_value = 1
                        gender_label = 'Male'
                    else:  # 'female' or 'f'
                        gender_value = 0
                        gender_label = 'Female'
                    
                    # Filter to officers with the specified gender
                    # Get unique officers with the specified gender
                    gender_officers = df_filtered[df_filtered['sex'] == gender_value]['pid_pde'].unique()
                    df_filtered = df_filtered[df_filtered['pid_pde'].isin(gender_officers)].copy()
                    
                    after_count = len(df_filtered)
                    after_officers = df_filtered.pid_pde.nunique()
                    removed_rows = before_count - after_count
                    removed_officers = before_officers - after_officers
                    time_stop(filter_timer, action=filter_act, nest=1)
                    print_v(f"✅ Filtered to {gender_label} officers only: {after_officers:,} officers ({after_count:,} rows)")
                    print_v(f"   • Removed: {removed_rows:,} rows ({removed_rows/before_count*100:.2f}% of rows)")
                    print_v(f"   • Removed: {removed_officers:,} officers ({removed_officers/before_officers*100:.2f}% of officers)")
                    current_count = after_count
                    current_officers = after_officers
            else:
                print_v("⚠️ 'sex' column not found, skipping gender filtering")
        
        # === YEAR GROUP FILTERING ===
        if filtering_params['filter_ygs']['enabled']:
            if filtering_params['filter_ygs']['include_specific']:
                filter_act = "Including specific year groups"
                filter_timer = time_start(filter_act, nest=1)
                ygs = filtering_params['filter_ygs']['include_specific']
                before_count = len(df_filtered)
                before_officers = df_filtered.pid_pde.nunique()
                df_filtered = df_filtered[df_filtered['yg'].isin(ygs)].copy()
                after_count = len(df_filtered)
                after_officers = df_filtered.pid_pde.nunique()
                removed_rows = before_count - after_count
                removed_officers = before_officers - after_officers
                time_stop(filter_timer, action=filter_act, nest=1)
                print_v(f"✅ Included year groups: {ygs}")
                print_v(f"   • Removed: {removed_rows:,} rows ({removed_rows/before_count*100:.2f}% of rows)")
                print_v(f"   • Removed: {removed_officers:,} officers ({removed_officers/before_officers*100:.2f}% of officers)")
                current_count = after_count
                current_officers = after_officers
            if filtering_params['filter_ygs']['exclude_specific']:
                filter_act = "Excluding specific year groups"
                filter_timer = time_start(filter_act, nest=1)
                exclude_ygs = filtering_params['filter_ygs']['exclude_specific']
                before_count = len(df_filtered)
                before_officers = df_filtered.pid_pde.nunique()
                exclude_officers = df_filtered[df_filtered['yg'].isin(exclude_ygs)]['pid_pde'].unique()
                df_filtered = df_filtered[~df_filtered['yg'].isin(exclude_ygs)].copy()
                after_count = len(df_filtered)
                after_officers = df_filtered.pid_pde.nunique()
                removed_rows = before_count - after_count
                removed_officers = len(exclude_officers)
                time_stop(filter_timer, action=filter_act, nest=1)
                print_v(f"✅ Excluded year groups: {exclude_ygs}")
                print_v(f"   • Removed: {removed_rows:,} rows ({removed_rows/before_count*100:.2f}% of rows)")
                print_v(f"   • Removed: {removed_officers:,} officers ({removed_officers/before_officers*100:.2f}% of officers)")
                current_count = after_count
                current_officers = after_officers
        
        # === DIVISION FILTERING ===
        if filtering_params['filter_divisions']['enabled']:
            if filtering_params['filter_divisions']['gtw_only'] and 'gtw_unit' in df_filtered.columns:
                filter_act = "GTW units only"
                filter_timer = time_start(filter_act, nest=1)
                before_count = len(df_filtered)
                before_officers = df_filtered.pid_pde.nunique()
                gtw_officers = df_filtered[df_filtered['gtw_unit'] == 1]['pid_pde'].unique()
                df_filtered = df_filtered[df_filtered['pid_pde'].isin(gtw_officers)].copy()
                after_count = len(df_filtered)
                after_officers = df_filtered.pid_pde.nunique()
                removed_rows = before_count - after_count
                removed_officers = before_officers - after_officers
                time_stop(filter_timer, action=filter_act, nest=1)
                print_v(f"✅ GTW units only: {len(gtw_officers):,} officers")
                print_v(f"   • Removed: {removed_rows:,} rows ({removed_rows/before_count*100:.2f}% of rows)")
                print_v(f"   • Removed: {removed_officers:,} officers ({removed_officers/before_officers*100:.2f}% of officers)")
                current_count = after_count
                current_officers = after_officers
            if filtering_params['filter_divisions']['prestige_only'] and 'prestige_unit' in df_filtered.columns:
                filter_act = "Prestige units only"
                filter_timer = time_start(filter_act, nest=1)
                before_count = len(df_filtered)
                before_officers = df_filtered.pid_pde.nunique()
                df_filtered = df_filtered[df_filtered['prestige_unit'] == 1].copy()
                after_count = len(df_filtered)
                after_officers = df_filtered.pid_pde.nunique()
                removed_rows = before_count - after_count
                removed_officers = before_officers - after_officers
                time_stop(filter_timer, action=filter_act, nest=1)
                print_v(f"✅ Prestige units only")
                print_v(f"   • Removed: {removed_rows:,} rows ({removed_rows/before_count*100:.2f}% of rows)")
                print_v(f"   • Removed: {removed_officers:,} officers ({removed_officers/before_officers*100:.2f}% of officers)")
                current_count = after_count
                current_officers = after_officers
            if filtering_params['filter_divisions']['include_divisions'] and 'div_name' in df_filtered.columns:
                filter_act = "Including specific divisions"
                filter_timer = time_start(filter_act, nest=1)
                divs = filtering_params['filter_divisions']['include_divisions']
                before_count = len(df_filtered)
                before_officers = df_filtered.pid_pde.nunique()
                include_officers = df_filtered[df_filtered['div_name'].isin(divs)]['pid_pde'].unique()
                df_filtered = df_filtered[df_filtered['pid_pde'].isin(include_officers)].copy()
                after_count = len(df_filtered)
                after_officers = df_filtered.pid_pde.nunique()
                removed_rows = before_count - after_count
                removed_officers = before_officers - after_officers
                time_stop(filter_timer, action=filter_act, nest=1)
                print_v(f"✅ Included divisions: {divs}")
                print_v(f"   • Removed: {removed_rows:,} rows ({removed_rows/before_count*100:.2f}% of rows)")
                print_v(f"   • Removed: {removed_officers:,} officers ({removed_officers/before_officers*100:.2f}% of officers)")
                current_count = after_count
                current_officers = after_officers
        
        # Summary
        final_count = len(df_filtered)
        final_officers = df_filtered.pid_pde.nunique()
        total_removed_rows = initial_count - final_count
        total_removed_officers = initial_officers - final_officers
        print_v(f"\n📊 Filtering Summary:")
        print_v(f"  Initial: {initial_count:,} rows ({initial_officers:,} officers)")
        print_v(f"  Final: {final_count:,} rows ({final_officers:,} officers)")
        print_v(f"  Total Removed: {total_removed_rows:,} rows ({total_removed_rows/initial_count*100:.2f}% of initial rows)")
        print_v(f"  Total Removed: {total_removed_officers:,} officers ({total_removed_officers/initial_officers*100:.2f}% of initial officers)")
        
        # Verify key columns
        key_vars = ['pid_pde', 'snpsht_dt', 'dor_cpt', 'sex', 'yg', TB_RATIO_RTR_COL, TB_RATIO_SNR_COL]
        missing_vars = [var for var in key_vars if var not in df_filtered.columns]
        if missing_vars:
            print_v(f"⚠️ Missing key variables: {missing_vars}")
        else:
            print_v(f"✅ All key variables present")
        
        # Save filtered data
        save_act = "Saving df_pipeline_09_filtered"
        save_timer = time_start(save_act, nest=1)
        store_feather(df_filtered, 'df_pipeline_09_filtered')
        time_stop(save_timer, action=save_act, nest=1)
        print_v(f"✅ Saved: df_pipeline_09_filtered.feather ({df_filtered.shape})")
        df_base = df_filtered.copy()
        
        time_stop(cell9, action=cell9_act, nest=0)
    else:
        print_v("By-passing CELL 9...")
        df_base = load_feather('df_pipeline_09_filtered')
    tyme()


In [ ]:
if null_reports:
    df_debug = load_feather('df_pipeline_09_filtered')
    df_null_report(df_debug)
df_debug =load_feather('df_pipeline_09_filtered')
'div_name' in df_debug.columns

# **📈 PHASE 5: COX ANALYSIS PREPARATION & EXECUTION**
## **CELL 10: Cox Data Preparation**
## **CELL 10.5: Standardize + Interactions**
## **CELL 11: Cox Analysis & Plotting**


# **📊 CELL 10: COX DATA PREPARATION**
##### **Prepare data for Cox regression analysis**
- **Time-to-Event**: Calculate time from DOR CPT to promotion/attrition/censoring
- **Event Indicators**: Promotion, attrition, censoring
- **Competing Risks**: Handle multiple event types
- **Output**: `df_pipeline_10_cox_ready` (base Cox-ready dataset)
- **Then (CELL 10.5)**: 'df_pipeline_10_5_cox_zscored' (standarized + interactions)
##### **Focus**: Create survival analysis dataset


In [ ]:
del df_base

In [ ]:
# === CELL 10: COX DATA PREPARATION ===
# Prepare time-varying survival data for Cox regression analysis with competing risks
# Creates survival intervals (one row per snapshot per officer) with time-varying covariates
# Reload pipeline_config to pick up any changes made 
# to configuration and other .py scripts
reload_pipeline_config()
with notebook_cell_logging('10'):
    if CELL10:
        # Define save directory for cox variables
        import os
        cox_var_dir = "./cox/cox_vars"
    
        # Create directory if it doesn't exist
        os.makedirs(cox_var_dir, exist_ok=True)
        
        print_v("\n📊 CELL 10: Preparing time-varying Cox data...")
        cell10_act = "CELL 10: Cox data preparation (time-varying)"
        cell10 = time_start(cell10_act, nest=0)
        
        # Load filtered data if not already loaded
        if 'df_base' not in locals():
            load_act = "Loading df_pipeline_09_filtered"
            load_timer = time_start(load_act, nest=1)
            df_base = load_feather('df_pipeline_09_filtered')
            time_stop(load_timer, action=load_act, nest=1)
        
        df_cox = df_base.copy()
        
        # Ensure required columns exist
        required_cols = ['pid_pde', 'snpsht_dt', 'dor_cpt', 'dor_maj', 'sex', 'yg']
        missing_cols = [col for col in required_cols if col not in df_cox.columns]
        if missing_cols:
            print_v(f"⚠️ ERROR: Missing required columns: {missing_cols}")
            print_v("Available columns:", list(df_cox.columns)[:30])
            raise ValueError(f"Missing required columns: {missing_cols}")
        else:
            print_v("✅ All required columns present")
        
        # === 10.1. CREATE OFFICER-LEVEL DATE SUMMARY ===
        print_v("\n🔧 Creating officer-level date summary...")
        officer_act = "Creating officer-level date summary"
        officer_timer = time_start(officer_act, nest=1)
        
        # Get unique officers with their key dates
        officer_dates = df_cox.groupby('pid_pde').agg({
            'sex': 'first',
            'yg': 'first',
            'age_cpt': 'first' if 'age_cpt' in df_cox.columns else None,
            'final_job_code': 'first' if 'final_job_code' in df_cox.columns else None,
            'dor_cpt': 'first',
            'dor_maj': 'max',
            'snpsht_dt': ['min', 'max']  # First and last snapshot dates
        }).reset_index()
        
        # Flatten column names
        officer_dates.columns = ['pid_pde', 'sex', 'yg', 'age_cpt', 'final_job_code',
                                 'dor_cpt', 'dor_maj', 'first_snapshot', 'last_snapshot']
        
        # DIAGNOSTIC: Check dor_maj values after aggregation
        # Remove None columns if they don't exist
        officer_dates = officer_dates[[col for col in officer_dates.columns if col is not None]]
        
        time_stop(officer_timer, action=officer_act, nest=1)
        print_v(f"✅ Officer dates created: {officer_dates.shape}")
        
        # === 10.2. CREATE TIME-VARYING SURVIVAL INTERVALS ===
        print_v("\n🔧 Creating survival intervals for each officer...")
        interval_act = "Creating survival intervals"
        interval_timer = time_start(interval_act, nest=1)
        
        # Study end date
        study_end_date = df_cox['snpsht_dt'].max()
        print_v(f"  • Study end date: {study_end_date}")
        
        # Sort snapshots by officer and date for efficient processing
        df_snapshots_sorted = df_cox.sort_values(by=['pid_pde', 'snpsht_dt']).reset_index(drop=True)
        # Create time-varying survival data
        survival_data_list = []
        
        # Identify scope for progress reporting
        total_pids = df_snapshots_sorted.pid_pde.nunique()
        report_portion = 20
        report_start = datetime.now()
        mile_stone = max(1, int(round(total_pids / report_portion)))
        print_v(f"  • Processing {total_pids:,} officers, reporting every {mile_stone:,} individuals")
        
        # Time-varying OER columns to include (from COLUMN_CONFIG)
        oer_cols = COLUMN_CONFIG.get('oer_variables', [])
        available_oer_cols = [col for col in oer_cols if col in df_snapshots_sorted.columns]
        print_v(
            f"  • Time-varying OER columns (from COLUMN_CONFIG): {len(available_oer_cols)} available \n({available_oer_cols})"
        )
        missing_oer = [c for c in oer_cols if c not in df_snapshots_sorted.columns]
        if missing_oer:
            print_v(f"⚠️ OER variables in config but missing from snapshot data: {missing_oer}")
            print_v("     → Re-run from CELL 4 (OER metrics) through CELL 9 so df_pipeline_09_filtered includes these columns.")
            
        other_tv_cols = [
            col for col in COLUMN_CONFIG.get('time_varying_cols', [])
            if col not in oer_cols
        ]
        # Snapshot of TV names from time_varying_cols only (before merging model lists)
        _from_tv_slice = set(other_tv_cols)
        # Interval-row carry-forward: model TV + optional extras (add new categorical TV cols in override lists, not here)
        _extra_sv = COLUMN_CONFIG.get('extra_survival_tv_cols', [])
        model_tv = list(dict.fromkeys(
            list(COLUMN_CONFIG.get('model_time_varying_cols', [])) + list(_extra_sv)
        ))
        _survival_tv_autofix = []
        for col in model_tv:
            if col in oer_cols:
                continue
            if col not in other_tv_cols:
                other_tv_cols.append(col)
                if col in df_snapshots_sorted.columns and col not in _from_tv_slice:
                    _survival_tv_autofix.append(col)
        if _survival_tv_autofix:
            print_v(
                "  • Survival TV carry-forward (in snapshots but omitted from COLUMN_CONFIG['time_varying_cols']):"
                f"{_survival_tv_autofix} — verify time_varying_cols aggregation in override; "
                f"pip_config_file={globals().get('pip_config_file', 'pipeline_config')!r}"
            )            

        available_other_tv_cols = [col for col in other_tv_cols if col in df_snapshots_sorted.columns]
        print_v(
            f"  • Other time-varying columns (time_varying_cols + model_time_varying_cols): {len(available_other_tv_cols)} available \n({available_other_tv_cols})"
        )            
        # Report model-only TV cols so div_name, etc. are explicitly visible (available vs missing in snapshot data)
        model_tv_available = [c for c in model_tv if c in df_snapshots_sorted.columns]
        model_tv_missing = [c for c in model_tv if c not in df_snapshots_sorted.columns]
        if model_tv_available:
            print_v(f"  • Model-only time-varying (in snapshot data): {model_tv_available}")
        if model_tv_missing:
            print_v(f"  ⚠️ Model-only time-varying missing from snapshot data: {model_tv_missing}")
            print_v(
                "     → Often expected: names like `z_*`, `*_sq`, and interaction columns are created in Cell 10.5 "
                "on the Cox-ready frame, not on `df_pipeline_09_filtered`. Run Cell 10.5 before fitting when "
                "`RUN_SCALE` / the model uses those columns (and load `df_pipeline_10_5_cox_zscored` in Cell 12 as configured). "
                "Use Cell 8/9 only if the column must exist on raw snapshots before Cell 10."
            )
        # Group by officer and process each group
        pid_count = 0; initial_report = False
        for pid, officer_snapshots in df_snapshots_sorted.groupby('pid_pde'):
            if pid_count == 0 and not initial_report:
                print_v(f"\n\n    ----  {pid_count:,} officers of {total_pids:,} complete ({(pid_count/total_pids):.0%}) at {tyme()}")
                initial_report = True
            officer_snapshots = officer_snapshots.reset_index(drop=True)
            
            # Get officer-level data
            officer_info = officer_dates[officer_dates['pid_pde'] == pid].iloc[0]
            
            # DIAGNOSTIC: Check this officer's dor_maj (first 5 officers only)
            # Skip if no dor_cpt
            if pd.isna(officer_info['dor_cpt']):
                continue
            
            # Filter out snapshots before dor_cpt (they shouldn't be in survival analysis)
            officer_snapshots = officer_snapshots[officer_snapshots['snpsht_dt'] >= officer_info['dor_cpt']].copy()
    
            # Filter out snapshots after the event date (for promoted officers, remove snapshots after dor_maj)
            if pd.notna(officer_info['dor_maj']):
                # Promoted - remove snapshots after promotion date
                officer_snapshots = officer_snapshots[officer_snapshots['snpsht_dt'] <= officer_info['dor_maj']].copy()
            elif officer_info['last_snapshot'] < study_end_date:
                # Attrited - remove snapshots after last snapshot
                officer_snapshots = officer_snapshots[officer_snapshots['snpsht_dt'] <= officer_info['last_snapshot']].copy()
            # For censored, keep all snapshots up to study_end_date
    
            # Reset index after filtering
            officer_snapshots = officer_snapshots.reset_index(drop=True)
                
            # Create intervals between snapshots
            n_snapshots = len(officer_snapshots)
            
            if n_snapshots == 0:
                continue
            
            # Create start times (days from dor_cpt)
            start_times = np.zeros(n_snapshots)
            if n_snapshots > 1:
                start_times[1:] = (officer_snapshots['snpsht_dt'].iloc[1:] - officer_info['dor_cpt']).dt.days
            # For last interval, start_time should be the last snapshot date
            # Ensure we get scalar values to avoid Series issues
            last_snapshot_dt  = pd.to_datetime(officer_snapshots['snpsht_dt'].iloc[-1])
            dor_cpt_dt = pd.to_datetime(officer_info['dor_cpt'])
            start_times[-1] = (last_snapshot_dt - dor_cpt_dt).days
            
            # Create end times
            end_times = np.zeros(n_snapshots)
            
            # For all but last snapshot: end at next snapshot
            if n_snapshots > 1:
                end_times[:-1] = (officer_snapshots['snpsht_dt'].iloc[1:] - officer_info['dor_cpt']).dt.days
            
            # For last snapshot: determine based on competing event
            # Ensure dor_cpt is a scalar datetime for calculations
            dor_cpt_dt = pd.to_datetime(officer_info['dor_cpt'])
            
            if pd.notna(officer_info['dor_maj']):
                # Promoted - end at promotion date
                dor_maj_dt = pd.to_datetime(officer_info['dor_maj'])
                end_times[-1] = (dor_maj_dt - dor_cpt_dt).days
                events = np.zeros(n_snapshots)
                events[-1] = 1  # Promoted
            elif officer_info['last_snapshot'] < study_end_date:
                # Attrited - end at last snapshot
                last_snapshot_dt = pd.to_datetime(officer_info['last_snapshot'])
                end_times[-1] = (last_snapshot_dt - dor_cpt_dt).days
                events = np.zeros(n_snapshots)
                events[-1] = 2  # Attrited
            else:
                # Censored - end at study end
                study_end_dt = pd.to_datetime(study_end_date)
                end_times[-1] = (study_end_dt - dor_cpt_dt).days
                events = np.zeros(n_snapshots)
                events[-1] = 0  # Censored
            
            # DIAGNOSTIC: Check what event ws assigned (first 5 officers only)
            # Create survival records with time-varying covariates
            survival_dict = {
                'pid_pde': pid,
                'start_time': start_times,
                'stop_time': end_times,
                'event': events,
                # Static covariates (repeated for each interval)
                'sex': officer_info['sex'],
                'yg': officer_info['yg'],
                # Time-varying covariates from snapshots
                'snpsht_dt': officer_snapshots['snpsht_dt'],
            }
            
            # Add optional static columns if they exist
            if 'age_cpt' in officer_info.index and pd.notna(officer_info.get('age_cpt')):
                survival_dict['age_cpt'] = officer_info['age_cpt']
            if 'final_job_code' in officer_info.index and pd.notna(officer_info.get('final_job_code')):
                survival_dict['final_job_code'] = officer_info['final_job_code']
            
            # Add time-varying OER columns
            for col in available_oer_cols:
                if col in officer_snapshots.columns:
                    survival_dict[col] = officer_snapshots[col].values
            
            # Add other time-varying columns if they exist
            string_fill_cols = {'div_name', 'asg_uic_pde', 'rank_pde', 'job_code'}
            for col in available_other_tv_cols:
                if col in officer_snapshots.columns:
                    fill_value = 'Unknown' if col in string_fill_cols else 0
                    if col == 'div_name':
                        miss_before = int(officer_snapshots[col].isna().sum())
                        print_v(f"[DIV_DEBUG] Missing div_name BEFORE fill in Cell 10: {miss_before:,}/{len(officer_snapshots):,}")
                    survival_dict[col] = officer_snapshots[col].fillna(fill_value).values
            
            # Create DataFrame for this officer
            survival_records = pd.DataFrame(survival_dict)
            
            # DIAGNOSTIC: Check the last interval before filtering (first 20 officers only)
            # Filter out invalid intervals (where stop_time <= start_time)
            valid_mask = survival_records['stop_time'] >= survival_records['start_time']
            survival_records = survival_records[valid_mask].copy()
            
            # DIAGNOSTIC: Check if last interval (with event) was filtered out (first 5 officers only)
            if len(survival_records) > 0:
                survival_data_list.append(survival_records)
            
            pid_count += 1
            if pid_count % mile_stone == 0 or pid_count == total_pids:
                elapsed = (datetime.now() - report_start).total_seconds()
                avg_secs = elapsed / max(pid_count, 1)
                remaining_secs = (total_pids - pid_count) * avg_secs
                eta = datetime.now() + timedelta(seconds=remaining_secs)
                print_v(
                    f"    ----  {pid_count:,} officers of {total_pids:,} complete ({(pid_count/total_pids):.0%}) at {tyme()}"
                    f"  |  ETA ~ {eta:%H:%M} ({remaining_secs/60:.1f} min)"
                )
        
        # Concatenate all survival records
        df_survival = pd.concat(survival_data_list, ignore_index=True)
        
        time_stop(interval_timer, action=interval_act, nest=1)
        print_v(f"✅ Survival intervals created: {df_survival.shape}")
        print_v(f"📊 Intervals per officer: {df_survival.groupby('pid_pde').size().describe()}")
        
        # === 10.3. VERIFY SURVIVAL DATA ===
        print_v("\n📊 Survival Data Verification:")
        verify_act = "Verifying survival data"
        verify_timer = time_start(verify_act, nest=1)
        
        # Check event distribution (one event per officer)
        event_counts = df_survival[['pid_pde', 'event']].drop_duplicates()['event'].value_counts().sort_index()
        event_pcts = df_survival[['pid_pde', 'event']].drop_duplicates()['event'].value_counts(normalize=True).sort_index()
        
        event_names = {0: 'Censored', 1: 'Promoted', 2: 'Attrited'}
        for event, count in event_counts.items():
            pct = event_pcts[event]
            print_v(f"  • {event_names[event]}: {count:,} officers ({pct:.1%})")
        
        # Check interval lengths
        df_survival['interval_length'] = df_survival['stop_time'] - df_survival['start_time']
        print_v(f"\n📊 Interval Statistics:")
        print_v(f"  • Mean interval length: {df_survival['interval_length'].mean():.1f} days")
        print_v(f"  • Median interval length: {df_survival['interval_length'].median():.1f} days")
        print_v(f"  • Max interval length: {df_survival['interval_length'].max():.1f} days")
        
        # Check time-varying OER columns
        if available_oer_cols:
            print_v(f"\n📊 Time-Varying OER Columns:")
            for col in available_oer_cols:
                non_null = df_survival[col].notna().sum()
                pct = (non_null / len(df_survival)) * 100
                print_v(f"  • {col}: {non_null:,} non-null ({pct:.1f}%)")
        
        time_stop(verify_timer, action=verify_act, nest=1)
            
        # === 10.4. SAVE RESULTS ===
        save_act = "Saving df_pipeline_10_cox_ready"
        save_timer = time_start(save_act, nest=1)
        store_feather(df_survival, 'df_pipeline_10_cox_ready')
        time_stop(save_timer, action=save_act, nest=1)
        
        df_cox = df_survival.copy()
        
        print_v("\n✅ Time-varying survival data ready for Cox analysis!")
        print_v("🎯 Next: Cox analysis with time-varying covariates")
        
        time_stop(cell10, action=cell10_act, nest=0)
    else:
        print_v("By-passing CELL 10...")
        df_cox = load_feather('df_pipeline_10_cox_ready')
    if null_reports:
        df_null_report(df_debug)
    tyme()


In [ ]:
'div_name' in df_cox.columns.tolist()

In [ ]:
# del df_cox

# **🧮 CELL 10.5: STANDARDIZE + INTERACTIONS**
##### **Z-score creation + interaction terms on Cox-ready data**
- Creates `z_` columns based on `STANDARDIZE_CONFIG`
- Creates quadratic terms from `QUADRATIC_CONFIG`
- Builds interaction terms from `INTERACTION_CONFIG`
- Saves to `df_pipeline_10_5_cox_zscored`

**When to use 10.5 vs 10.5S**
- Use **10.5** for standard z-scores + quadratics + interactions (current default)
- Use **10.5S** if you want spline basis terms (controlled by `SPLINE_CONFIG`)


In [ ]:
# === CELL 10.5: STANDARDIZE + INTERACTIONS ===
# Creates z-score columns and interaction terms on Cox-ready data
reload_pipeline_config()
with notebook_cell_logging('10_5'):
    if 'df_cox' not in locals():
        df_cox = load_feather('df_pipeline_10_cox_ready')

    # === 10.5.1. Z-SCORE CREATION ===
    if STANDARDIZE_CONFIG.get('enabled', False):
        prefix = STANDARDIZE_CONFIG.get('prefix', 'z_')
        base_cols = STANDARDIZE_CONFIG.get('cols') or (
            COLUMN_CONFIG.get('model_static_cols', []) + COLUMN_CONFIG.get('model_time_varying_cols', [])
        )
        wins = STANDARDIZE_CONFIG.get('winsorize', {})
        created = []
        skipped = []

        for col in base_cols:
            if col not in df_cox.columns:
                skipped.append(col)
                continue
            series = pd.to_numeric(df_cox[col], errors='coerce')
            if wins.get('enabled', False):
                lower_q = wins.get('lower_q', 0.01)
                upper_q = wins.get('upper_q', 0.99)
                lo = series.quantile(lower_q)
                hi = series.quantile(upper_q)
                series = series.clip(lower=lo, upper=hi)
            mean = series.mean()
            std = series.std(ddof=0)
            z_col = f"{prefix}{col}"
            if std and std > 0:
                df_cox[z_col] = (series - mean) / std
            else:
                df_cox[z_col] = pd.NA
            created.append(z_col)

        if created:
            print_v(f"✅ Created z-score columns: {created}")
        if skipped:
            print_v(f"⚠️ Skipped (missing in df_cox): {skipped}")
    else:
        print_v("By-passing z-score creation (STANDARDIZE_CONFIG['enabled']=False)")

    prefix = STANDARDIZE_CONFIG.get('prefix', 'z_')

    # === 10.5.2. QUADRATIC TERMS ===
    created_quads = []
    skipped_quads = []
    quad_cfg = QUADRATIC_CONFIG or {}
    quad_wins = quad_cfg.get('winsorize') or STANDARDIZE_CONFIG.get('winsorize', {})
    quad_terms = list(quad_cfg.get('terms', []) or [])
    quad_bases = list(quad_cfg.get('bases', []) or [])
    default_q_z = quad_cfg.get('default_zscore', False)

    if quad_cfg.get('enabled', False):
        if not quad_terms and quad_bases:
            quad_terms = [
                {'base': b, 'name': f"{b}_sq", 'zscore': default_q_z}
                for b in quad_bases
            ]

        for term in quad_terms:
            name = term.get('name')
            base = term.get('base')
            if not base:
                continue
            if base not in df_cox.columns:
                skipped_quads.append({'name': name, 'base': base})
                continue
            if not name:
                name = f"{base}_sq"
            series = pd.to_numeric(df_cox[base], errors='coerce')
            df_cox[name] = series ** 2
            created_quads.append(name)

            if term.get('zscore', False):
                z_series = pd.to_numeric(df_cox[name], errors='coerce')
                if quad_wins.get('enabled', False):
                    lower_q = quad_wins.get('lower_q', 0.01)
                    upper_q = quad_wins.get('upper_q', 0.99)
                    lo = z_series.quantile(lower_q)
                    hi = z_series.quantile(upper_q)
                    z_series = z_series.clip(lower=lo, upper=hi)
                mean = z_series.mean()
                std = z_series.std(ddof=0)
                z_name = f"{prefix}{name}"
                if std and std > 0:
                    df_cox[z_name] = (z_series - mean) / std
                else:
                    df_cox[z_name] = pd.NA
                created_quads.append(z_name)

        if created_quads:
            print_v(f"✅ Created quadratic columns: {created_quads}")
        if skipped_quads:
            print_v(f"⚠️ Skipped quadratic terms (missing inputs): {skipped_quads}")
    else:
        print_v("By-passing quadratic terms (QUADRATIC_CONFIG['enabled']=False)")

    # === 10.5.3. INTERACTION TERMS ===
    created_terms = []
    missing_terms = []
    wins = STANDARDIZE_CONFIG.get('winsorize', {})

    for term in INTERACTION_CONFIG.get('terms', []):
        name = term.get('name')
        left = term.get('left')
        right = term.get('right')
        if not name or not left or not right:
            continue
        if left not in df_cox.columns or right not in df_cox.columns:
            missing_terms.append({
                'name': name,
                'left': left,
                'right': right
            })
            continue
        df_cox[name] = pd.to_numeric(df_cox[left], errors='coerce') * pd.to_numeric(df_cox[right], errors='coerce')
        created_terms.append(name)

        if term.get('zscore', False):
            series = pd.to_numeric(df_cox[name], errors='coerce')
            if wins.get('enabled', False):
                lower_q = wins.get('lower_q', 0.01)
                upper_q = wins.get('upper_q', 0.99)
                lo = series.quantile(lower_q)
                hi = series.quantile(upper_q)
                series = series.clip(lower=lo, upper=hi)
            mean = series.mean()
            std = series.std(ddof=0)
            z_name = f"{prefix}{name}"
            if std and std > 0:
                df_cox[z_name] = (series - mean) / std
            else:
                df_cox[z_name] = pd.NA
            created_terms.append(z_name)

    if created_terms:
        print_v(f"✅ Created interaction columns: {created_terms}")
    if missing_terms:
        print_v(f"⚠️ Skipped interactions (missing inputs): {missing_terms}")

    # Save updated Cox-ready dataset
    
    save_act = "Saving df_pipeline_10_5_cox_zscored (standardized)"
    save_timer = time_start(save_act, nest=1)
    store_feather(df_cox, 'df_pipeline_10_5_cox_zscored')
    time_stop(save_timer, action=save_act, nest=1)
    print_v("✅ Saved: df_pipeline_10_5_cox_zscored.feather (with z_ + interactions)")


# **🧩 CELL 10.5S: STANDARDIZE + SPLINES + INTERACTIONS (FUTURE)**
##### **Alternate 10.5 with spline support; disabled by default**
- Same as Cell 10.5, plus optional spline basis creation
- Controlled by `SPLINE_CONFIG` (disabled by default)
- Keeps existing 10.5 intact so you can switch later


In [ ]:
# === CELL 10.5S: STANDARDIZE + SPLINES + INTERACTIONS (FUTURE) ===
# Same as Cell 10.5, with optional spline basis creation
reload_pipeline_config()
with notebook_cell_logging('10_5S'):
    if CELL10_5S:
        if 'df_cox' not in locals():
            df_cox = load_feather('df_pipeline_10_cox_ready')
    
        # === 10.5S.1. Z-SCORE CREATION ===
        if STANDARDIZE_CONFIG.get('enabled', False):
            prefix = STANDARDIZE_CONFIG.get('prefix', 'z_')
            base_cols = STANDARDIZE_CONFIG.get('cols') or (
                COLUMN_CONFIG.get('model_static_cols', []) + COLUMN_CONFIG.get('model_time_varying_cols', [])
            )
            wins = STANDARDIZE_CONFIG.get('winsorize', {})
            created = []
            skipped = []
    
            for col in base_cols:
                if col not in df_cox.columns:
                    skipped.append(col)
                    continue
                series = pd.to_numeric(df_cox[col], errors='coerce')
                if wins.get('enabled', False):
                    lower_q = wins.get('lower_q', 0.01)
                    upper_q = wins.get('upper_q', 0.99)
                    lo = series.quantile(lower_q)
                    hi = series.quantile(upper_q)
                    series = series.clip(lower=lo, upper=hi)
                mean = series.mean()
                std = series.std(ddof=0)
                z_col = f"{prefix}{col}"
                if std and std > 0:
                    df_cox[z_col] = (series - mean) / std
                else:
                    df_cox[z_col] = pd.NA
                created.append(z_col)
    
            if created:
                print_v(f"✅ Created z-score columns: {created}")
            if skipped:
                print_v(f"⚠️ Skipped (missing in df_cox): {skipped}")
        else:
            print_v("By-passing z-score creation (STANDARDIZE_CONFIG['enabled']=False)")
    
        prefix = STANDARDIZE_CONFIG.get('prefix', 'z_')
    
        # === 10.5S.2. QUADRATIC TERMS ===
        created_quads = []
        skipped_quads = []
        quad_cfg = QUADRATIC_CONFIG or {}
        quad_wins = quad_cfg.get('winsorize') or STANDARDIZE_CONFIG.get('winsorize', {})
        quad_terms = list(quad_cfg.get('terms', []) or [])
        quad_bases = list(quad_cfg.get('bases', []) or [])
        default_q_z = quad_cfg.get('default_zscore', False)
    
        if quad_cfg.get('enabled', False):
            if not quad_terms and quad_bases:
                quad_terms = [
                    {'base': b, 'name': f"{b}_sq", 'zscore': default_q_z}
                    for b in quad_bases
                ]
    
            for term in quad_terms:
                name = term.get('name')
                base = term.get('base')
                if not base:
                    continue
                if base not in df_cox.columns:
                    skipped_quads.append({'name': name, 'base': base})
                    continue
                if not name:
                    name = f"{base}_sq"
                series = pd.to_numeric(df_cox[base], errors='coerce')
                df_cox[name] = series ** 2
                created_quads.append(name)
    
                if term.get('zscore', False):
                    z_series = pd.to_numeric(df_cox[name], errors='coerce')
                    if quad_wins.get('enabled', False):
                        lower_q = quad_wins.get('lower_q', 0.01)
                        upper_q = quad_wins.get('upper_q', 0.99)
                        lo = z_series.quantile(lower_q)
                        hi = z_series.quantile(upper_q)
                        z_series = z_series.clip(lower=lo, upper=hi)
                    mean = z_series.mean()
                    std = z_series.std(ddof=0)
                    z_name = f"{prefix}{name}"
                    if std and std > 0:
                        df_cox[z_name] = (z_series - mean) / std
                    else:
                        df_cox[z_name] = pd.NA
                    created_quads.append(z_name)
    
            if created_quads:
                print_v(f"✅ Created quadratic columns: {created_quads}")
            if skipped_quads:
                print_v(f"⚠️ Skipped quadratic terms (missing inputs): {skipped_quads}")
        else:
            print_v("By-passing quadratic terms (QUADRATIC_CONFIG['enabled']=False)")
    
        # === 10.5S.3. SPLINE TERMS (FUTURE) ===
        created_splines = []
        skipped_splines = []
        spline_cfg = SPLINE_CONFIG or {}
        spline_wins = spline_cfg.get('winsorize') or STANDARDIZE_CONFIG.get('winsorize', {})
    
        if spline_cfg.get('enabled', False):
            try:
                from patsy import dmatrix
    
                for term in spline_cfg.get('terms', []):
                    base = term.get('base')
                    if not base:
                        continue
                    if base not in df_cox.columns:
                        skipped_splines.append({'base': base})
                        continue
    
                    df_val = int(term.get('df', 4))
                    prefix_s = term.get('prefix', 'spline_')
                    use_z = bool(term.get('use_z', False))
    
                    series = pd.to_numeric(df_cox[base], errors='coerce')
                    if spline_wins.get('enabled', False):
                        lower_q = spline_wins.get('lower_q', 0.01)
                        upper_q = spline_wins.get('upper_q', 0.99)
                        lo = series.quantile(lower_q)
                        hi = series.quantile(upper_q)
                        series = series.clip(lower=lo, upper=hi)
    
                    tmp = pd.DataFrame({'x': series})
                    spline_df = dmatrix(
                        f"bs(x, df={df_val}, include_intercept=False)",
                        tmp,
                        return_type='dataframe'
                    )
    
                    for i, col in enumerate(spline_df.columns, start=1):
                        out_col = f"{prefix_s}{base}_s{i}"
                        df_cox[out_col] = spline_df[col].to_numpy()
                        created_splines.append(out_col)
    
                        if use_z:
                            z_series = pd.to_numeric(df_cox[out_col], errors='coerce')
                            mean = z_series.mean()
                            std = z_series.std(ddof=0)
                            z_name = f"{prefix}{out_col}"
                            if std and std > 0:
                                df_cox[z_name] = (z_series - mean) / std
                            else:
                                df_cox[z_name] = pd.NA
                            created_splines.append(z_name)
    
                if created_splines:
                    print_v(f"✅ Created spline columns: {created_splines}")
                if skipped_splines:
                    print_v(f"⚠️ Skipped spline terms (missing inputs): {skipped_splines}")
            except Exception as e:
                print_v(f"⚠️ Spline creation skipped (patsy missing or error): {e}")
        else:
            print_v("By-passing spline terms (SPLINE_CONFIG['enabled']=False)")
    
        # === 10.5S.4. INTERACTION TERMS ===
        created_terms = []
        missing_terms = []
        wins = STANDARDIZE_CONFIG.get('winsorize', {})
    
        for term in INTERACTION_CONFIG.get('terms', []):
            name = term.get('name')
            left = term.get('left')
            right = term.get('right')
            if not name or not left or not right:
                continue
            if left not in df_cox.columns or right not in df_cox.columns:
                missing_terms.append({
                    'name': name,
                    'left': left,
                    'right': right
                })
                continue
            df_cox[name] = pd.to_numeric(df_cox[left], errors='coerce') * pd.to_numeric(df_cox[right], errors='coerce')
            created_terms.append(name)
    
            if term.get('zscore', False):
                series = pd.to_numeric(df_cox[name], errors='coerce')
                if wins.get('enabled', False):
                    lower_q = wins.get('lower_q', 0.01)
                    upper_q = wins.get('upper_q', 0.99)
                    lo = series.quantile(lower_q)
                    hi = series.quantile(upper_q)
                    series = series.clip(lower=lo, upper=hi)
                mean = series.mean()
                std = series.std(ddof=0)
                z_name = f"{prefix}{name}"
                if std and std > 0:
                    df_cox[z_name] = (series - mean) / std
                else:
                    df_cox[z_name] = pd.NA
                created_terms.append(z_name)
    
        if created_terms:
            print_v(f"✅ Created interaction terms: {created_terms}")
        if missing_terms:
            print_v(f"⚠️ Missing interaction inputs: {missing_terms}")
    
        # Save output
        store_feather(df_cox, 'df_pipeline_10_5_cox_zscored')
        print_v("✅ Saved: df_pipeline_10_5_cox_zscored")
        tyme()
    else:
        print_v("By-passing CELL 10.5S...")
    tyme()


# **📈 CELL 11: COX ANALYSIS & PLOTTING**
##### **Cox Regression Analysis and Visualization**
- **Cox Models**: Fit Cox proportional hazards models
- **Plotting**: Kaplan-Meier curves, survival functions, competing risks
- **Analysis**: Variable effects, hazard ratios, promotion probabilities
##### **Focus**: Analyze promotion timing using Cox regression
**Note**: Port the Cox analysis cells from `513_marry_cox_with_perf_models_working.ipynb` starting from the Cox model fitting cells.


In [ ]:
# del df_cox

In [ ]:
# === CELL 11: COX ANALYSIS & PLOTTING ===
# Flexible configuration-based Cox regression analysis and visualization
# Works with time-varying survival interval data
# Reload pipeline_config to pick up any changes made 
# to configuration and other .py scripts
reload_pipeline_config()
with notebook_cell_logging('11'):
    if CELL11:
        print_v("\n📈 CELL 11: Cox Analysis & Plotting")
        cell11_act = "CELL 11: Cox analysis and plotting"
        cell11 = time_start(cell11_act, nest=0)
        
        # Import plot helper functions
        from cox_plot_helpers import (
            generate_plot_filename, get_plot_metadata, format_plot_config_text,
            format_plot_title, format_plot_subtitle, format_legend_label,
            add_configuration_box, save_plot_metadata_card,
            plot_competing_risks_cif_bars,
            apply_unknown_group_by_filter,
            cr_discrete_group_colors,
        )
        
        # Load Cox-ready data
        if 'df_cox' not in locals():
            load_act = "Loading df_pipeline_10_5_cox_zscored"
            load_timer = time_start(load_act, nest=1)
            df_cox = load_feather('df_pipeline_10_5_cox_zscored')
            time_stop(load_timer, action=load_act, nest=1)
        
        print_v(f"📊 Cox-ready data: {df_cox.shape}")
        print_v(f"📊 Data structure: Time-varying survival intervals")
        print_v(f"  • Intervals: {len(df_cox):,}")
        print_v(f"  • Officers: {df_cox['pid_pde'].nunique():,}")
        
        # === 11.1. COLUMN SELECTION CONFIGURATION ===
        # COLUMN_CONFIG is imported from pipeline_config.py
        # Edit pipeline_config.py to modify column selection settings
        # The COLUMN_CONFIG dictionary contains:
        #   - basic_demographic_cols: Required columns for survival analysis
        #   - static_cols: Columns that don't change over time (one value per officer)
        #   - time_varying_cols: Columns that can change over time (values per snapshot)
        #   - oer_variables: List of OER-related variables (for filtering purposes)
        #   - dummy_variables: Configuration for creating dummy variables from categorical columns
        print_v("\n🔧 Setting up column selection configuration...")
        col_config_act = "Setting up column selection"
        col_config_timer = time_start(col_config_act, nest=1)
        
        # COLUMN_CONFIG is already imported from pipeline_config.py in CELL 0
        # No need to redefine it here - just use the imported version
        
        # Save COLUMN_CONFIG to a JSON file (for reference/debugging)
        store_json(COLUMN_CONFIG,'COLUMN_CONFIG',var_dir)
        
        # === 11.2. PREPARE ANALYSIS DATASET ===
        print_v("\n🔧 Preparing analysis dataset with selected columns...")
        prep_act = "Preparing analysis dataset"
        prep_timer = time_start(prep_act, nest=1)
        
        # Start with basic demographic columns
        analysis_cols = COLUMN_CONFIG['basic_demographic_cols'].copy()
        
        # Add static columns (if they exist in the data)
        missing_static = []
        for col in COLUMN_CONFIG['static_cols']:
            if col in df_cox.columns:
                analysis_cols.append(col)
            else:
                missing_static.append(col)
        
        if missing_static:
            print_v(f"⚠️ Static columns not found (will be skipped): {missing_static}")
        
        # Add time-varying columns (if they exist in the data)
        missing_tv = []
        for col in COLUMN_CONFIG['time_varying_cols']:
            if col in df_cox.columns:
                analysis_cols.append(col)
            else:
                missing_tv.append(col)
        
        if missing_tv:
            print_v(f"⚠️ Time-varying columns not found (will be skipped): {missing_tv}")
        
        # Add model time-varying columns (interactions, quadratics) so they exist in df_analysis for plotting
        model_tv = COLUMN_CONFIG.get('model_time_varying_cols', [])
        for col in model_tv:
            if col in df_cox.columns and col not in analysis_cols:
                analysis_cols.append(col)
        added_model_tv = [c for c in model_tv if c in df_cox.columns and c not in COLUMN_CONFIG['time_varying_cols']]
        if added_model_tv:
            print_v(f"  • Added model time-varying columns for plotting: {added_model_tv}")
            
        # Option A: auto-incude z_ columns created in Cell 11
        if PLOT_CONFIG.get('include_z_cols', False):
            prefix = STANDARDIZE_CONFIG.get('prefix', 'z_')
            model_static = COLUMN_CONFIG.get('model_static_cols', [])
            model_tv = COLUMN_CONFIG.get('model_time_varying_cols', [])
            base_cols = STANDARDIZE_CONFIG.get('cols') or (model_static + model_tv)
            static_z = []
            tv_z = []
            for base in base_cols:
                z_col = f"{prefix}{base}"
                if z_col not in df_cox.columns:
                    continue
                if base in COLUMN_CONFIG['static_cols'] or base in model_static:
                    static_z.append(z_col)
                elif base in COLUMN_CONFIG['time_varying_cols'] or base in model_tv:
                    tv_z.append(z_col)
                else:
                    tv_z.append(z_col)
            # Add z-score interaction columns (always time-varying)
            for term in INTERACTION_CONFIG.get('terms', []):
                if term.get('zscore', False):
                    name = term.get('name')
                    if name:
                        z_name = f"{prefix}{name}"
                        if z_name  in df_cox.columns:
                            tv_z.append(z_name)
            # De-duplicate and append
            for z_col in sorted(set(static_z)):
                if z_col not in analysis_cols:
                    analysis_cols.append(z_col)
            for z_col in sorted(set(tv_z)):
                if z_col not in analysis_cols:
                    analysis_cols.append(z_col)
            if static_z or tv_z:
                print_v(f"  • Added z_ static columns: {sorted(set(static_z))}")
                print_v(f"  • Added z_ time-varying columns: {sorted(set(tv_z))}")
        
        # Create analysis dataset with only selected columns
        df_analysis = df_cox[analysis_cols].copy()
        
        # Report what we have
        print_v(f"\n📊 Analysis Dataset Summary:")
        print_v(f"  • Total columns: {len(df_analysis.columns):,}")
        # print_v(f"Columns:{df_analysis.columns}")
        print_v(f"  • Basic demographic: {len(COLUMN_CONFIG['basic_demographic_cols'])}")
        print_v(f"  • Static columns: {len([c for c in COLUMN_CONFIG['static_cols'] if c in df_analysis.columns])}")
        print_v(f"  • Time-varying columns: {len([c for c in COLUMN_CONFIG['time_varying_cols'] if c in df_analysis.columns])}")
        print_v(f"\n📊 Available columns for analysis:")
        print_v(f"  • Static: {len([c for c in COLUMN_CONFIG['static_cols'] if c in df_analysis.columns])}")
        print_v(f"  • Time-varying: {len([c for c in COLUMN_CONFIG['time_varying_cols'] if c in df_analysis.columns])}")
        
        time_stop(prep_timer, action=prep_act, nest=1)
        time_stop(col_config_timer, action=col_config_act, nest=1)
        
        # === 11.3. CREATE DUMMY VARIABLES ===
        # Convert categorical variables to dummy variables for Cox regression
        if COLUMN_CONFIG['dummy_variables']['create_dummies']:
            print_v("\n🔧 Creating dummy variables for categorical columns...")
            dummy_act = "Creating dummy variables"
            dummy_timer = time_start(dummy_act, nest=1)
            
            dummy_config = COLUMN_CONFIG['dummy_variables']
            exclude_ref = dummy_config['exclude_reference']
            categorical_cols = dummy_config['categorical_cols']
            
            dummy_vars_created = []
            for col_name, col_config in categorical_cols.items():
                if col_name not in df_analysis.columns:
                    print_v(f"⚠️ Categorical column '{col_name}' not found in analysis dataset, skipping")
                    continue
                
                # Get column data
                col_data = df_analysis[col_name]
                
                # Handle missing values (fill with 'Unknown' for categorical)
                if col_data.isnull().any():
                    print_v(f"  • Filling missing values in '{col_name}' with 'Unknown'")
                    if col_name == 'yg':
                        continue
                    col_data = col_data.fillna('Unknown')
                    df_analysis[col_name] = col_data
                
                # Determine reference category
                ref_category = col_config.get('reference', None)
                if ref_category is None:
                    # Use most common category as reference
                    ref_category = col_data.mode().iloc[0] if len(col_data.mode()) > 0 else col_data.iloc[0]
                    print_v(f"  • Using most common category as reference for '{col_name}': {ref_category}")
                else:
                    print_v(f"  • Using specified reference category for '{col_name}': {ref_category}")
                
                # Get prefix
                prefix = col_config.get('prefix', col_name)
                
                # Create dummy variables
                dummies = pd.get_dummies(col_data, prefix=prefix, drop_first=exclude_ref)
                
                # If exclude_reference=False, we still need to handle the reference category
                if not exclude_ref:
                    # Remove the reference category dummy if it exists
                    ref_dummy_col = f"{prefix}_{ref_category}"
                    if ref_dummy_col in dummies.columns:
                        dummies = dummies.drop(columns=[ref_dummy_col])
                        print_v(f"  • Dropped reference category dummy: {ref_dummy_col}")
                
                # Add dummy variables to dataframe
                df_analysis = pd.concat([df_analysis, dummies], axis=1)
                dummy_vars_created.extend(list(dummies.columns))
                
                print_v(f"  ✅ Created {len(dummies.columns)} dummy variables for '{col_name}': {list(dummies.columns)}")
            
            # Optionally drop original categorical columns (keep them for now for plotting)
            # Uncomment the following lines if you want to remove original categorical columns:
            # for col_name in categorical_cols.keys():
            #     if col_name in df_analysis.columns:
            #         df_analysis = df_analysis.drop(columns=[col_name])
            #         print_v(f"  • Dropped original categorical column: {col_name}")
            
            print_v(f"\n📊 Dummy Variable Summary:")
            print_v(f"  • Total dummy variables created: {len(dummy_vars_created)}")
            print_v(f"  • Dummy variable columns: {dummy_vars_created[:10]}{'...' if len(dummy_vars_created) > 10 else ''}")
            
            time_stop(dummy_timer, action=dummy_act, nest=1)
        else:
            print_v("\n⏭️ Skipping dummy variable creation (create_dummies=False)")
        
        # === 11.4. PLOTTING CONFIGURATION ===
        # PLOT_CONFIG is imported from pipeline_config.py
        # Edit pipeline_config.py to modify plot settings and add/remove plots
        # The PLOT_CONFIG dictionary contains:
        #   - plot_dir: Directory where plots will be saved
        #   - figsize: Figure size (width, height)
        #   - dpi: Resolution for saved plots
        #   - font_size: Base font size (None = auto-calculate)
        #   - reference_lines: Settings for promotion zone reference lines
        #   - plots: List of plot specifications (each plot is a dictionary)
        print_v("\n🔧 Setting up plotting configuration...")
        config_act = "Setting up plotting configuration"
        config_timer = time_start(config_act, nest=1)
        
        # PLOT_CONFIG is already imported from pipeline_config.py in CELL 0
        # No need to redefine it here - just use the imported version
        
        # Create plot directory if it doesn't exist
        import os
        os.makedirs(PLOT_CONFIG['plot_dir'], exist_ok=True)
        
        time_stop(config_timer, action=config_act, nest=1)
        print_v(f"✅ Configured {len(PLOT_CONFIG['plots'])} plots")
        
        # === 11.5. ENHANCED PLOTTING UTILITIES ===
        # Ported from 513 Cell 9: Enhanced visualization functions
        print_v("\n🔧 Setting up enhanced plotting utilities...")
        util_act = "Setting up enhanced plotting utilities"
        util_timer = time_start(util_act, nest=1)
        
        # Reference line colors
        c1 = 'gray'  # Primary zone line color
        c2 = 'xkcd:golden rod'      # Primary zone label color
        c3 = 'black' # Above zone line color
        c4 = 'xkcd:hazel'      # Above zone label color
        
        def get_plot_scales(fig_sz, fnt_sz=None, show=True):
            """
            Get plot scale relationships (proportional font sizes)
            Ported from 513 Cell 9
            
            Parameters:
            -----------
            fig_sz : tuple - Figure size (width, height)
            fnt_sz : float or None - Base font size (if None, auto-calculates from figsize)
            show : bool - Print scale values
            
            Returns:
            --------
            tuple: (fnt_sz, suptitl_sz, title_sz, axis_sz, ref_ln_label_sz, tick_sz, lgnd_sz)
            """
            if not fnt_sz:
                fnt_sz = (fig_sz[0] * fig_sz[1] / 8) + 4
                if show:
                    print_v(f"  • Auto-calculated font size from figsize: {fnt_sz:.1f}")
            
            suptitl_sz = fnt_sz * 1.5
            title_sz = fnt_sz
            axis_sz = fnt_sz * 0.888
            ref_ln_label_sz = fnt_sz * 0.4
            tick_sz = fnt_sz * 0.5
            lgnd_sz = fnt_sz * 0.5
            
            if show:
                print_v(f"  • Font scales: fnt={fnt_sz:.0f}, title={title_sz:.0f}, axis={axis_sz:.0f}, tick={tick_sz:.0f}, legend={lgnd_sz:.0f}")
            
            return fnt_sz, suptitl_sz, title_sz, axis_sz, ref_ln_label_sz, tick_sz, lgnd_sz
        
        def insert_pz_line(ax, pz_ref_line, START_TIME, fnt_sz, ref_ln_label_sz, c1, c2):
            """
            Insert primary zone reference line (6 years from DOR CPT)
            Ported from 513 Cell 9
            
            Parameters:
            -----------
            ax : matplotlib axis
            pz_ref_line : float - Primary zone reference line (days)
            START_TIME : float - Start time for plots (days)
            fnt_sz : float - Base font size
            ref_ln_label_sz : float - Reference line label font size
            c1 : str - Line color
            c2 : str - Label color
            """
            if pz_ref_line > START_TIME:
                ax.axvline(x=pz_ref_line, color=c1, linestyle='--', linewidth=1.5, alpha=0.7)
                ax.text(pz_ref_line,
                        0.95,  # Slightly offset y-position
                        f'  begin\n  Primary Zone',
                        color=c2,
                        size=ref_ln_label_sz,
                        style='italic',
                        stretch='semi-condensed',
                        verticalalignment='top')
        
        def insert_az_line(ax, pz_ref_line, START_TIME, fnt_sz, ref_ln_label_sz, c3, c4):
            """
            Insert above zone reference line (7 years from DOR CPT, end of Primary Zone)
            Ported from 513 Cell 9
            
            Parameters:
            -----------
            ax : matplotlib axis
            pz_ref_line : float - Primary zone reference line (days)
            START_TIME : float - Start time for plots (days)
            fnt_sz : float - Base font size
            ref_ln_label_sz : float - Reference line label font size
            c3 : str - Line color
            c4 : str - Label color
            """
            az_ref_line = pz_ref_line + 365  # Above zone = Primary zone + 1 year
            if az_ref_line > START_TIME:
                ax.axvline(x=az_ref_line, color=c3, linestyle='--', linewidth=1.5, alpha=0.7)
                ax.text(az_ref_line,
                        0.9,  # Slightly offset y-position
                        f'  end\n  Primary Zone',
                        color=c4,
                        size=ref_ln_label_sz,
                        style='italic',
                        stretch='semi-condensed',
                        verticalalignment='top')
        
        def build_subtitle_text(total_pids, low_window, high_window, gender_text="", branch_option=None):
            """
            Build subtitle text with officer counts and time windows
            Ported from 513 Cell 9
            
            Parameters:
            -----------
            total_pids : int - Total number of officers
            low_window : str - Low time window (e.g., "0.0 years")
            high_window : str - High time window (e.g., "9.0 years")
            gender_text : str - Gender text (e.g., "Male " or "Female ")
            branch_option : list or None - Branch options
            
            Returns:
            --------
            str - Formatted subtitle text
            """
            if branch_option:
                if len(branch_option) == 1:
                    branch_text = f"Belonging to {branch_option[0]} Branch Only"
                elif len(branch_option) > 10:
                    branch_text = f"Belonging to {len(branch_option)} Branches"
                else:
                    branch_text = f"Belonging to Branches {branch_option}"
                subtitle_text = f"\n{low_window} - {high_window}   {total_pids:,} Total {gender_text}Officers\n{branch_text}"
            else:
                subtitle_text = f"\n{low_window} - {high_window}   {total_pids:,} Total {gender_text}Officers"
            
            return subtitle_text
        
        # Create dictionary for legend labels
        legend_dict = {
            'sex':{0:'Female',1:'Male','0':'Female','1':'Male'},
            'prestige':{0:'Without',1:'With','0':'Without','1':'With'},
            'marriage':{1:'Married',0:'Unmarried','1':'Married','0':'Unmarried'}
        }
        # Function to create legend labels
        def get_legend_label(plot_spec,group):
            if plot_spec['plot_type'] == 'competing_risks':
                if not plot_spec['variable'] and plot_spec['group_by'] in legend_dict:
                    legend_label = legend_dict[plot_spec['group_by']][group]
                else:
                    legend_label = str(group)            
                return legend_label
        
        # Calculate plot scales
        fnt_sz, suptitl_sz, title_sz, axis_sz, ref_ln_label_sz, tick_sz, lgnd_sz = get_plot_scales(
            PLOT_CONFIG['figsize'], 
            PLOT_CONFIG['font_size'],
            show=True
        )
        
        # Calculate reference lines
        ref_config = PLOT_CONFIG['reference_lines']
        pz_ref_line = ref_config['pz_years'] * 365  # Primary zone in days
        START_TIME = ref_config['start_time']
        
        print_v(f"  • Primary zone reference: {ref_config['pz_years']} years ({pz_ref_line} days)")
        
        time_stop(util_timer, action=util_act, nest=1)
        
        # === 11.6. HELPER FUNCTIONS ===
        print_v("\n🔧 Setting up helper functions...")
        helper_act = "Setting up helper functions"
        helper_timer = time_start(helper_act, nest=1)
        
        def create_bins(df, variable, n_bins=3, method='quantile', strict_equal_n=True):
            """
            Create bins for continuous variables using all available values
            
            Parameters:
            -----------
            df : DataFrame
            variable : str - column name
            n_bins : int - number of bins
            method : str - 'quantile' or 'equal_width'
            strict_equal_n : bool - force equal-N bins using rank-based qcut
            
            Returns:
            --------
            Series with bin labels
            """
            # Get all non-null values for binning
            valid_values = df[variable].dropna()
            
            if len(valid_values) == 0:
                return pd.Series([np.nan] * len(df), index=df.index)
            def _label_by_edges(binned, prefix="Bin"):
                if not hasattr(binned, 'cat'):
                    return binned
                labels = []
                for i, interval in enumerate(binned.cat.categories):
                    labels.append(f"{prefix} {i+1} ({interval.left:.2f}-{interval.right:.2f})")
                return binned.cat.rename_categories(dict(zip(binned.cat.categories, labels)))
                
            if method == 'quantile':
                # Optional strict equal-N using rank to break ties
                if strict_equal_n:
                    try:
                        ranks = valid_values.rank(method='first')
                        binned = pd.qcut(ranks, q=n_bins, duplicates='drop')
                        # Build labels from rank-based bins but keep variable ranges
                        labels = []
                        for i, interval in enumerate(binned.cat.categories):
                            left_rank = interval.left
                            right_rank = interval.right
                            rank_mask = (ranks > left_rank) & (ranks <= right_rank)
                            bin_vals = valid_values[rank_mask]
                            if len(bin_vals) == 0:
                                labels.append(f"Bin {i+1}")
                            else:
                                labels.append(f"Bin {i+1} ({bin_vals.min():.2f}-{bin_vals.max():.2f})")
                        binned = binned.cat.rename_categories(dict(zip(binned.cat.categories, labels)))
                        # Reindex to full df
                        binned_full = pd.Series(np.nan, index=df.index)
                        binned_full.loc[valid_values.index] = binned
                        return binned_full
                    except Exception as e:
                        print_v(f"⚠️ strict_equal_n failed for {variable}: {e}. Falling back to quantile bins.")
                  
                try:
                    quantiles = valid_values.quantile([i/n_bins for i in range(n_bins+1)])
                    binned = pd.cut(df[variable], bins=quantiles, include_lowest=True, duplicates='drop')
                    # Detect collapsed bins
                    if hasattr(binned, 'cat') and len(binned.cat.categories) < n_bins:
                        print_v(
                            f"⚠️ Quantile bins collapsed for {variable}: "
                            f"{len(binned.cat.categories)}/{n_bins} bins. "
                            "Switching to strict equal-N bins."
                        )
                        return create_bins(df, variable, n_bins=n_bins, method='quantile',strict_equal_n=True)
                    return _label_by_edges(binned, prefix="Q")
                except Exception as e:
                    print_v(f"⚠️ Quantile binning failed for {variable}: {e}. Falling back to equal_width.")
                    binned = pd.cut(df[variable], bins=n_bins, include_lowest=True, duplicates='drop')
                    return _label_by_edges(binned)             
            else:  # equal_width
                binned = pd.cut(df[variable], bins=n_bins, include_lowest=True, duplicates='drop')
                return _label_by_edges(binned)        
                
        def prepare_plot_data(df, plot_spec):
            """
            Prepare data for plotting based on plot specification
            Handles time-varying survival interval data
            
            For Kaplan-Meier plots: Aggregates to officer-level FIRST, then creates groups
            This ensures each officer appears in exactly one group based on their final characteristics
            
            Returns:
            --------
            DataFrame ready for plotting, or None if insufficient data
            """
            plot_df = df.copy()
            if plot_spec.get('group_by') == 'div_name' and 'div_name' in plot_df.columns:
                print_v("[DIV_DEBUG] Cell 11 prepare_plot_data() div_name BEFORE filters (top 20):")
                print(plot_df['div_name'].value_counts(dropna=Fales).head(20))
            
            # For Kaplan-Meier plots, we MUST aggregate to officer-level first
            # Otherwise, officers can appear in multiple groups as their cumulative metrics change
            # For competing risks, we also aggregate to officer-level for KM fitting
            
            # Always aggregate to officer-level for plotting (KM requires one row per officer)
            # Use last interval for each officer (contains final event and stop_time)
            officer_df = plot_df.sort_values(['pid_pde', 'stop_time']).groupby('pid_pde').last().reset_index()
            
            # Now work with officer-level data
            working_df = officer_df.copy()
            # print(f"\n\n\n WORKING_DF.COLUMNS:  {working_df.columns}") #####################  DEBUG ######################
            # DEBUG: time alignment for backward evals (uses last snapshot per officer)
            # Optional debug: print_v(f"Variable: {plot_spec.get('variable')}")
            if plot_spec.get('variable') and '_bwd' in str(plot_spec.get('variable')):
                if 'eval_thru_dt_bwd' in working_df.columns and 'snpsht_dt' in working_df.columns:
                    bwd_missing = working_df['eval_thru_dt_bwd'].isna().mean()
                    bwd_future = (working_df['eval_thru_dt_bwd'] > working_df['snpsht_dt']).mean()
                    print_v(f"  • DEBUG bwd time alignment: missing eval_thru_dt_bwd={bwd_missing:.1%}, eval_thru_dt_bwd after snpsht_dt={bwd_future:.1%}")
                else:
                    print_v("  ⚠️ DEBUG skipped bwd time alignment check (required columns missing)")
                
                
            # Filter out officers with NO OER data if requested and variable is OER-related
            # NOTE: We only filter officers who have NO OER data at all (NaN), not those with OER data but 0 top blocks
            if plot_spec.get('filter_zero_oer', False) and plot_spec['variable'] is not None:
                var_col = plot_spec['variable']
                if var_col in working_df.columns:
                    # Check if variable is OER-related using manual list from COLUMN_CONFIG
                    oer_variables = COLUMN_CONFIG.get('oer_variables', [])
                    is_oer_variable = var_col in oer_variables
                    
                    if is_oer_variable:
                        before_count = len(working_df)
                        
                        # Check if officer has ANY OER data by looking at OER variables from config
                        # If they have OER data, at least one of these should be non-NaN
                        # Find which OER columns exist in the data
                        available_oer_cols = [col for col in oer_variables if col in working_df.columns]
                        
                        if len(available_oer_cols) > 0:
                            # Officer has OER data if ANY OER column is non-NaN
                            # (even if the specific variable we're plotting is NaN)
                            has_oer_data = working_df[available_oer_cols].notna().any(axis=1)
                            working_df = working_df[has_oer_data].copy()
                        else:
                            # Fallback: just check the specific variable (if no other OER columns available)
                            # Filter out officers where variable is NaN (no OER data for this specific metric)
                            working_df = working_df[working_df[var_col].notna()].copy()
                        
                        after_count = len(working_df)
                        removed = before_count - after_count
                        if removed > 0:
                            print_v(f"  • Filtered {removed:,} officers with NO OER data ({removed/before_count*100:.1f}%)")
                            print_v(f"  • Remaining officers: {after_count:,} (includes officers with 0 top blocks but with OER data)")
            
            # Optionally drop rows where the plotted variable is NaN
            if plot_spec.get('filter_nan_variable', False) and plot_spec['variable'] is not None:
                var_col = plot_spec['variable']
                if var_col in working_df.columns:
                    before_count = len(working_df)
                    working_df = working_df[working_df[var_col].notna()].copy()
                    removed = before_count - len(working_df)
                    if removed > 0:
                        print_v(
                            f"  • Filtered {removed:,} rows with NaN '{var_col}' "
                            f"({removed/before_count*100:.1f}%)"
                        )
                        print_v(f"  • Remaining officers: {len(working_df):,}")
            
            # DEBUG: report slice used for binning
            print_v(f"  • DEBUG slice for {plot_spec['variable']}: rows={len(working_df):,}")
            if plot_spec['variable'] in working_df.columns:
                print_v(f"  non-null rate: {working_df[plot_spec['variable']].notna().mean():.3f}")
            
            # Optionally drop rows where the group_by variable is NaN
            if plot_spec.get('filter_nan_group_by', False) and plot_spec['group_by'] is not None:
                group_col = plot_spec['group_by']
                if group_col in working_df.columns:
                    before_count = len(working_df)
                    working_df = working_df[working_df[group_col].notna()].copy()
                    removed = before_count - len(working_df)
                    if removed > 0:
                        print_v(
                            f"  • Filtered {removed:,} rows with NaN group_by '{group_col}' "
                            f"({removed/before_count*100:.1f}%)"
                        )
                        print_v(f"  • Remaining officers: {len(working_df):,}")
                        
            # Unknown div_name (etc.): see cox_plot_helpers.apply_unknown_group_by_filter + PLOT_CONFIG['exclude_unknown_div_name']
            working_df = apply_unknown_group_by_filter(working_df, plot_spec, PLOT_CONFIG, print_v)
            
            # Handle variable binning if needed
            if plot_spec['variable'] is not None:
                if plot_spec['variable'] not in working_df.columns:
                    print_v(f"⚠️ Variable '{plot_spec['variable']}' not found in columns: {list(working_df.columns)[:10]}...")
                    print_v(f"⚠️ Skipping plot: {plot_spec['name']}")
                    return None
                
                if plot_spec['bin_continuous']:
                    var_col = plot_spec['variable']
                    if working_df[var_col].dtype in ['float64', 'int64']:
                        # Check if it's actually continuous (more than 10 unique values)
                        if working_df[var_col].nunique() > 10:
                            working_df[f"{var_col}_binned"] = create_bins(
                                working_df, var_col, 
                                n_bins=plot_spec['n_bins'],
                                method=plot_spec['bin_method'],
                                strict_equal_n=plot_spec.get('strict_equal_n', False)
                            )
                            grouping_col = f"{var_col}_binned"
    
                            # DIAGNOSTIC: print_bin_ranges
                            print_v(f"\n📊 Bin ranges for {var_col}:")
                            for bin_label in working_df[grouping_col].cat.categories if hasattr(working_df[grouping_col], 'cat') else working_df[grouping_col].unique():
                                if pd.notna(bin_label):
                                    bin_data = working_df[working_df[grouping_col] == bin_label][var_col]
                                    print_v(f"  • {bin_label}: range [{bin_data.min():.2f} - {bin_data.max():.2f}], mean={bin_data.mean():.2f}, n={len(bin_data):,}")
                        else:
                            grouping_col = var_col
                    else:
                        grouping_col = var_col
                else:
                    grouping_col = plot_spec['variable']
            else:
                grouping_col = None
            
            # Use group_by column (if specified)
            group_col = plot_spec.get('group_by', None)
            
            if group_col is not None:
                if group_col not in working_df.columns:
                    print_v(f"⚠️ Grouping column '{group_col}' not found in columns: {list(working_df.columns)[:10]}...")
                    print_v(f"⚠️ Skipping plot: {plot_spec['name']}")
                    return None
            
            # Optional: bin the group_by column separately (for 3x3, 4x2, etc.)
            group_col_for_plot = group_col
            if group_col is not None and plot_spec.get('bin_group_by', False):
                group_bins = plot_spec.get('group_bins', plot_spec.get('n_bins', 3))
                group_bin_method = plot_spec.get('group_bin_method', plot_spec.get('bin_method', 'quantile'))
                if group_col in working_df.columns:
                    if working_df[group_col].dtype in ['float64', 'int64'] and working_df[group_col].nunique() > 10:
                        working_df[f"{group_col}_binned"] = create_bins(
                            working_df, group_col,
                            n_bins=group_bins,
                            method=group_bin_method,
                            strict_equal_n=plot_spec.get('strict_equal_n', False)
                        )
                        group_col_for_plot = f"{group_col}_binned"
                        # DIAGNOSTIC: print_bin_ranges for group_by
                        print_v(f"\n📊 Group-by bin ranges for {group_col}:")
                        for bin_label in working_df[group_col_for_plot].cat.categories if hasattr(working_df[group_col_for_plot], 'cat') else working_df[group_col_for_plot].unique():
                            if pd.notna(bin_label):
                                bin_data = working_df[working_df[group_col_for_plot] == bin_label][group_col]
                                print_v(f"  • {bin_label}: range [{bin_data.min():.2f} - {bin_data.max():.2f}], mean={bin_data.mean():.2f}, n={len(bin_data):,}")
                    else:
                        group_col_for_plot = group_col
            
            # plot_group must not be group_by | <each distinct float> when variable is continuous and
            # bin_continuous=False (that creates hundreds of spurious curves). Omit variable from the
            # composite when group_only_curves is True, or auto when numeric nunique is large.
            if group_col_for_plot is not None and grouping_col is not None:
                gn = grouping_col
                if gn in working_df.columns and pd.api.types.is_numeric_dtype(working_df[gn]) and not plot_spec.get('bin_continuous', False):
                    nu = int(working_df[gn].nunique())
                    if plot_spec.get('group_only_curves') or nu > 20:
                        if nu > 20 and not plot_spec.get('group_only_curves'):
                            print_v(
                                f"  • Auto: omit `{gn}` from plot_group (nunique={nu}) with bin_continuous=False and "
                                f"group_by={group_col_for_plot!r} — one curve per group. "
                                f"Set plot_spec['group_only_curves']=True to silence, or bin_continuous=True to bin the variable."
                            )
                        grouping_col = None
            
            # Create final grouping column
            if grouping_col is not None and group_col_for_plot is not None:
                # Combine variable and group_by for multi-level grouping
                working_df['plot_group'] = (working_df[group_col_for_plot].astype(str) + " | " + 
                                           working_df[grouping_col].astype(str))
            elif group_col_for_plot is not None:
                # Just use group_by column (possibly binned)
                working_df['plot_group'] = working_df[group_col_for_plot].astype(str)
            elif grouping_col is not None:
                # Just use the binned variable (no group_by)
                working_df['plot_group'] = working_df[grouping_col].astype(str)
            else:
                # Fallback: use a default group
                working_df['plot_group'] = 'All'
            
            # Filter to minimum group size (count rows since we're at officer-level)
            group_counts = working_df['plot_group'].value_counts()
            
            print_v(f"  • Group counts (before filtering): {dict(group_counts.sort_values(ascending=False))}")
            
            valid_groups = group_counts[group_counts >= plot_spec['min_group_size']].index
            working_df = working_df[working_df['plot_group'].isin(valid_groups)].copy()
            
            if len(working_df) == 0:
                print_v(f"⚠️ No groups meet minimum size requirement ({plot_spec['min_group_size']} officers)")
                print_v(f"⚠️ Largest group had {group_counts.max() if len(group_counts) > 0 else 0} officers")
                print_v(f"⚠️ Skipping plot: {plot_spec['name']}")
                return None
            
            print_v(f"  • Valid groups after filtering: {len(valid_groups)} groups with >= {plot_spec['min_group_size']} officers each")
            return working_df
        
        def plot_kaplan_meier(df, plot_spec, save_path, metadata):
            """Create Kaplan-Meier plot using scikit-survival with enhanced formatting"""
            from sksurv.nonparametric import kaplan_meier_estimator
            from sksurv.util import Surv
            
            fig, ax = plt.subplots(figsize=PLOT_CONFIG['figsize'])
            
            # Get unique groups and sort by average of binned variable (if avaialable)
            # This ensures that legend shows bins from highest to lowest average value
            groups = df['plot_group'].unique()
            
            # Get variable name for legend enhancement
            var_name = plot_spec.get('variable')
            
            # If a variable is being binned, sort groups by its average value
            if var_name is not None and var_name in df.columns:
                # Calculate average of variable for each group
                group_means = df.groupby('plot_group')[var_name].mean()
                # Sort groups by average value (highest to lowest)
                groups = sorted(groups, key=lambda g: group_means.get(g, float('-inf')), reverse=True)
                print_v(f"  • Sorted groups by average {var_name} (highest to lowest)")
                    
            colors = plt.cm.Set1(np.linspace(0, 1, len(groups)))
            
            for i, group in enumerate(groups):
                group_data = df[df['plot_group'] == group]
                
                if len(group_data) == 0:
                    continue
                
                # Data is already at officer-level from prepare_plot_data()
                # Filter out invalid data (NaN or non-positive durations)
                valid_mask = (group_data['stop_time'].notna()) & (group_data['stop_time'] > 0)
                officer_data = group_data[valid_mask].copy()
                
                if len(officer_data) == 0:
                    print_v(f"  ⚠️ No valid officers in group '{group}' after filtering")
                    continue
                
                # Debug: Check event distribution
                event_counts = officer_data['event'].value_counts().sort_index()
                print_v(f"  • Group '{group}': {len(officer_data)} officers")
                print_v(f"    Event distribution: {dict(event_counts)}")
                print_v(f"    stop_time range: {officer_data['stop_time'].min():.0f} - {officer_data['stop_time'].max():.0f} days")
                
                # Create event indicator: True if promoted (event == 1), False otherwise (attrited or censored)
                # For KM, we treat promotion as the event of interest, others as censored
                event_observed = (officer_data['event'] == 1).astype(bool)  # Promotion events
                
                # Debug: Check promotion events
                n_promoted = event_observed.sum()
                n_total = len(event_observed)
                print_v(f"    Promotions: {n_promoted}/{n_total} ({n_promoted/n_total*100:.1f}%)")
                
                # Create scikit-survival Surv structure
                y = Surv.from_arrays(
                    event=event_observed,
                    time=officer_data['stop_time'].values
                )
                
                # Calculate the Kaplan-Meier estimator
                time, survival_prob = kaplan_meier_estimator(y['event'], y['time'])
                 
                # Calculate promotion probability = 1 - survival function
                promotion_prob = 1 - survival_prob
                
                # Use enhanced legend label
                legend_label = format_legend_label(plot_spec, group, df, var_name)
                ax.plot(time, promotion_prob, 
                        color=colors[i], label=legend_label, linewidth=2)
            
            # Enhanced title/subtitle with publication toggles
            main_title = format_plot_title(plot_spec)
            subtitle = format_plot_subtitle(plot_spec, metadata)
            show_titles = PLOT_CONFIG.get('show_plot_titles', True)
            show_subtitles = PLOT_CONFIG.get('show_plot_subtitles', False)
            km_legend_outside = PLOT_CONFIG.get('km_legend_outside', True)

            if show_titles:
                ax.set_title(main_title, fontsize=title_sz, fontweight='bold', pad=22)
            if show_subtitles:
                ax.text(0.5, 1.00, subtitle, transform=ax.transAxes, fontsize=axis_sz*0.8,
                       ha='center', va='bottom', style='italic', color='gray')

            ax.set_xlabel('Days from Promotion to CPT', fontsize=axis_sz)
            ax.set_ylabel('Probability of Promotion', fontsize=axis_sz)
            if km_legend_outside:
                ax.legend(title='Group', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=lgnd_sz, title_fontsize=lgnd_sz)
            else:
                ax.legend(title='Group', loc='upper left', fontsize=lgnd_sz, title_fontsize=lgnd_sz)
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 1)  # Set y-axis limits to match competing risks plots for consistent text positioning
            ax.tick_params(axis='both', which='major', labelsize=tick_sz)
            # Optional x-axis max (trim long tails/outliers)
            x_max_days = plot_spec.get('x_max_days')
            if x_max_days is not None:
                ax.set_xlim(right=x_max_days)
            x_min_days = plot_spec.get('x_min_days')
            if x_min_days is not None:
                ax.set_xlim(left=x_min_days)            
            # Add configuration box
            # add_configuration_box(ax, plot_spec, metadata, font_size=lgnd_sz)
            
            # Add reference lines if enabled
            if PLOT_CONFIG['reference_lines']['show_pz_line']:
                insert_pz_line(ax, pz_ref_line, START_TIME, fnt_sz, ref_ln_label_sz, c1, c2)
            if PLOT_CONFIG['reference_lines']['show_az_line']:
                insert_az_line(ax, pz_ref_line, START_TIME, fnt_sz, ref_ln_label_sz, c3, c4)
            
            plt.tight_layout()
            plt.savefig(save_path, dpi=PLOT_CONFIG['dpi'], bbox_inches='tight')
            try:
                plt.show()  # Display plot in notebook output
            except Exception as e:
                print_v(f"⚠️ Could not display plot (non-fatal): {e}")
            plt.close()
            
            print_v(f"✅ Saved: {save_path}")
        
        def calculate_cif(group_data, event_type):
            """
            Calculate Cumulative Incidence Function (CIF) using simple proportion approach
            Based on 511 implementation - gives realistic rates that match observed proportions
            
            This calculates cumulative incidence as a simple proportion of the total population,
            which is appropriate for officer-level data where we want unconditional probabilities.
            
            Parameters:
            -----------
            group_data : DataFrame - Officer-level data with 'stop_time' and 'event' columns
            event_type : int - Event type to calculate CIF for (1=promotion, 2=attrition)
            
            Returns:
            --------
            times : array - Time points
            cif : array - Cumulative incidence values
            """
            import numpy as np
            
            # Sort by stop_time
            data = group_data.sort_values('stop_time').copy()
            
            # Get event indicator
            if event_type == 1:  # Promotion
                event_indicator = (data['event'] == 1)
            elif event_type == 2:  # Attrition
                event_indicator = (data['event'] == 2)
            else:
                raise ValueError(f"Invalid event_type: {event_type}. Must be 1 (promotion) or 2 (attrition)")
            
            # Get unique time points (sorted)
            times = sorted(data['stop_time'].unique())
            
            # Total population (officers) - this is the key: use total initial population
            total_population = len(data)
            
            if total_population == 0:
                return np.array([]), np.array([])
            
            # Calculate cumulative incidence as simple proportion
            cumulative_incidence = []
            
            for t in times:
                # Count events of type event_type at exactly time t
                events_at_t = ((data['stop_time'] == t) & event_indicator).sum()
                
                # Calculate cumulative incidence as proportion of TOTAL population
                if len(cumulative_incidence) == 0:
                    ci = events_at_t / total_population
                else:
                    ci = cumulative_incidence[-1] + (events_at_t / total_population)
                
                cumulative_incidence.append(ci)
            
            return np.array(times), np.array(cumulative_incidence)
        
        def plot_competing_risks(df, plot_spec, save_path, metadata):
            """Create competing risks plot using simple proportion CIF calculation"""
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(PLOT_CONFIG['figsize'][0], PLOT_CONFIG['figsize'][1]*1.5))
            
            # Get unique groups and sort by average of binned variable (if avaialable)
            # This ensures that legend shows bins from highest to lowest average value
            groups = df['plot_group'].unique()
            
            # Get variable name for legend enhancement
            var_name = plot_spec.get('variable')

            def _hex_to_rgb(hex_color):
                hex_color = hex_color.lstrip('#')
                return tuple(int(hex_color[i:i+2], 16) / 255.0 for i in (0, 2, 4))

            def _rgb_to_hex(rgb):
                return "#%02x%02x%02x" % tuple(int(round(c * 255)) for c in rgb)

            def _build_hex_gradient(hex_colors, n_colors):
                if not hex_colors:
                    return None
                if len(hex_colors) == n_colors:
                    return hex_colors
                if len(hex_colors) == 1:
                    return [hex_colors[0]] * n_colors
                start = _hex_to_rgb(hex_colors[0])
                end = _hex_to_rgb(hex_colors[-1])
                gradient = []
                for i in range(n_colors):
                    t = i / max(n_colors - 1, 1)
                    rgb = (
                        start[0] + (end[0] - start[0]) * t,
                        start[1] + (end[1] - start[1]) * t,
                        start[2] + (end[2] - start[2]) * t,
                    )
                    gradient.append(_rgb_to_hex(rgb))
                return gradient
            
            # If a variable is being binned, sort groups by its average value
            ordered_groups = list(groups)
            if var_name is not None and var_name in df.columns:
                # Calculate average of variable for each group
                group_means = df.groupby('plot_group')[var_name].mean()
                # Sort groups by average value (highest to lowest)
                groups = sorted(groups, key=lambda g: group_means.get(g, float('-inf')), reverse=True)
                # Ascending order for palette mapping (Q1..Qn)
                ordered_groups = sorted(group_means.index, key=lambda g: group_means.get(g, float('inf')))
                print_v(f"  • Sorted groups by average {var_name} (highest to lowest)")

            # Palettes: default blue/orange gradients; optional discrete colors (see cox_plot_helpers.cr_discrete_group_colors)
            cmode = plot_spec.get('cr_color_mode')
            if cmode in ('set1', 'tab20', 'husl', 'glasbey'):
                promo_palette = cr_discrete_group_colors(len(ordered_groups), cmode)
                attr_palette = list(promo_palette)
            else:
                promo_palette = plot_spec.get('cif_bar_palette_promo') or ["#d7e6f5", "#1f5aa6"]
                attr_palette = plot_spec.get('cif_bar_palette_attr') or ["#fde2cf", "#c0581a"]
                promo_palette = _build_hex_gradient(promo_palette, len(ordered_groups))
                attr_palette = _build_hex_gradient(attr_palette, len(ordered_groups))
            
            promo_color_map = {g: promo_palette[i] for i, g in enumerate(ordered_groups)}
            attr_color_map = {g: attr_palette[i] for i, g in enumerate(ordered_groups)}
            
            promo_colors = [promo_color_map.get(g, '#4c4c4c') for g in groups]
            attr_colors = [attr_color_map.get(g, '#4c4c4c') for g in groups]
            # print("# Plot 1: Promotion events (event == 1)******************************************************")
            # Plot 1: Promotion events (event == 1)
            for i, group in enumerate(groups):
                group_data = df[df['plot_group'] == group].copy()
                
                if len(group_data) == 0:
                    continue
                # print("# Filter out invalid data)******************************************************")
                # Filter out invalid data
                valid_mask = (group_data['stop_time'].notna()) & (group_data['stop_time'] > 0)
                group_data = group_data[valid_mask].copy()
                
                if len(group_data) == 0:
                    print_v(f"  ⚠️ No valid officers in group '{group}' after filtering")
                    continue
                # print("# Calculate CIF for promotion using simple proportion approach)******************************************************")
                # Calculate CIF for promotion using simple proportion approach
                times, promotion_cif = calculate_cif(group_data, event_type=1)
                # print("# Debug: Check final CIF value******************************************************")
                # Debug: Check final CIF value
                final_cif = promotion_cif[-1] if len(promotion_cif) > 0 else 0
                n_promoted = (group_data['event'] == 1).sum()
                n_total = len(group_data)
                print_v(f"  • Group '{group}': Promotion CIF final={final_cif:.3f}, actual rate={n_promoted/n_total:.3f} ({n_promoted:,}/{n_total:,})")
                # print("# ACTUAL PLOT COMMAND ax1.plot(times, promotion_cif, ******************************************************")
                # Use enhanced legend label
                legend_label = format_legend_label(plot_spec, group, df, var_name)
                ax1.plot(times, promotion_cif, 
                        color=promo_colors[i], label=legend_label, linewidth=2)
            
            # Enhanced formatting for promotion plot
            main_title = format_plot_title(plot_spec)
            subtitle = format_plot_subtitle(plot_spec, metadata)
            show_titles = PLOT_CONFIG.get('show_plot_titles', True)
            show_subtitles = PLOT_CONFIG.get('show_plot_subtitles', False)
            cr_legend_outside = PLOT_CONFIG.get('cr_legend_outside', False)

            if show_titles:
                ax1.set_title(f"Promotion Probability (CIF): {main_title}", fontsize=title_sz, fontweight='bold', pad=22)
            if show_subtitles:
                ax1.text(0.5, 1.00, subtitle, transform=ax1.transAxes, fontsize=axis_sz*0.8, 
                        ha='center', va='bottom', style='italic', color='gray')
            ax1.set_xlabel('Days from Promotion to CPT', fontsize=axis_sz)
            ax1.set_ylabel('Cumulative Incidence of Promotion', fontsize=axis_sz)
            if cr_legend_outside:
                ax1.legend(title='Group', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=lgnd_sz, title_fontsize=lgnd_sz)
            else:
                ax1.legend(title='Group', loc='upper left', fontsize=lgnd_sz, title_fontsize=lgnd_sz)
            ax1.grid(True, alpha=0.3)
            ax1.set_ylim(0, 1)
            ax1.tick_params(axis='both', which='major', labelsize=tick_sz)
            # Optional x-axis max (trim long tails/outliers)
            x_max_days = plot_spec.get('x_max_days')
            if x_max_days is not None:
                ax1.set_xlim(right=x_max_days)
            x_min_days = plot_spec.get('x_min_days')
            if x_min_days is not None:
                ax1.set_xlim(left=x_min_days)
                
            # Add configuration box to first subplot REMOVED FOR PAPER
            # add_configuration_box(ax1, plot_spec, metadata, font_size=lgnd_sz)
            
            # Add reference lines if enabled
            if PLOT_CONFIG['reference_lines']['show_pz_line']:
                insert_pz_line(ax1, pz_ref_line, START_TIME, fnt_sz, ref_ln_label_sz, c1, c2)
            if PLOT_CONFIG['reference_lines']['show_az_line']:
                insert_az_line(ax1, pz_ref_line, START_TIME, fnt_sz, ref_ln_label_sz, c3, c4)
            # print("# Plot 2: Attrition events (event == 2)******************************************************")
            # Plot 2: Attrition events (event == 2)
            for i, group in enumerate(groups):
                group_data = df[df['plot_group'] == group].copy()
                
                if len(group_data) == 0:
                    continue
                # print("# Filter out invalid data)******************************************************")              
                # Filter out invalid data
                valid_mask = (group_data['stop_time'].notna()) & (group_data['stop_time'] > 0)
                group_data = group_data[valid_mask].copy()
                
                if len(group_data) == 0:
                    continue
                # print("# Calculate CIF for promotion using simple proportion approach)******************************************************")                
                # Calculate CIF for attrition using simple proportion approach
                times, attrition_cif = calculate_cif(group_data, event_type=2)
                # print("# Debug: Check final CIF value******************************************************")                
                # Debug: Check final CIF value
                final_cif = attrition_cif[-1] if len(attrition_cif) > 0 else 0
                n_attrited = (group_data['event'] == 2).sum()
                n_total = len(group_data)
                print_v(f"  • Group '{group}': Attrition CIF final={final_cif:.3f}, actual rate={n_attrited/n_total:.3f} ({n_attrited:,}/{n_total:,})")
                # print("# ACTUAL PLOT COMMAND ax2.plot(times, attrition_cif,  ******************************************************")                
                # Use enhanced legend label
                legend_label = format_legend_label(plot_spec, group, df, var_name)
                ax2.plot(times, attrition_cif, 
                        color=attr_colors[i], label=legend_label, linewidth=2)
            
            # Enhanced formatting for attrition plot
            if show_titles:
                ax2.set_title(f"Attrition Probability (CIF): {main_title}", fontsize=title_sz, fontweight='bold', pad=22)
            if show_subtitles:
                ax2.text(0.5, 1.00, subtitle, transform=ax2.transAxes, fontsize=axis_sz*0.8, 
                        ha='center', va='bottom', style='italic', color='gray')
            ax2.set_xlabel('Days from Promotion to CPT', fontsize=axis_sz)
            ax2.set_ylabel('Cumulative Incidence of Attrition', fontsize=axis_sz)
            if cr_legend_outside:
                ax2.legend(title='Group', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=lgnd_sz, title_fontsize=lgnd_sz)
            else:
                ax2.legend(title='Group', loc='upper left', fontsize=lgnd_sz, title_fontsize=lgnd_sz)
            ax2.grid(True, alpha=0.3)
            ax2.set_ylim(0, 1)
            ax2.tick_params(axis='both', which='major', labelsize=tick_sz)
            # Optional x-axis max (trim long tails/outliers)
            x_max_days = plot_spec.get('x_max_days')
            if x_max_days is not None:
                ax2.set_xlim(right=x_max_days)
            x_min_days = plot_spec.get('x_min_days')
            if x_min_days is not None:
                ax2.set_xlim(left=x_min_days)
                
            # Add reference lines if enabled
            if PLOT_CONFIG['reference_lines']['show_pz_line']:
                insert_pz_line(ax2, pz_ref_line, START_TIME, fnt_sz, ref_ln_label_sz, c1, c2)
            if PLOT_CONFIG['reference_lines']['show_az_line']:
                insert_az_line(ax2, pz_ref_line, START_TIME, fnt_sz, ref_ln_label_sz, c3, c4)
            
            plt.tight_layout()
            plt.savefig(save_path, dpi=PLOT_CONFIG['dpi'], bbox_inches='tight')
            try:
                plt.show()  # Display plot in notebook output
            except Exception as e:
                print_v(f"⚠️ Could not display plot (non-fatal): {e}")
            plt.close()
            
            print_v(f"✅ Saved: {save_path}")
        
        time_stop(helper_timer, action=helper_act, nest=1)
        
        # === 11.7. GENERATE PLOTS ===
        print_v("\n📊 Generating plots...")
        plot_act = "Generating plots"
        plot_timer = time_start(plot_act, nest=1)
        
        plots_created = 0
        plots_skipped = 0
        
        # === 11.7A. DIAGNOSTIC: Raw vs Z (tb_ratio_bwd_snr) ===
        diag_var = TB_RATIO_SNR_COL
        diag_z = f"{Z_PREFIX}{diag_var}"
        base_spec = None
        # print("# === 11.7A. DIAGNOSTIC: Raw vs Z (tb_ratio_bwd_snr) ===  ******************************************************")                
        for ps in PLOT_CONFIG['plots']:
            if ps.get('variable') == diag_var and ps.get('plot_type') == 'competing_risks':
                base_spec = ps
                break

        if base_spec:
            # Create a temporary z column if missing
            if diag_z not in df_analysis.columns and diag_var in df_analysis.columns:
                series = pd.to_numeric(df_analysis[diag_var], errors='coerce')
                mean = series.mean()
                std = series.std(ddof=0)
                if std and std > 0:
                    df_analysis[diag_z] = (series - mean) / std
                    print_v(f"  ✅ Created temporary {diag_z} for diagnostics")
                else:
                    print_v(f"  ⚠️ Skipping z diagnostic (std=0 for {diag_var})")     
        
            if diag_z in df_analysis.columns:
                for label, var in [("raw", diag_var), ("z", diag_z)]:
                    diag_spec = dict(base_spec)
                    diag_spec['variable'] = var
                    diag_spec['name'] = f"diag_cr_{label}_{var}"
                    print_v(f" $$$$$ Diagnostic plot: {diag_spec['name']}")
    
                    plot_df = prepare_plot_data(df_analysis, diag_spec)
                    if plot_df is None:
                        print_v("  ⚠️ Diagnostic skipped (no data)")
                        continue
    
                    metadata = get_plot_metadata(diag_spec, df_analysis, plot_df)
                    filename_base = generate_plot_filename(diag_spec, metadata['data_stats'])
                    filename_base = f"diag_{label}_{filename_base}"
                    save_path = os.path.join(PLOT_CONFIG['plot_dir'], filename_base)
                    # print("# === 11.7A. DIAGNOSTIC:plot_competing_risks(plot_df, diag_spec, save_path, metadata)  ******************************************************")         
                    plot_competing_risks(plot_df, diag_spec, save_path, metadata)
                    if diag_spec.get('plot_cif_bars', False) or PLOT_CONFIG.get('plot_cif_bars', False):
                        plot_competing_risks_cif_bars(
                            plot_df,
                            diag_spec,
                            metadata,
                            output_dir=PLOT_CONFIG['plot_dir'],
                            filename_base=filename_base,
                            dpi = PLOT_CONFIG.get('dpi', 300),
                            figsize=PLOT_CONFIG.get('figsize', [10,8]),
                            legend_outside=PLOT_CONFIG.get('cif_bar_legend_outside', True)
                        )
        else:
            print_v(" ⚠️ Diagnostic raw vs z skipped (base plot or z columns missing)")
        
        # Begin main (non-diagnostic) plots
        # Use df_analysis (with selected columns) for plotting
        for plot_spec in PLOT_CONFIG['plots']:
            plot_name = plot_spec['name']
            print_v(f"\n  🔄 Creating plot: {plot_name}")
            print_v(f"     Variable: {plot_spec.get('variable', 'None')}")
            print_v(f"     Group by: {plot_spec.get('group_by', 'None')}")
            print_v(f"     Plot type: {plot_spec.get('plot_type', 'None')}")

            # Appearance toggles with publication-first defaults
            plot_spec.setdefault('legend_show_final_cif', PLOT_CONFIG.get('legend_show_final_cif', False))
            plot_spec.setdefault('cif_bar_show_titles', PLOT_CONFIG.get('cif_bar_show_titles', False))
            plot_spec.setdefault('cif_bar_show_xlabels', PLOT_CONFIG.get('cif_bar_show_xlabels', False))

            # Prepare data
            plot_df = prepare_plot_data(df_analysis, plot_spec)
            # print("plot_df = prepare_plot_data(df_analysis, plot_spec)  ******************************************************")         
           
            if plot_df is None:
                print_v(f"  ❌ Skipped plot: {plot_name}")
                plots_skipped += 1
                continue
            
            print_v(f"  ✅ Data prepared: {len(plot_df):,} rows, {plot_df['plot_group'].nunique()} groups")
            # Optional debug: event rates by bin for key TB ratios
            # DEBUG: event rates by bin for key TB ratios
            debug_vars = {
                'tb_ratio_fwd_snr',
                'tb_ratio_bwd_snr',
                'tb_ratio_bwd_snr_legacy',
            }
            if plot_spec.get('variable') in debug_vars:
                try:
                    grp = plot_df.groupby('plot_group', dropna=False)
                    promo_rate=grp['event'].apply(lambda s: (s == 1).mean())
                    attr_rate=grp['event'].apply(lambda s: (s == 2).mean())            
                    print_v(f"  DEBUG event rates by bin (promotion / attrition):")
                    for g in promo_rate.index:
                        print_v(f"     {g}: {promo_rate[g]:.3f} / {attr_rate[g]:.3f}")
                except Exception as e:
                    print_v(f"⚠️ DEBUG event rate calc failed: {e}")
            # Optional debug end: event rates by bin for key TB ratios
            # print(f"\n# Generate metadata ******************************************************")                                  
            # Generate metadata
            metadata = get_plot_metadata(plot_spec, df_analysis, plot_df)
            # print("# Generate enhanced filename ******************************************************")                     
            # Generate enhanced filename
            filename_base = generate_plot_filename(plot_spec, metadata['data_stats'])
            save_path = os.path.join(PLOT_CONFIG['plot_dir'], filename_base)
            # Create plot based on plot type
            # Create plot based on type
            if plot_spec['plot_type'] == 'kaplan_meier':
                plot_kaplan_meier(plot_df, plot_spec, save_path, metadata)
                plots_created += 1
                print_v(f"  ✅ Created Kaplan-Meier plot: {filename_base}")
            elif plot_spec['plot_type'] == 'competing_risks':
                plot_competing_risks(plot_df, plot_spec, save_path, metadata)
                plots_created += 1
                print_v(f"  ✅ Created competing risks plot: {filename_base}")

                # Optional bar chart of final CIF by bin
                if plot_spec.get('plot_cif_bars', False) or PLOT_CONFIG.get('plot_cif_bars', False):
                    # Optional debug: CIF bar chart block
                    _cif_figsize = (PLOT_CONFIG['figsize'][0], PLOT_CONFIG['figsize'][1] )
                    plot_competing_risks_cif_bars(
                        plot_df,
                        plot_spec,
                        metadata,
                        output_dir=PLOT_CONFIG['plot_dir'],
                        filename_base=filename_base,
                        dpi=PLOT_CONFIG.get('dpi', 400),
                        figsize=_cif_figsize,
                        tick_sz=tick_sz,
                        legend_outside=PLOT_CONFIG.get('cif_bar_legend_outside', True)
                    )

                    print_v(f"  ✅ Created CIF bar plot: {filename_base.replace('.png','')}_cif_bars.png, {plot_spec['n_bins']} bins, {plot_spec['bin_method']}, jobs: {filtering_params['filter_job_codes']['include_specific']}")
            else:
                print_v(f"⚠️ Unknown plot type: {plot_spec['plot_type']}, skipping")
                plots_skipped += 1
            
            # Save metadata JSON file
            try:
                metadata_path = save_plot_metadata_card(metadata, PLOT_CONFIG['plot_dir'], filename_base.replace('.png', ''))
                print_v(f"  ✅ Saved metadata: {os.path.basename(metadata_path)}")
                if PLOT_CONFIG.get('show_metadata_cards', False):
                    display(Image(metadata_path))
            except Exception as e:
                print_v(f"  ⚠️ Could not save metadata: {e}")
        
        time_stop(plot_timer, action=plot_act, nest=1)
        
        print_v(f"\n✅ Plotting complete!")
        print_v(f"  • Created: {plots_created} plots")
        print_v(f"  • Skipped: {plots_skipped} plots")
        print_v(f"  • Output directory: {PLOT_CONFIG['plot_dir']}")
        
        # Save the analysis dataset for future use
        save_act = "Saving analysis dataset"
        save_timer = time_start(save_act, nest=1)
        if 'yg' in df_analysis:
            df_analysis['yg'] = pd.to_numeric(df_analysis['yg'], errors='coerce').astype('Int64')
        store_feather(df_analysis, 'df_pipeline_11_cox_analysis')
        time_stop(save_timer, action=save_act, nest=1)
        print_v(f"✅ Saved analysis dataset: df_pipeline_11_cox_analysis.feather")
        
        time_stop(cell11, action=cell11_act, nest=0)
    else:
        print_v("By-passing CELL 11...")
    tyme()


### Senior Rater {Pool Distribution}

In [ ]:
reload_pipeline_config()
# === Pool size distributions (mode-aware) ===
from cox_plot_helpers import plot_var_distribution
cox_results_dir = "./cox/cox_results"
df_plot = df_analysis  # post‑filtering, matches CR/CIF sample

# Detect which rater level(s) are used in plots
plot_vars = []
for spec in PLOT_CONFIG.get('plots', []):
    if isinstance(spec, dict):
        plot_vars.extend([spec.get('variable'), spec.get('group_by')])
plot_vars = [v for v in plot_vars if isinstance(v, str)]
use_rtr = any('_rtr' in v for v in plot_vars)
use_snr = any('_snr' in v for v in plot_vars)

pool_cols = []
if use_rtr:
    pool_cols.append((POOL_SIZE_RTR_COL, "Rater pool size"))
if use_snr:
    pool_cols.append((POOL_SIZE_SNR_COL, "Senior rater pool size"))

if not pool_cols:
    print_v("ℹ️ No _rtr/_snr variables in plots; skipping pool size distributions.")
else:
    for col, label in pool_cols:
        if col in df_plot.columns:
            plot_var_distribution(
                df_plot[col],
                label,
                bins=100,
                min_bins=10,
                max_bins=60,
                color="steelblue",
                log_x=False,
                log_y=True,
                output_dir=cox_results_dir,
                filename=f"distribution_{col}",
                dpi=300,
            )
        else:
            print_v(f"⚠️ Missing column: {col}")


# **🔍 CELL 12: COX REGRESSION MODELS**
##### **Comprehensive Cox regression analysis using scikit-survival**
- **12.1**: Prepare data for Cox regression
- **12.2**: Fit static model (demographics only)
- **12.3**: Fit full model (static + time-varying covariates)
- **12.4**: Model comparison and signal ratios
- **12.5**: Competing risks analysis
- **12.6**: Partial effects plots
- **12.6A**: Combined linear + quadratic effects (base + `_sq` pairs)
- **12.7**: Interaction effects (3D surface, contour, slices; `INTERACTION_CONFIG`)
- **12.8**: Model-based promotion curves (Cox `1−S(t)` at fixed low/mid/high X×Y; other covariates at medians)
##### **Focus**: Modular Cox regression analysis with progressive saves for troubleshooting


In [ ]:
df_analysis.div_name.value_counts()

## === 📦 12.1. PREPARE DATA FOR COX REGRESSION MODELS ===


In [ ]:
# === CELL 12.1: PREPARE DATA FOR COX REGRESSION MODELS ===
# Reload pipeline_config to pick up any changes made 
# to configuration and other .py scripts
reload_pipeline_config()
fig_save_suffix = '_stdz' if STANDARDIZE_CONFIG['use_for_model'] else ''
with notebook_cell_logging('12.1'):
    import os
    import numpy as np
    
    # Define output directoried for models and results
    cox_model_dir = "./cox/cox_models"
    
    # Respect dummy-variable creation setting from pipeline_config
    create_dummies = COLUMN_CONFIG.get('dummy_variables', {}).get('create_dummies', True)
    cox_results_dir = "./cox/cox_results"
    # Create directories if they don't exist
    os.makedirs(cox_model_dir, exist_ok=True)
    os.makedirs(cox_results_dir, exist_ok=True)
    
    # === 12.0. VERIFY PREREQUISITES ===
    # Check that required outputs from CELL 10 and CELL 11 exist
    
    # Check for COLUMN_CONFIG from CELL 11
    if 'COLUMN_CONFIG' not in globals():
        # Check for COLUMN_CONFIG in cox_var_dir directory
        if os.path.exists(cox_var_dir + 'COLUMN_CONFIG.json'):
            load_json('COLUMN_CONFIG', cox_var_dir)
        else:
            raise NameError("❌ ERROR: COLUMN_CONFIG not found.  Please run CELL 11 first to define column configuration.")
        
    # Check for data - prefer df_analysis from CELL 11 (has dummy variables), fallback to df_cox from CELL 11
    required_cols = ['pid_pde', 'start_time', 'stop_time', 'event']
    if STANDARDIZE_CONFIG.get('use_for_model', False):
        # Prefer Cox-ready data when using standardized columns
        print_v(f"✅ Using df_pipeline_10_5_cox_zscored (standardized model inputs)")
        df_cox = load_feather('df_pipeline_10_5_cox_zscored')
        missing_cols = [col for col in required_cols if col not in df_cox.columns]
        if missing_cols:
            raise ValueError(f"❌ ERROR: df_pipeline_10_5_cox_zscored missing required columns: {missing_cols}.  Please run CELL 10.")    
    elif 'df_analysis' in globals():
        # Use df_analysis from CELL 11 (already has dummy variables)
        print_v("✅ Using df_analysis from CELL 11 (already has dummy variables)")
        df_cox = df_analysis.copy()
        missing_cols = [col for col in required_cols if col not in df_cox.columns]
        if missing_cols:
            raise ValueError(f"❌ ERROR: df_analysis from CELL 11 missing required columns: {missing_cols}.  Please run CELL 11 completely.")
    else:
        # Try to load df_analysis form saved file (CELL 11 output)
        try:
            print_v(f"🔍 Attempting to load df_analysis from saved file (df_pipeline_11_cox_analysis)...")
            df_analysis = load_feather('df_pipeline_11_cox_analysis')
            print_v(f"✅ Loaded df_pipeline_11_cox_analysis from saved file (includes dummy variables)")    
            missing_cols = [col for col in required_cols if col not in df_analysis.columns]
            if missing_cols:
                raise ValueError(f"❌ ERROR: Loaded df_analysis missing required columns: {missing_cols}.  Please run CELL 11 completely.")
        except:
            # Fall back to df_cox from CELL 10
            if 'df_cox' in globals():
                print_v("✅ Using df_cox from CELL 10")
                df_cox = df_analysis .copy()
                missing_cols = [col for col in required_cols if col not in df_cox.columns]
                if missing_cols:
                    raise ValueError(f"❌ ERROR: df_cox from CELL 10 missing required columns: {missing_cols}.  Please run CELL 10 first.")
            else:
                raise NameError("❌ ERROR: Neither df_analysis (from CELL 11) nor df_cox (from CELL 11) found. Please run CELL 11 first.")
    
    print_v(f"✅ Data ready: {len(df_cox):,} intervals for {df_cox['pid_pde'].nunique():,} officers")
    
    # === 12.1. PREPARE DATA FOR COX REGRESSION) ===
    # Data should already be prepared in CELL 10 (df_cox with start_time, stop_time, event)
    # This section just verifies data is ready
    if CELL12_1:
        print_v("\n📊 12.1: Preparing data for Cox regression...")
        prep_act = "12.1: Prepare data"
        prep_timer = time_start(prep_act, nest=1)
        
        # Save prepared data for reference
        store_feather(df_cox, 'df_pipeline_12_01_prepared')
        print_v(f"✅ Saved prepared data:  df_pipeline_12_01_prepared.feather")
        
        time_stop(prep_timer, action=prep_act, nest=1)
    else:
        print_v(f"⏭️ Skipping 12.1: Data preparation")
        if CELL12_2:
            df_cox = load_feather('df_pipeline_12_01_prepared')      


In [ ]:
if null_reports:
    df_debug =load_feather('df_pipeline_12_01_prepared')
    df_null_report(df_debug)
    if 'event' in df_debug:
        print(df_debug.event.value_counts())
    else: 
        print("EVENT not in the DataFrame yet...")


## === 📦 12.2. FIT STATIC MODEL (DEMOGRAPHICS ONLY) ===


In [ ]:
# === 12.2. FIT STATIC MODEL (DEMOGRAPHICS ONLY) ===
# Uses one row per officer (last interval) with static demographics only
# This provides a baseline for comparison with the full time-varying model
with notebook_cell_logging('12_2'): 
    if CELL12_2:
        print_v("\n📊 12.2: Fitting static model (demographics only)...")
        static_act = "12.2: Fit static model"
        static_timer = time_start(static_act, nest=1)
        # Create static dataset:  one row per officer (last interval)
        # Group by officer and take the last interval (highest stop_time)
        df_static = df_cox.sort_values('stop_time').groupby('pid_pde').tail(1).copy()
        # Define static variables using COLUMN_CONFIG from CELL 11
        # Combine basic demographic cols (sex, yg) with static cols (age_cpt, etc.)
        # Exclude survival analysis columns (pid_pde, start_time, stop_time, event)
        static_model_vars = []
        # Get static variables from COLUMN_CONFIG (required from CELL 11)
        # Basic demographic cols that are static (exclude survival analysis cols)
        basic_static = [v for v in COLUMN_CONFIG.get('basic_demographic_cols', [])
                       if v not in ['pid_pde', 'start_time', 'stop_time', 'event']]
        # Static columns from config
        config_static = COLUMN_CONFIG.get('static_cols', [])
        # Combine and filter to what's available in df_static
        potential_static_vars = list(set(basic_static + config_static))
        for var in potential_static_vars:
            if var in df_static.columns:
                static_model_vars.append(var)       
                
        # Check for existing dummy variables BEFORE creating df_static_model
        # This ensures we include dummy variables if they already exist from CELL 11
        if 'yg' in static_model_vars:
            yg_dummy_cols = [col for col in df_static.columns if col.startswith('yg_')]
            if len(yg_dummy_cols) > 0:
                # Dummy variables already exist, use them instead of original
                print_v(f"  • Found existing yg dummy variables: {yg_dummy_cols}")
                static_model_vars = [v for v in static_model_vars if v != 'yg'] + yg_dummy_cols        
        
        if 'final_job_code' in static_model_vars:
            final_job_dummy_cols = [col for col in df_static.columns if col.startswith('final_job_') and col != 'final_job_code']
            if len(final_job_dummy_cols) > 0:
                # Dummy variables already exist, use them instead of original
                print_v(f"  • Found existing final_job_code dummy variables: {final_job_dummy_cols}")
                static_model_vars = [v for v in static_model_vars if v != 'final_job_code'] + final_job_dummy_cols           
        
        print_v(f"  • Static variables: {static_model_vars}")
        if len(static_model_vars) == 0:
            print_v("⚠️ WARNING: No static variables found!")
            cph_static = None
            static_model_vars = []
            c_index_static = None
        else:
            # Prepare model data (now includes dummy variables if they exist)
            df_static_model = df_static[static_model_vars + ['pid_pde', 'start_time', 'stop_time', 'event']].copy()
            
            # Handle categorical variables that don't have dummy variables yet
            # (Only create dummies if they weren't found above)
            if 'yg' in df_static_model.columns:
                if create_dummies:
                    # Create dummy variables for yg (they don't exist yet)
                    print_v(f"  • Creating yg dummy variables...")
                    yg_dummies = pd.get_dummies(df_static_model['yg'], prefix='yg', drop_first=True)
                    df_static_model = pd.concat([df_static_model.drop(columns=['yg']), yg_dummies], axis=1)
                    # Update static_model_vars list
                    static_model_vars = [v for v in static_model_vars if v != 'yg'] + list(yg_dummies.columns)
                else:
                    print_v(f"  • Skipping yg dummy creation (create_dummies=False); dropping 'yg'")
                    df_static_model = df_static_model.drop(columns=['yg'])
                    static_model_vars = [v for v in static_model_vars if v != 'yg']
                
            if 'final_job_code' in df_static_model.columns:
                if create_dummies:
                    # Create dummy variables for final_job_code (they don't exist yet)
                    print_v(f"  • Creating final_job_code dummy variables...")
                    final_job_dummies = pd.get_dummies(df_static_model['final_job_code'], prefix='final_job', drop_first=True)
                    df_static_model = pd.concat([df_static_model.drop(columns=['final_job_code']), final_job_dummies], axis=1)
                    # Update static_model_vars list
                    static_model_vars = [v for v in static_model_vars if v != 'final_job_code'] + list(final_job_dummies.columns)            
                else:
                    print_v(f"  • Skipping final_job_code dummy creation (create_dummies=False); dropping 'final_job_code'")
                    df_static_model = df_static_model.drop(columns=['final_job_code'])
                    static_model_vars = [v for v in static_model_vars if v != 'final_job_code']
                    
            if not create_dummies:
                non_numeric_cols = [
                    c for c in df_static_model.columns
                    if c not in ['pid_pde', 'start_time', 'stop_time', 'event']
                    and not pd.api.types.is_numeric_dtype(df_static_model[c])
                ]
                if non_numeric_cols:
                    print_v(f"  • Dropping non-numeric columns (create_dummies=False): {non_numeric_cols}")
                    df_static_model = df_static_model.drop(columns=non_numeric_cols)
                    static_model_vars = [v for v in static_model_vars if v not in non_numeric_cols]
                    
            # Remove missing values
            df_static_model = df_static_model.dropna()
            valid_indices = df_static_model.index
            
            # Get event and time from df_static_model (after dropna) to ensure alignment
            # Use the same indices that survived dropna()
            # Convert to numpy arrays immediately to avoid pandas Series issues
            static_event = np.asarray((df_static_model['event'] == 1), dtype=bool)
            static_time = np.asarray(df_static_model['stop_time'] - df_static_model['start_time'], dtype=float)
            
            # Ensure they are 1D arrays (flatten if needed)
            static_event = static_event.ravel(); print(f"length of static_event: {len(static_event)}")
            static_time = static_time.ravel(); print(f"length of static_time: {len(static_time)}")
            # Verify array shapes match
            if len(static_event) != len(static_time):
                raise ValueError(f"Mismatch: static_event length ({len(static_event)}) != static_time length ({len(static_time)})")
            
            # Verify arrays are not empty
            if len(static_event) == 0:
                raise ValueError(f"static_event is empty after dropna()")
            # Verify arrays have the correct shape (1D)
            if static_event.ndim != 1 or static_time.ndim != 1:
                raise ValueError(f"Arrays must be 1D: static_event.ndim={static_event.ndim}, static_time.ndim={static_time.ndim}")
                                 
            # Get final model variables (excluding pid_pde, start_time, stop_time, event)
            final_static_vars = [v for v in df_static_model.columns if v not in ['pid_pde', 'start_time', 'stop_time', 'event']]
            
            print_v(f"\n📊 Static Model Data Summary:")
            print_v(f"  • Total officers: {len(df_static_model):,}")
            print_v(f"  • Model variables: {len(final_static_vars):,}")
            print_v(f"  • Variable list: {final_static_vars}")
            
            # Create survival array
            static_surv = Surv.from_arrays(
                event=static_event,
                time=static_time
            )
            
             # Fit static model with regularization fallbacks
            print_v("\n🔍 Attempting static model fit...")
            cph_static = None
            alphas = [0.0, 0.1, 1.0, 10.0]  # Try no regularization, then increasing levels
            
            for alpha in alphas:
                try:
                    cph_static = CoxPHSurvivalAnalysis(alpha=alpha)
                    cph_static.fit(df_static_model[final_static_vars], static_surv)
                    print_v(f"✅ Static model fitted successfully (alpha={alpha})")
                    break
                except Exception as e:
                    if alpha == alphas[-1]:
                        print_v(f"❌ Static model fit failed with all regularization levels: {e}")
                        cph_static = None
                    else:
                        print_v(f"⚠️ Fit failed with alpha={alpha}, trying next...")
            
            # Calculate concordance index
            if cph_static is not None:
                try:
                    # Get predictions and ensure they're numpy arrays
                    static_predictions = cph_static.predict(df_static_model[final_static_vars])
                    static_predictions = np.asarray(static_predictions, dtype=float).ravel()
        
                    # Verify all arrays have matching lengths
                    if len(static_event) != len(static_predictions):
                        raise ValueError(f"Mismatch: static_event length ({len(static_event)}) != predictions length ({len(static_predictions)})")
                    
                    c_index_static = concordance_index_censored(
                        static_event,
                        static_time,
                        static_predictions
                    )[0]
                    print_v(f"  • C-index: {c_index_static:.4f}")
                except Exception as e:
                    print_v(f"  ⚠️ C-index: Could not calculate:  {e}")
                    c_index_static = None
            
                # Save static model
                save_act = "Saving static model"
                save_timer = time_start(save_act, nest=2)
                # with open(os.path.join(cox_model_dir, 'cph_static_model.pkl'), 'wb') as f:
                #     pickle.dump(cph_static, f)
                
                store_pickle(cph_static,'cph_static_model',cox_model_dir)
                # Save coefficients
                coef_df_static = pd.DataFrame({
                    'variable': final_static_vars,
                    'coefficient': cph_static.coef_,
                    'signal_ratio': np.exp(cph_static.coef_)
                })
                coef_df_static.to_csv(os.path.join(cox_results_dir, 'static_model_coefficients.csv'), index=False)
                time_stop(save_timer, action=save_act, nest=2)
                print_v(f"✅ Saved static model and coefficients")
    
                # Update static_model_vars to final list
                static_model_vars = final_static_vars
            else:
                c_index_static = None
                static_model_vars =[]
        time_stop(static_timer, action=static_act, nest=1)
    else:
        print_v(f"⏭️ Skipping 12.2: Static model")
        try:
            cph_static = load_pickle('cph_static_model', cox_results_dir)
            coef_df_static = pd.read_csv(os.path.join(cox_results_dir, 'static_model_coefficients.csv'))
            static_model_vars = coef_df_static['variable'].tolist()
            # Try to get c_index_static from saved results or calculate it
            c_index_static = None
            print_v(f"✅ Loaded static model from saved file")
        except:
            print_v("⚠️ Could not load static model, will skip static model comparisons")
            cph_static = None
            static_model_vars = []
            c_index_static = None
 


In [ ]:
df_cox.columns

## === 📦 12.3. FIT FULL MODEL (STATIC + TIME-VARYING) ===


In [ ]:
# === 12.3. FIT FULL MODEL (STATIC + TIME-VARYING) ===
# Uses interval-level data (df_cox) with time-varying covariates
# This allows covariates to change values across intervals for the same officer
with notebook_cell_logging('12_3'):       
    reload_pipeline_config()
    fig_save_suffix = '_stdz' if STANDARDIZE_CONFIG['use_for_model'] else ''
    if CELL12_3:
        print_v("\n📊 12.3: Fitting full model (static + time-varying)...")
        full_act = "12.3: Fit full model"
        full_timer = time_start(full_act, nest=1)
        
        # Define static variables using COLUMN_CONFIG from CELL 11
        # Combine basic demographic cols (sex, yg) with static cols (age_cpt, etc.)
        # Exclude survival analysis columns (pid_pde, start_time, stop_time, event)
        # Basic demographic cols that are static (exclude survival analysis cols)
        basic_static = [v for v in COLUMN_CONFIG.get('basic_demographic_cols', [])
                       if v not in ['pid_pde', 'start_time', 'stop_time', 'event']]
        # Static columns from config (model subset if provided)
        config_static = COLUMN_CONFIG.get('model_static_cols', []) or COLUMN_CONFIG.get('static_cols', [])
        # Combine and filter to what's available in df_static
        potential_static_vars = list(set(basic_static + config_static))
        static_vars = [v for v in potential_static_vars if v in df_cox.columns]
        # Check for existing dummy variables BEFORE creating df_full_model
        # This ensures we include dummy variables if they already exist from CELL 11
        if 'yg' in static_vars:
            yg_dummy_cols = [col for col in df_cox.columns if col.startswith('yg_')]
            if len(yg_dummy_cols) > 0:
                # Dummy variables already exist, use them instead of original
                print_v(f"  • Found existing yg dummy variables: {yg_dummy_cols}")
                static_vars = [v for v in static_vars if v != 'yg'] + yg_dummy_cols        
        
        if 'final_job_code' in static_vars:
            final_job_dummy_cols = [col for col in df_cox.columns if col.startswith('final_job_') and col != 'final_job_code']
            if len(final_job_dummy_cols) > 0:
                # Dummy variables already exist, use them instead of original
                print_v(f"  • Found existing final_job_code dummy variables: {final_job_dummy_cols}")
                static_vars = [v for v in static_vars if v != 'final_job_code'] + final_job_dummy_cols    
                
        # Define time-varying variables using COLUMN_CONFIG from CELL 11
        # Use only model_time_varying_cols (set by run profile); do not add raw _sq from QUADRATIC_CONFIG
        # when the run uses z-scored columns, or you get a mix of z_ and raw vars and unstable 12.6A plots.
        potential_tv_vars = COLUMN_CONFIG.get('model_time_varying_cols', []) or COLUMN_CONFIG.get('time_varying_cols', [])
        print_v(f"  • Potential time-varying variables from COLUMN_CONFIG: {potential_tv_vars}")
        print_v(f"  • Available columns in df_cox: {sorted(df_cox.columns.tolist())}")
        time_varying_vars = [v for v in potential_tv_vars if v in df_cox.columns]
        
        if len(time_varying_vars) < len(potential_tv_vars):
            missing_tv = [v for v in potential_tv_vars if v not in df_cox.columns]
            print_v(f"⚠️ Time-varying variables not found in df_cox: {missing_tv}!")
            
        print_v(f"  • Static variables: {static_vars}")
        print_v(f"  • Time-varying variables: {time_varying_vars}")
        
        if len(time_varying_vars) == 0:
            print_v("⚠️ WARNING: No time-varying variables found!")
            print_v("   Full model will be same as static model")
        
        # Prepare data for full model (use ALL intervals, not just last one)
        df_full = df_cox.copy()
        
        # Create survival object for time-varying model
        # For time-varying, we need start_time and stop_time (interval boundaries)
        # Event: 1 if promoted, 0 otherwise (censored or attrited)
        full_event = (df_full['event'] == 1).astype(bool)
        
        # For scikit-survival with time-varying covariates, we use stop_time - start_time as duration
        # and start_time as the entry time (left-truncation)
        # However, scikit-survival's CoxPHSurvivalAnalysis expects a single time vector
        # So we use the interval duration: stop_time - start_time
        full_time = df_full['stop_time'] - df_full['start_time']
        
        # Prepare model data with both static and time-varying variables
        # (static_vars now eincludes dummy variables if they exist from CELL 11)
        model_vars = static_vars + time_varying_vars
        df_full_model = df_full[model_vars + ['pid_pde', 'start_time', 'stop_time']].copy()
        # Handle categorical variables that don't have dummy variables yet
        # (Only create dummies if the original categorical column is still present)
        if 'yg' in df_full_model.columns:
            if create_dummies:
                # Create dummy variables for yg (they don't exist yet)
                print_v(f"  • Creating yg dummy variables...")
                yg_dummies = pd.get_dummies(df_full_model['yg'], prefix='yg', drop_first=True)
                df_full_model = pd.concat([df_full_model.drop(columns=['yg']), yg_dummies], axis=1)
                # Update static_vars and model_vars lists
                static_vars = [v for v in static_vars if v != 'yg'] + list(yg_dummies.columns)
                model_vars = static_vars + [v for v in time_varying_vars if v != 'yg']
            else:
                print_v(f"  • Skipping yg dummy creation (create_dummies=False); dropping 'yg'")
                df_full_model = df_full_model.drop(columns=['yg'])
                static_vars = [v for v in static_vars if v != 'yg']
                model_vars = static_vars + [v for v in time_varying_vars if v != 'yg']
  
        if 'final_job_code' in df_full_model.columns:        
            if create_dummies:                
                # Create dummy variables for final_job_code (they don't exist yet)
                print_v(f"  • Creating final_job_code dummy variables...")
                final_job_dummies = pd.get_dummies(df_full_model['final_job_code'], prefix='final_job', drop_first=True)
                df_full_model = pd.concat([df_full_model.drop(columns=['final_job_code']), final_job_dummies], axis=1)
                # Update static_vars and model_vars lists
                static_vars = [v for v in static_vars if v != 'final_job_code'] + list(final_job_dummies.columns)
                model_vars = static_vars + [v for v in time_varying_vars if v != 'final_job_code']        
            else:
                print_v(f"  • Skipping final_job_code dummy creation (create_dummies=False); dropping 'final_job_code'")
                df_full_model = df_full_model.drop(columns=['final_job_code'])
                static_vars = [v for v in static_vars if v != 'final_job_code']
                model_vars = static_vars + [v for v in time_varying_vars if v != 'final_job_code']
                          
        if not create_dummies:
            non_numeric_cols = [
                c for c in df_full_model.columns
                if c not in ['pid_pde', 'start_time', 'stop_time']
                and not pd.api.types.is_numeric_dtype(df_full_model[c])
            ]
            if non_numeric_cols:
                print_v(f"  • Dropping non-numeric columns (create_dummies=False): {non_numeric_cols}")
                df_full_model = df_full_model.drop(columns=non_numeric_cols)
                model_vars = [v for v in model_vars if v not in non_numeric_cols]
                static_vars = [v for v in static_vars if v not in non_numeric_cols]
                time_varying_vars = [v for v in time_varying_vars if v not in non_numeric_cols]
        
        if 'div_name' in time_varying_vars:
            # Check if div_name dummy variables already exist (from CELL 11)
            div_dummy_cols = [col for col in df_full_model.columns if col.startswith('div_')]
            if len(div_dummy_cols) > 0:
                # Dummy variables already exist, use them
                print_v(f"  • Using existing div_name dummy variables: {div_dummy_cols}")
                df_full_model = df_full_model.drop(columns=['div_name'])
                time_varying_vars = [v for v in time_varying_vars if v != 'div_name'] + div_dummy_cols
                model_vars = static_vars + time_varying_vars
            else:                
                if create_dummies:
                    # Create dummy variables for div_name
                    div_dummies = pd.get_dummies(df_full_model['div_name'], prefix='div', drop_first=True)
                    df_full_model = pd.concat([df_full_model.drop(columns=['div_name']), div_dummies], axis=1)
                    # Update time_varying_vars list
                    time_varying_vars = [v for v in time_varying_vars if v != 'div_name'] + list(div_dummies.columns)
                    model_vars = static_vars + time_varying_vars
                else:
                    print_v(f"  • Skipping div_name dummy creation (create_dummies=False); dropping 'div_name'")
                    df_full_model = df_full_model.drop(columns=['div_name'])
                    time_varying_vars = [v for v in time_varying_vars if v != 'div_name']
                    model_vars = static_vars + time_varying_vars                   
        # print("\n\n$$$$$$$ df_full_model pre-dropna:",df_full_model.shape); print(df_full_model[model_vars].isna().mean().sort_values(ascending=False).head(15))
        store_feather(df_full_model,'df_trash_test')
        # Remove missing values
        df_full_model = df_full_model.dropna()
        valid_indices = df_full_model.index
        full_event = full_event.loc[valid_indices]
        full_time = full_time.loc[valid_indices]
        # print(f"\n$$$$$$$ df_full_model.shape post-dropna:  {df_full_model.shape}");print(df_full_model[model_vars].isna().mean().sort_values(ascending=False).head(10))
        # Convert to numpy arrays for concordance_index_censored
        # Ensure they are 1D arrays with matching lengths
        full_event = np.asarray(full_event, dtype=bool).ravel()
        full_time = np.asarray(full_time, dtype=float).ravel()
        # Verify array shapes match
        if len(full_event) != len(full_time):
            raise ValueError(f"Mismatch: full_event length ({len(full_event)}) != full_time length ({len(full_time)})")
        
        # Get final model variables (excluding pid_pde, start_time, stop_time)
        final_model_vars = [v for v in df_full_model.columns if v not in ['pid_pde', 'start_time', 'stop_time']]
        
        print_v(f"\n📊 Full Model Data Summary:")
        print_v(f"  • Total intervals: {len(df_full_model):,}")
        print_v(f"  • Unique officers: {df_full_model['pid_pde'].nunique():,}")
        print_v(f"  • Average intervals per officer: {len(df_full_model) / df_full_model['pid_pde'].nunique():.1f}")
        print_v(f"  • Model variables: {len(final_model_vars)} ({len(static_vars)} static + {len(time_varying_vars)} time-varying)")
        print_v(f"  • Variable list: {final_model_vars[:10]}{'...' if len(final_model_vars) > 10 else ''}")
        
        # Create survival array for time-varying model
        cox_surv_full = Surv.from_arrays(
            event=full_event,
            time=full_time
        )
        
        # Fit full model with regularization fallbacks
        print_v("\n🔍 Attempting full model fit with time-varying covariates...")
        cph_full = None
        alphas = [0.0, 0.1, 1.0, 10.0]  # Try no regularization, then increasing levels
        
        for alpha in alphas:
            try:
                cph_full = CoxPHSurvivalAnalysis(alpha=alpha)
                cph_full.fit(df_full_model[final_model_vars], cox_surv_full)
                print_v(f"✅ Full model fitted successfully (alpha={alpha})")
                break
            except Exception as e:
                if alpha == alphas[-1]:
                    print_v(f"❌ Full model fit failed with all regularization levels: {e}")
                    raise
                else:
                    print_v(f"⚠️ Fit failed with alpha={alpha}, trying next...")
        
        # Calculate concordance index
        print_v(f"🧮 Calculating Concordance Indexes...")
        CI_act = f"🧮 Calculating Concordance Indexes..." ;CI_timer = time_start(CI_act, nest=2)
        try:
            # Get predictions and ensure they're numpy arrays
            full_predictions = cph_full.predict(df_full_model[final_model_vars])
            full_predictions = np.asarray(full_predictions, dtype=float).ravel()
            # Verify all arrays have matching lengths
            if len(full_event) != len(full_predictions):
                raise ValueError(f"Mismatch: full_event length ({len(full_event)}) != predictions length ({len(full_predictions)})")
                        
            c_index_full = concordance_index_censored(
                full_event,
                full_time,
                full_predictions
            )[0]
            print_v(f"  • C-index: {c_index_full:.4f}")
        except Exception as e:
            print_v(f"  ⚠️ C-index: Could not calculate:  {e}")
        time_stop(CI_timer,action=CI_act,nest=2)
        # Save full model
        save_act = "Saving full model"
        save_timer = time_start(save_act, nest=2)
        # with open(os.path.join(cox_model_dir, 'cph_full_model.pkl'), 'wb') as f:
        #     pickle.dump(cph_full, f)
        
        store_pickle(cph_full,'cph_full_model',cox_model_dir)
        
        # Save coefficients
        coef_df_full = pd.DataFrame({
            'variable': final_model_vars,
            'coefficient': cph_full.coef_,
            'signal_ratio': np.exp(cph_full.coef_)
        })
        coef_df_full.to_csv(os.path.join(cox_results_dir, f'full_model_coefficients{fig_save_suffix}.csv'), index=False)
        # Save full-model data for cox_lifelines_significance.py (p-values, CIs)
        df_for_lifelines = df_full_model[final_model_vars].copy()
        df_for_lifelines['duration'] = full_time
        df_for_lifelines['event'] = full_event.astype(int)
        df_for_lifelines.to_csv(os.path.join(cox_results_dir, f'full_model_for_lifelines{fig_save_suffix}.csv'), index=False)   
        time_stop(save_timer, action=save_act, nest=2)
        print_v(f"✅ Saved full model and coefficients")
        print_v(f"✅ Saved full_model_for_lifelines{fig_save_suffix}.csv (for cox_lifelines_significance.py)")
        
        # Print signal ratios
        print_v(f"\n📊 Full Model Signal Ratios (Top 10):")
        coef_df_full_sorted = coef_df_full.sort_values('signal_ratio', key=abs, ascending=False)
        for _, row in coef_df_full_sorted.head(10).iterrows():
            print_v(f"  • {row['variable']}: SR = {row['signal_ratio']:.4f} (coef = {row['coefficient']:.4f})")
        
        # Compare to static model if available
        if cph_static is not None:
            print_v(f"\n📊 Model Comparison:")
            print_v(f"  • Static model C-index: {c_index_static:.4f}")
            print_v(f"  • Full model C-index: {c_index_full:.4f}")
            improvement = c_index_full - c_index_static
            print_v(f"  • Improvement: {improvement:+.4f} ({improvement/c_index_static*100:+.2f}%)")
        
        time_stop(full_timer, action=full_act, nest=1)
    else:
        print_v("⏭️ Skipping 12.3: Full model")
        try:
            # with open(os.path.join(cox_model_dir, 'cph_full_model.pkl'), 'rb') as f:
            #     cph_full = pickle.load(f)
            load_pickle('cph_full_model', cox_model_dir)
            
            coef_df_full = pd.read_csv(os.path.join(cox_results_dir, f'full_model_coefficients{fig_save_suffix}.csv'))
            final_model_vars = coef_df_full['variable'].tolist()
            print_v(f"✅ Loaded full model from saved file")
        except:
            print_v("⚠️ Could not load full model, will skip full model sections")
            cph_full = None
            final_model_vars = []


In [ ]:

# Lifelines significance (p-values, CIs) for full Cox model
try:
    from cox_lifelines_significance import fit_and_test_significance, run_from_saved_data
    run_sig = True
except ModuleNotFoundError:
    print(" ⚠️  Skipping, cox_lifelines_significance module not found or loaded")
    run_sig = False
if run_sig:
    if cph_full is not None  and final_model_vars:
        try:
            # Use in-memory data if we just ran 12.3
            _df = df_full_model[final_moel_vars].copy()
            _dur = full_time
            _ev = full_event.astype(int)
            summary_df, summary_str = fit_and_test_significance(
                _df, duration=_dur, event=_ev, variables=final_model_vars,
                output_dir=cox_results_dir, save_suffix=fig_save_suffix,
            )
        except NameError:
            # Load from CSV saved by Cell 12.3
            summary_df, summary_str = run_from_saved_data(
                results_dir=cox_results_dir,
                filename="full_model_for_lifelines",
                suffix=fig_save_suffix,
            )
        print("Cox model (lifelines) statistical significance")
        print("="*60)
        print(summary_str)
    else:
        print(" ⚠️  Skipping, run Cell 12.3 first to fit the full model")

## === 📦 12.4. MODEL COMPARISON AND SIGNAL RATIOS ===


In [ ]:
# === 12.4. MODEL COMPARISON AND SIGNAL RATIOS ===
# Visualize model comparison and signal ratios
with notebook_cell_logging('12_4'):       
    if CELL12_4:        
        print_v("\n📊 12.4: Model comparison and signal ratios...")
        comp_act = "12.4: Model comparison"
        comp_timer = time_start(comp_act, nest=1)
        
        # Check if models exist (handle cases where static model might not be defined)
        try:
            static_exists = cph_static is not None and 'c_index_static' in locals()
            full_exists = cph_full is not None and 'c_index_full' in locals()
        except:
            static_exists = False
            full_exists = cph_full is not None
        
        if not full_exists:
            print_v("⚠️ Full model required for comparison. Skipping.")
        else:
            # Create comparison DataFrame
            comparison_data = []
            
            # Static model coefficients (if available)
            if static_exists and hasattr(cph_static, 'coef_'):
                try:
                    static_model_vars_list = static_model_vars if 'static_model_vars' in locals() else []
                    if len(static_model_vars_list) == len(cph_static.coef_):
                        for i, var in enumerate(static_model_vars_list):
                            comparison_data.append({
                                'variable': var,
                                'model': 'Static',
                                'coefficient': cph_static.coef_[i],
                                'signal_ratio': np.exp(cph_static.coef_[i])
                            })
                except:
                    pass
            
            # Full model coefficients
            if hasattr(cph_full, 'coef_'):
                for i, var in enumerate(final_model_vars):
                    comparison_data.append({
                        'variable': var,
                        'model': 'Full',
                        'coefficient': cph_full.coef_[i],
                        'signal_ratio': np.exp(cph_full.coef_[i])
                    })
            
            df_comparison = pd.DataFrame(comparison_data)
            
            # Combine base + _sq into single metric for display (effect at z=1: exp(β₁+β₂))
            coef_map = dict(zip(coef_df_full['variable'], coef_df_full['coefficient']))
            used = set()
            sr_display_rows = []
            for var in coef_df_full['variable'].unique():
                if var in used:
                    continue
                if var.endswith('_sq'):
                    base = var[:-3]
                    if base in coef_map:
                        b1, b2 = coef_map[base], coef_map[var]
                        combined_sr = np.exp(b1 + b2)
                        sr_display_rows.append({'variable': base, 'coefficient': b1 + b2, 'signal_ratio': combined_sr})
                        used.add(base); used.add(var)
                        continue
                # Base (or standalone): skip if this base has a _sq pair — we already added combined row above
                if (var + '_sq') in coef_map:
                    used.add(var)
                    continue
                row = coef_df_full[coef_df_full['variable'] == var].iloc[0]
                sr_display_rows.append({'variable': row['variable'], 'coefficient': row['coefficient'], 'signal_ratio': row['signal_ratio']})
                used.add(var)
            sr_display_df = pd.DataFrame(sr_display_rows)
            
            # Create visualization
            fig, axes = plt.subplots(2, 2, figsize=(16, 12))
            
            # Plot 1: C-index comparison
            ax1 = axes[0, 0]
            if static_exists:
                models = ['Static', 'Full']
                c_indices = [c_index_static, c_index_full]
            else:
                models = ['Full']
                c_indices = [c_index_full]
            colors = ['steelblue', 'darkgreen'][:len(models)]
            bars = ax1.bar(models, c_indices, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
            ax1.set_ylabel('C-Index', fontsize=12, fontweight='bold')
            ax1.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
            ax1.set_ylim([0, 1])
            ax1.grid(axis='y', alpha=0.3)
            # Add value labels on bars
            for bar, c_idx in zip(bars, c_indices):
                height = bar.get_height()
                ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f'{c_idx:.4f}', ha='center', va='bottom', fontweight='bold')
            
            # Plot 2: Signal ratios (top 15 from full model; base+_sq combined as one)
            ax2 = axes[0, 1]
            top_vars = sr_display_df.sort_values('signal_ratio', key=abs, ascending=False).head(15)
            y_pos = np.arange(len(top_vars))
            colors_sr = ['red' if sr < 1 else 'green' for sr in top_vars['signal_ratio']]
            bars = ax2.barh(y_pos, top_vars['signal_ratio'], color=colors_sr, alpha=0.7, edgecolor='black')
            ax2.set_yticks(y_pos)
            ax2.set_yticklabels(top_vars['variable'], fontsize=9)
            ax2.set_xlabel('Signal Ratio (exp(β))', fontsize=12, fontweight='bold')
            ax2.set_title('Top 15 Signal Ratios (Full Model)', fontsize=14, fontweight='bold')
            ax2.axvline(x=1, color='black', linestyle='--', linewidth=1, alpha=0.5)
            ax2.grid(axis='x', alpha=0.3)
            
            # Plot 3: Coefficient comparison (for variables in both models)
            ax3 = axes[1, 0]
            if static_exists:
                try:
                    static_model_vars_list = static_model_vars if 'static_model_vars' in locals() else []
                    common_vars = set(static_model_vars_list) & set(final_model_vars)
                    if len(common_vars) > 0:
                        common_vars_list = sorted(list(common_vars))[:15]  # Top 15
                        static_coefs = [cph_static.coef_[static_model_vars_list.index(v)] for v in common_vars_list]
                        full_coefs = [cph_full.coef_[final_model_vars.index(v)] for v in common_vars_list]
                        x = np.arange(len(common_vars_list))
                        width = 0.35
                        ax3.bar(x - width/2, static_coefs, width, label='Static', color='steelblue', alpha=0.7)
                        ax3.bar(x + width/2, full_coefs, width, label='Full', color='darkgreen', alpha=0.7)
                        ax3.set_ylabel('Coefficient', fontsize=12, fontweight='bold')
                        ax3.set_title('Coefficient Comparison (Common Variables)', fontsize=14, fontweight='bold')
                        ax3.set_xticks(x)
                        ax3.set_xticklabels(common_vars_list, rotation=45, ha='right', fontsize=8)
                        ax3.legend()
                        ax3.grid(axis='y', alpha=0.3)
                        ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
                    else:
                        ax3.text(0.5, 0.5, 'No common variables\nbetween models', 
                                ha='center', va='center', transform=ax3.transAxes, fontsize=12)
                        ax3.set_title('Coefficient Comparison', fontsize=14, fontweight='bold')
                except:
                    ax3.text(0.5, 0.5, 'Static model\nnot available', 
                            ha='center', va='center', transform=ax3.transAxes, fontsize=12)
                    ax3.set_title('Coefficient Comparison', fontsize=14, fontweight='bold')
            else:
                ax3.text(0.5, 0.5, 'Static model\nnot available', 
                        ha='center', va='center', transform=ax3.transAxes, fontsize=12)
                ax3.set_title('Coefficient Comparison', fontsize=14, fontweight='bold')
            
            # Plot 4: Signal ratio distribution (base+_sq combined)
            ax4 = axes[1, 1]
            signal_ratios = sr_display_df['signal_ratio']
            ax4.hist(signal_ratios, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
            ax4.axvline(x=1, color='red', linestyle='--', linewidth=2, label='No effect (SR=1)')
            ax4.set_xlabel('Signal Ratio', fontsize=12, fontweight='bold')
            ax4.set_ylabel('Frequency', fontsize=12, fontweight='bold')
            ax4.set_title('Signal Ratio Distribution (Full Model)', fontsize=14, fontweight='bold')
            ax4.legend()
            ax4.grid(axis='y', alpha=0.3)
            
            plt.tight_layout()
            save_path = os.path.join(cox_results_dir, f'model_comparison{fig_save_suffix}.png')
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.show()
            plt.close()
            print_v(f"✅ Saved model comparison plot: {save_path}")
            
            # Save comparison table
            try:
                static_vars_count = len(static_model_vars) if 'static_model_vars' in locals() else 0
                static_intervals = len(df_pipeline_12_01_prepared) if 'df_pipeline_12_01_prepared' in locals() else 0
                static_c_index = c_index_static if static_exists else None
            except:
                static_vars_count = 0
                static_intervals = 0
                static_c_index = None
            
            comparison_table = pd.DataFrame({
                'Model': ['Static', 'Full'] if static_exists else ['Full'],
                'C-Index': [static_c_index, c_index_full] if static_exists else [c_index_full],
                'Variables': [static_vars_count, len(final_model_vars)] if static_exists else [len(final_model_vars)],
                'Intervals': [static_intervals, len(df_full_model)] if static_exists else [len(df_full_model)]
            }) 
            
            comparison_table.to_csv(os.path.join(cox_results_dir, f'model_comparison_table{fig_save_suffix}.csv'), index=False)
            print_v(f"✅ Saved comparison table: model_comparison_table{fig_save_suffix}.csv")
        
        time_stop(comp_timer, action=comp_act, nest=1)
    else:
        print_v("⏭️ Skipping 12.4: Model comparison")
    


## === 📦 12.5. COMPETING RISKS ANALYSIS ===


In [ ]:
# === 12.5. COMPETING RISKS ANALYSIS ===
# Analyze competing risks (promotion vs attrition) using scikit-survival
with notebook_cell_logging('12_5'):    
    if CELL12_5:
        print_v("\n📊 12.5: Competing risks analysis...")
        cr_act = "12.5: Competing risks analysis"
        cr_timer = time_start(cr_act, nest=1)
        
        if cph_full is None:
            print_v("⚠️ Full model required for competing risks analysis. Skipping.")
        else:
            # Prepare data for competing risks
            # We need to create separate models for promotion (event=1) and attrition (event=2)
            # Use df_full to get event column (df_full_model doesn't have it)
            
            # Promotion model (event=1 vs all others)
            print_v("\n  🔄 Fitting promotion model (event=1)...")
            promo_act = "🔄 Fitting promotion model (event=1)..."; promo_timer = time_start(promo_act, nest=2)
            promo_event = (df_full.loc[df_full_model.index, 'event'] == 1).astype(bool)
            promo_time = full_time.copy()
            
            # Convert to numpy arrays for concordance_index_censored
            # Ensure they are 1D arrays with matching lengths
            promo_event = np.asarray(promo_event, dtype=bool).ravel()
            promo_time = np.asarray(promo_time, dtype=float).ravel()
            # Verify array shapes match
            if len(promo_event) != len(promo_time):
                raise ValueError(f"Mismatch: promo_event length ({len(promo_event)}) != promo_time length ({len(promo_time)})")
        
            promo_surv = Surv.from_arrays(event=promo_event, time=promo_time)
            
            try:
                cph_promo = CoxPHSurvivalAnalysis(alpha=0.1)
                cph_promo.fit(df_full_model[final_model_vars], promo_surv)
                # Model fitted successfully, now try C-index calculation
                try:    
                    # Get predictions and ensure they're numpy arrays
                    promo_predictions = cph_promo.predict(df_full_model[final_model_vars])
                    promo_predictions = np.asarray(promo_predictions, dtype=float).ravel()
        
                    # Verify all arrays have matching lengths
                    if len(promo_event) != len(promo_predictions):
                        raise ValueError(f"Mismatch: promo_event length ({len(promo_event)}) != predictions length ({len(promo_predictions)})")
                            
                    promo_c_index = concordance_index_censored(
                        promo_event,
                        promo_time,
                        promo_predictions
                    )[0]
                    print_v(f"  ✅ Promotion model C-index: {promo_c_index:.4f}")
                except Exception as epc:
                    # Model fitted but C-index calculation failed
                    print_v(f"  ⚠️ C-index calculation failed: {epc}")
                    promo_c_index = None
                    # Note: cph_promo is still valid and can be used                    
            except Exception as ep:
                # Model fitting failed
                print_v(f"  ⚠️ Promotion model fit failed: {ep}")
                cph_promo = None
                promo_c_index = None
            time_stop(promo_timer,action=promo_act,nest=2)
            
            # Attrition model (event=2 vs all others)
            print_v("\n  🔄 Fitting attrition model (event=2)...")
            attmo_act = "🔄 Fitting attrition model (event=2)..."; attmo_timer = time_start(attmo_act, nest=2)
            attr_event = (df_full.loc[df_full_model.index, 'event'] == 2).astype(bool)
            attr_time = full_time.copy()
            
            # Convert to numpy arrays for concordance_index_censored
            # Ensure they are 1D arrays with matching lengths
            attr_event = np.asarray(attr_event, dtype=bool).ravel()
            attr_time = np.asarray(attr_time, dtype=float).ravel()
            # Verify array shapes match
            if len(attr_event) != len(attr_time):
                raise ValueError(f"Mismatch: attr_event length ({len(attr_event)}) != attr_time length ({len(attr_time)})")
            
            attr_surv = Surv.from_arrays(event=attr_event, time=attr_time)
            
            try:
                cph_attr = CoxPHSurvivalAnalysis(alpha=0.1)
                cph_attr.fit(df_full_model[final_model_vars], attr_surv)
                # Model fitted successfully, now try C-index calculation
                try:    
                    # Get predictions and ensure they're numpy arrays
                    attr_predictions = cph_attr.predict(df_full_model[final_model_vars])
                    attr_predictions = np.asarray(attr_predictions, dtype=float).ravel()
        
                    # Verify all arrays have matching lengths
                    if len(attr_event) != len(attr_predictions):
                        raise ValueError(f"Mismatch: attr_event length ({len(attr_event)}) != predictions length ({len(attr_predictions)})")
                            
                    attr_c_index = concordance_index_censored(
                        attr_event,
                        attr_time,
                        attr_predictions
                    )[0]
                    print_v(f"  ✅ Attrition model C-index: {attr_c_index:.4f}")
                except Exception as eac:
                    # Model fitted but C-index calculation failed
                    print_v(f"  ⚠️ C-index calculation failed: {eac}")
                    attr_c_index = None
                    # Note: cph_attr is still valid and can be used
            except Exception as ea:
            # Model fitting failed
                print_v(f"  ⚠️ Attrition model fit failed: {ea}")
                cph_attr = None
                attr_c_index = None
            time_stop(attmo_timer,action=attmo_act,nest=2)
            
            # Compare coefficients between promotion and attrition models
            if cph_promo is not None and cph_attr is not None:
                print_v("\n  📊 Comparing promotion vs attrition coefficients...")
                
                promo_coefs = pd.DataFrame({
                    'variable': final_model_vars,
                    'coefficient_promo': cph_promo.coef_,
                    'signal_ratio_promo': np.exp(cph_promo.coef_)
                })
                
                attr_coefs = pd.DataFrame({
                    'variable': final_model_vars,
                    'coefficient_attr': cph_attr.coef_,
                    'signal_ratio_attr': np.exp(cph_attr.coef_)
                })
                            
                cr_comparison = promo_coefs.merge(attr_coefs, on='variable')
                cr_comparison['difference'] = cr_comparison['coefficient_promo'] - cr_comparison['coefficient_attr']
                
                # Save competing risks comparison
                cr_comparison.to_csv(os.path.join(cox_results_dir, f'competing_risks_comparison{fig_save_suffix}.csv'), index=False)
                print_v(f"  ✅ Saved competing risks comparison: competing_risks_comparison{fig_save_suffix}.csv")
                
                # Combine base + _sq into one row per variable for display (effect at z=1)
                cr_by_var = cr_comparison.set_index('variable')
                used = set()
                cr_display_rows = []
                for var in cr_comparison['variable'].unique():
                    if var in used:
                        continue
                    if var.endswith('_sq'):
                        base = var[:-3]
                        if base in cr_by_var.index:
                            rb = cr_by_var.loc[base].iloc[0] if isinstance(cr_by_var.loc[base], pd.DataFrame) else cr_by_var.loc[base]
                            rs = cr_by_var.loc[var].iloc[0] if isinstance(cr_by_var.loc[var], pd.DataFrame) else cr_by_var.loc[var]
                            cr_display_rows.append({
                                'variable': base,
                                'coefficient_promo': rb['coefficient_promo'] + rs['coefficient_promo'],
                                'coefficient_attr': rb['coefficient_attr'] + rs['coefficient_attr'],
                                'signal_ratio_promo': np.exp(rb['coefficient_promo'] + rs['coefficient_promo']),
                                'signal_ratio_attr': np.exp(rb['coefficient_attr'] + rs['coefficient_attr']),
                            })
                            used.add(base); used.add(var)
                            continue
                    if (var + '_sq') in cr_by_var.index:
                        used.add(var)
                        continue
                    row = cr_comparison[cr_comparison['variable'] == var].iloc[0]
                    cr_display_rows.append({
                        'variable': row['variable'],
                        'coefficient_promo': row['coefficient_promo'],
                        'coefficient_attr': row['coefficient_attr'],
                        'signal_ratio_promo': row['signal_ratio_promo'],
                        'signal_ratio_attr': row['signal_ratio_attr'],
                    })
                    used.add(var)
                cr_display = pd.DataFrame(cr_display_rows)
                cr_display['difference'] = cr_display['coefficient_promo'] - cr_display['coefficient_attr']

                # Print top differences (base+_sq combined)
                print_v("\n  📊 Top 10 Variables with Largest Coefficient Differences (base+_sq combined):")
                top_diff = cr_display.sort_values('difference', key=abs, ascending=False).head(10)
                for _, row in top_diff.iterrows():
                    print_v(f"    • {row['variable']}: Promo={row['coefficient_promo']:.4f}, Attr={row['coefficient_attr']:.4f}, Diff={row['difference']:.4f}")
                
                # Create visualization (use combined cr_display)
                fig, axes = plt.subplots(1, 2, figsize=(16, 6))
                
                # Plot 1: Coefficient comparison
                ax1 = axes[0]
                top_15 = cr_display.sort_values('difference', key=abs, ascending=False).head(15)
                x = np.arange(len(top_15))
                width = 0.35
                ax1.bar(x - width/2, top_15['coefficient_promo'], width, label='Promotion', color='green', alpha=0.7)
                ax1.bar(x + width/2, top_15['coefficient_attr'], width, label='Attrition', color='red', alpha=0.7)
                ax1.set_ylabel('Coefficient', fontsize=12, fontweight='bold')
                ax1.set_title('Promotion vs Attrition Coefficients (Top 15)', fontsize=14, fontweight='bold')
                ax1.set_xticks(x)
                ax1.set_xticklabels(top_15['variable'], rotation=45, ha='right', fontsize=9)
                ax1.legend()
                ax1.grid(axis='y', alpha=0.3)
                ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
                
                # Plot 2: Signal ratio comparison
                ax2 = axes[1]
                ax2.scatter(top_15['signal_ratio_promo'], top_15['signal_ratio_attr'], 
                           s=100, alpha=0.6, edgecolors='black', linewidth=1)
                # Add diagonal line
                max_sr = max(top_15['signal_ratio_promo'].max(), top_15['signal_ratio_attr'].max())
                min_sr = min(top_15['signal_ratio_promo'].min(), top_15['signal_ratio_attr'].min())
                ax2.plot([min_sr, max_sr], [min_sr, max_sr], 'k--', alpha=0.5, label='Equal effect')
                # Add variable labels
                for _, row in top_15.iterrows():
                    ax2.annotate(row['variable'], 
                               (row['signal_ratio_promo'], row['signal_ratio_attr']),
                               fontsize=8, alpha=0.7)
                ax2.set_xlabel('Promotion Signal Ratio', fontsize=12, fontweight='bold')
                ax2.set_ylabel('Attrition Signal Ratio', fontsize=12, fontweight='bold')
                ax2.set_title('Signal Ratio Comparison (Top 15)', fontsize=14, fontweight='bold')
                ax2.legend()
                ax2.grid(alpha=0.3)
                
                plt.tight_layout()
                save_path = os.path.join(cox_results_dir, f'competing_risks_analysis{fig_save_suffix}.png')
                plt.savefig(save_path, dpi=300, bbox_inches='tight')
                plt.show()
                plt.close()
                print_v(f"  ✅ Saved competing risks plot: {save_path}")
        
        time_stop(cr_timer, action=cr_act, nest=1)
    else:
        print_v("⏭️ Skipping 12.5: Competing risks analysis")
    


## === 📦 12.6. PARTIAL EFFECTS PLOTS ===


In [ ]:
# === 12.6. PARTIAL EFFECTS PLOTS ===
# Create partial effects plots showing the effect of each variable on survival
reload_pipeline_config()
with notebook_cell_logging('12_6'):    
    if CELL12_6:
        print_v("\n📊 12.6: Partial effects plots...")
        pe_act = "12.6: Partial effects plots"
        pe_timer = time_start(pe_act, nest=1)
        
        if cph_full is None:
            print_v("⚠️ Full model required for partial effects plots. Skipping.")
        else:
            # Select key variables for partial effects plots
            # Focus on continuous variables and important categorical variables
            key_vars = []
            for var in final_model_vars:
                if var in df_full_model.columns:
                    if df_full_model[var].dtype in ['float64', 'int64']:
                        # Continuous variable
                        if df_full_model[var].nunique() > 5:  # Actually continuous
                            key_vars.append(var)
                    # Also include top signal ratio variables
                    elif var in coef_df_full.sort_values('signal_ratio', key=abs, ascending=False).head(10)['variable'].values:
                        key_vars.append(var)
            
            # Limit to top 12 variables to avoid too many plots
            if len(key_vars) > 12:
                # Prioritize by signal ratio
                top_sr_vars = coef_df_full.sort_values('signal_ratio', key=abs, ascending=False).head(12)['variable'].tolist()
                key_vars = [v for v in top_sr_vars if v in key_vars][:12]
            
            # One plot per logical variable: drop _sq (12.6A does combined for pairs) and drop base when it has _sq
            key_vars_display = []
            for var in list(dict.fromkeys(key_vars)):
                if var.endswith('_sq'):
                    continue
                if (var + '_sq') in final_model_vars:
                    continue
                key_vars_display.append(var)
            
            print_v(f"  • Creating partial effects plots for {len(key_vars_display)} variables (base+_sq pairs in 12.6A)")
            
            # Create subplots
            n_vars = len(key_vars_display)
            n_cols = 3
            n_rows = (n_vars + n_cols - 1) // n_cols
            
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
            if n_rows == 1:
                axes = axes.reshape(1, -1)
            axes = axes.flatten()
            
            for idx, var in enumerate(key_vars_display):
                ax = axes[idx]
                
                if var not in df_full_model.columns:
                    ax.text(0.5, 0.5, f'Variable\n{var}\nnot found', 
                           ha='center', va='center', transform=ax.transAxes)
                    ax.set_title(var, fontsize=10, fontweight='bold')
                    continue
                
                var_data = df_full_model[var]
                
                if var_data.dtype in ['float64', 'int64'] and var_data.nunique() > 5:
                    # Continuous variable: create range and predict
                    var_min = var_data.min()
                    var_max = var_data.max()
                    var_range = np.linspace(var_min, var_max, 100)
                    
                    # Create baseline data (median values for other variables)
                    baseline_data = df_full_model[final_model_vars].median().to_dict()
                    baseline_df = pd.DataFrame([baseline_data] * len(var_range))
                    baseline_df[var] = var_range
                    
                    # Predict hazard ratios
                    try:
                        predictions = cph_full.predict(baseline_df[final_model_vars])
                        # Convert to hazard ratio (relative to median)
                        median_pred = cph_full.predict(df_full_model[final_model_vars].median().to_frame().T)[0]
                        hazard_ratios = np.exp(predictions - median_pred)
                        
                        ax.plot(var_range, hazard_ratios, linewidth=2, color='steelblue')
                        ax.axhline(y=1, color='red', linestyle='--', linewidth=1, alpha=0.5, label='No effect')
                        ax.set_xlabel(var, fontsize=10, fontweight='bold')
                        ax.set_ylabel('Hazard Ratio', fontsize=10, fontweight='bold')
                        ax.set_title(f'Partial Effect: {var}', fontsize=11, fontweight='bold')
                        ax.grid(alpha=0.3)
                        ax.legend(fontsize=8)
                    except Exception as e:
                        ax.text(0.5, 0.5, f'Prediction\nfailed:\n{str(e)[:50]}', 
                               ha='center', va='center', transform=ax.transAxes, fontsize=8)
                        ax.set_title(var, fontsize=10, fontweight='bold')
                else:
                    # Categorical or discrete: bar plot of coefficients
                    if var in final_model_vars:
                        var_idx = final_model_vars.index(var)
                        coef = cph_full.coef_[var_idx]
                        sr = np.exp(coef)
                        
                        ax.barh([0], [sr], color='steelblue' if sr > 1 else 'red', alpha=0.7, edgecolor='black')
                        ax.axvline(x=1, color='black', linestyle='--', linewidth=1, alpha=0.5)
                        ax.set_xlabel('Signal Ratio', fontsize=10, fontweight='bold')
                        ax.set_yticks([0])
                        ax.set_yticklabels([var[:30]], fontsize=9)
                        ax.set_title(f'Signal Ratio: {var}', fontsize=11, fontweight='bold')
                        ax.text(sr, 0, f'{sr:.3f}', ha='left' if sr > 1 else 'right', 
                               va='center', fontweight='bold', fontsize=9)
                        ax.grid(axis='x', alpha=0.3)
                    else:
                        ax.text(0.5, 0.5, f'Variable\n{var}\nnot in model', 
                               ha='center', va='center', transform=ax.transAxes)
                        ax.set_title(var, fontsize=10, fontweight='bold')
            
            # Hide unused subplots
            for idx in range(n_vars, len(axes)):
                axes[idx].axis('off')
            
            plt.tight_layout()
            save_path = os.path.join(cox_results_dir, f'partial_effects_plots{fig_save_suffix}.png')
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.show()
            plt.close()
            print_v(f"  ✅ Saved partial effects plots: {save_path}")
        
        time_stop(pe_timer, action=pe_act, nest=1)
    else:
        print_v("⏭️ Skipping 12.6: Partial effects plots")


In [ ]:
final_model_vars

## === 📈 12.6A. COMBINED EFFECT (LINEAR + QUADRATIC) ===


In [ ]:
# === 12.6A. COMBINED EFFECT (LINEAR + QUADRATIC) ===
reload_pipeline_config()
with notebook_cell_logging('12.6A'):
    if CELL12_6:
        print_v("\n📊 12.6A: Combined linear + quadratic effect plot...")
        combo_act = "12.6A: Combined effect plot"
        combo_timer = time_start(combo_act, nest=1)

        if cph_full is None:
            print_v(f"⚠️ Full model required for combined effect plot.  Skipping.")
        else:
            # Collect all linear+quadratic pairs from the model (so we plot each quadratic's partial effect)
            quadratic_pairs = []
            _preferred = globals().get('COMBINED_EFFECT_BASE_COL', None)
            if _preferred and _preferred in final_model_vars:
                _sq = f"{_preferred}_sq"
                if _sq in final_model_vars:
                    quadratic_pairs.append((_preferred, _sq))
            for v in final_model_vars:
                if v.endswith('_sq'):
                    candidate_base = v[:-3]
                    if candidate_base in final_model_vars and (candidate_base, v) not in quadratic_pairs:
                        quadratic_pairs.append((candidate_base, v))

            if not quadratic_pairs:
                print_v("⚠️ Combined effect plot skipped (no linear+quadratic pair found in model)")
            else:
                for combo_base, combo_sq in quadratic_pairs:
                    print_v(f"  • Combined effect pair: {combo_base} + {combo_sq}")
                    if combo_base not in df_full_model.columns or combo_sq not in df_full_model.columns:
                        print_v(f"⚠️ Missing columns in df_full_model: {combo_base} or {combo_sq}")                
                    else:
                        var_data = df_full_model[combo_base]
                        var_min = var_data.min()
                        var_max = var_data.max()
                        var_range = np.linspace(var_min, var_max, 200)
    
                        baseline_data = df_full_model[final_model_vars].median().to_dict()
                        baseline_df = pd.DataFrame([baseline_data] * len(var_range))
                        baseline_df[combo_base] = var_range
                        baseline_df[combo_sq] = var_range ** 2
    
                        # Report coefficients and implied turning point
                        try:
                            if hasattr(cph_full, "params_"):
                                params = cph_full.params_
                                beta1 = float(params[combo_base])
                                beta2 = float(params[combo_sq])
                            elif hasattr(cph_full, "coef_"):
                                coefs = np.ravel(cph_full.coef_)
                                if hasattr(cph_full, "feature_names_in_"):
                                    names = list(cph_full.feature_names_in_)
                                else:
                                    names = list(final_model_vars)
                                coef_map = dict(zip(names, coefs))
                                beta1 = float(coef_map[combo_base])
                                beta2 = float(coef_map[combo_sq])
                            else:
                                raise AttributeError("Model has no params_ or coef_ attributes")                        
                            
                            print_v(f"  • Coef({combo_base}) = {beta1:.4f}")
                            print_v(f"  • Coef({combo_sq}) = {beta2:.4f}")
                            if beta2 != 0:
                                turning_point = -beta1 / (2*beta2)
                                in_range = var_min <= turning_point <= var_max
                                print_v(
                                    f"  • Turning point at {turning_point:.3f} "
                                    f"(in range: {in_range})"
                                )
                            else:
                                print_v(f"  • Turning point not defined (beta2 = 0)")
                        except Exception as e:
                            print_v(f"⚠️ Coefficient summary failed: {e}")
                                    
                        try:
                            predictions = cph_full.predict(baseline_df[final_model_vars])
                            median_pred = cph_full.predict(df_full_model[final_model_vars].median().to_frame().T)[0]
                            hazard_ratios = np.exp(predictions - median_pred)
    
                            fig, ax = plt.subplots(1, 1, figsize=(7, 4))
                            ax.plot(var_range, hazard_ratios, linewidth=2, color='steelblue')
                            ax.axhline(y=1, color='red', linestyle='--', linewidth=1, alpha=0.5, label='No effect')
                            ax.set_xlabel(combo_base, fontsize=10, fontweight='bold')
                            ax.set_ylabel('Hazard Ratio', fontsize=10, fontweight='bold')
                            ax.set_title(f'Combined Effect: {combo_base} + {combo_sq}', fontsize=11, fontweight='bold')
                            ax.grid(alpha=0.3)
                            ax.legend(fontsize=8)
                            fig.text(
                                0.01, -0.08,
                                "Full model effect after adjusting for all covariates. "
                                "When other signals overlap, the non-linear hump can flatten.",
                                fontsize=9, fontweight='bold', ha='left', va='top', wrap=True
                            )
                            
                            combo_path = os.path.join(cox_results_dir, f'combined_effect_{combo_base}{fig_save_suffix}.png')
                            plt.tight_layout()
                            plt.savefig(combo_path, dpi=300, bbox_inches="tight")
                            plt.show()
                            plt.close()
                            # print_v(f"  ✅ Saved combined effect plot: {combo_path}")
                        except Exception as e:
                            print_v(f"⚠️ Combined effect plot failed: {e}")
                        
                        # Minimal model (linear + quadratic only)
                        try:
                            min_vars = [combo_base, combo_sq]
                            cph_min = CoxPHSurvivalAnalysis(alpha=0.0)
                            cph_min.fit(df_full_model[min_vars].to_numpy(), cox_surv_full)
                            min_pred = cph_min.predict(baseline_df[min_vars].to_numpy())
                            min_median = cph_min.predict(df_full_model[min_vars].median().to_frame().T.to_numpy())[0]
                            min_hr = np.exp(np.asarray(min_pred, dtype=float) - min_median)
    
                            fig_m, ax_m = plt.subplots(1, 1, figsize=(7, 4))
                            ax_m.plot(var_range, min_hr, linewidth=2, color='darkgreen')
                            ax_m.axhline(y=1, color='red', linestyle='--', linewidth=1, alpha=0.5, label='No effect')
                            ax_m.set_xlabel(combo_base, fontsize=10, fontweight='bold')
                            ax_m.set_ylabel('Hazard Ratio', fontsize=10, fontweight='bold')
                            ax_m.set_title(f'Minimal Model: {combo_base} + {combo_sq}', fontsize=11, fontweight='bold')
                            ax_m.grid(alpha=0.3)
                            ax_m.legend(fontsize=8)
                            fig_m.text(
                                0.01, -0.08,
                                "Pool mean plus square only; shows the unadjusted non-linear shape. "
                                "This is the closest model-based mirror of the CIF bins.",
                                fontsize=9, fontweight='bold', ha='left', va='top', wrap=True
                            )
                            
                            min_path = os.path.join(cox_results_dir, f'combined_effect_{combo_base}{fig_save_suffix}.png')
                            plt.tight_layout()
                            plt.savefig(min_path, dpi=300, bbox_inches="tight")
                            plt.show()
                            plt.close()
                            # print_v(f"  ✅ Saved minimal effect plot: {min_path}")
    
                            tb_var = 'tb_ratio_bwd_snr'
                            if tb_var in df_full_model.columns:
                                cph_tb = CoxPHSurvivalAnalysis(alpha=0.0)
                                cph_tb.fit(df_full_model[[tb_var]].to_numpy(), cox_surv_full)
                                tb_range = np.linspace(df_full_model[tb_var].min(), df_full_model[tb_var].max(), 200)
                                tb_df = pd.DataFrame({tb_var: tb_range})
                                tb_pred = cph_tb.predict(tb_df[[tb_var]].to_numpy())
                                tb_med = cph_tb.predict(df_full_model[[tb_var]].median().to_frame().T.to_numpy())[0]
                                tb_hr = np.exp(np.asarray(tb_pred, dtype=float) - tb_med)
        
                                fig_t, ax_t = plt.subplots(1, 1, figsize=(7, 4))
                                ax_t.plot(tb_range, tb_hr, linewidth=2, color='purple')
                                ax_t.axhline(y=1, color='red', linestyle='--', linewidth=1, alpha=0.5, label='No effect')
                                ax_t.set_xlabel(tb_var, fontsize=10, fontweight='bold')
                                ax_t.set_ylabel('Hazard Ratio', fontsize=10, fontweight='bold')
                                ax_t.set_title(f'TB-only mode: {tb_var}', fontsize=11, fontweight='bold')
                                ax_t.grid(alpha=0.3)
                                fig_t.text(
                                    0.01, -0.08,
                                    "TB ratio only: this is the raw, unconditional effect. "
                                    "Use it as a benchmark for direction and strength.",
                                    fontsize=9, fontweight='bold', ha='left', va='top', wrap=True
                                )

                                tb_path = os.path.join(cox_results_dir, f'tb_only_effect_{tb_var}{fig_save_suffix}.png')
                                plt.tight_layout()
                                plt.savefig(tb_path, dpi=300, bbox_inches="tight")
                                plt.show()
                                plt.close()
                                # print_v(f"  ✅ Saved TB-only effect plot: {tb_path}")
                                
                            # Build "units": each unit is [v] or [base, base_sq] so we add/remove linear+sq together
                            other_vars = [v for v in final_model_vars if v not in [combo_base, combo_sq]]
                            units = []
                            used = set()
                            for v in other_vars:
                                if v in used:
                                    continue
                                if v.endswith("_sq"):
                                    base = v[:-3]
                                    if base in other_vars:
                                        units.append([base, v])
                                        used.add(base)
                                        used.add(v)
                                    else:
                                        units.append([v])
                                        used.add(v)
                                else:
                                    sq = v + "_sq"
                                    if sq in other_vars:
                                        units.append([v, sq])
                                        used.add(v)
                                        used.add(sq)
                                    else:
                                        units.append([v])
                                        used.add(v)                            
                            inc_units = units[:5]
                            if inc_units:
                                fig_i, ax_i = plt.subplots(1, 1, figsize=(7, 4))
                                for unit in inc_units:
                                    vars_i = [combo_base, combo_sq] + unit
                                    cph_i = CoxPHSurvivalAnalysis(alpha=0.0)
                                    cph_i.fit(df_full_model[vars_i].to_numpy(), cox_surv_full)
                                    base_i = df_full_model[vars_i].median().to_dict()
                                    df_i = pd.DataFrame([base_i] * len(var_range))
                                    df_i[combo_base] = var_range
                                    df_i[combo_sq] = var_range ** 2
                                    pred_i = cph_i.predict(df_i[vars_i].to_numpy())
                                    med_i = cph_i.predict(df_full_model[vars_i].median().to_frame().T.to_numpy())[0]
                                    hr_i = np.exp(np.asarray(pred_i, dtype=float) - med_i)
                                    label_i = unit[0] if len(unit) == 1 else f"{unit[0]} (+ sq)"
                                    ax_i.plot(var_range, hr_i, linewidth=1.5, label=label_i)
                                ax_i.axhline(y=1, color='red', linestyle='--', linewidth=1, alpha=0.5, label='No effect')
                                ax_i.set_xlabel(combo_base, fontsize=10, fontweight='bold')
                                ax_i.set_ylabel('Hazard Ratio', fontsize=10, fontweight='bold')
                                ax_i.set_title(f'Incremental add (one covariate at a time)', fontsize=11, fontweight='bold')
                                ax_i.grid(alpha=0.3)
                                ax_i.legend(fontsize=8)
                                fig_i.text(
                                    0.01, -0.12,
                                    "Pool terms plus one covariate at a time. "
                                    "Curves show how each covariate shifts or flattens the hump.",
                                    fontsize=9, fontweight='bold', ha='left', va='top', wrap=True
                                )
                                inc_path = os.path.join(cox_results_dir, f'incremental_effect_{combo_base}{fig_save_suffix}.png')
                                plt.tight_layout()
                                plt.savefig(inc_path, dpi=300, bbox_inches="tight")
                                plt.show()
                                plt.close()
                                # print_v(f"  ✅ Saved incremental effect plot: {inc_path}")                  
    
                            # Remove-one test (full model minus one unit at a time; linear+sq removed together)
                            drop_units = units
                            if drop_units:
                                fig_r, ax_r = plt.subplots(1, 1, figsize=(7, 4))
                                # Plot full model curve as baseline
                                full_pred = cph_full.predict(baseline_df[final_model_vars])
                                full_med = cph_full.predict(df_full_model[final_model_vars].median().to_frame().T)[0]
                                full_hr = np.exp(np.asarray(full_pred, dtype=float) - full_med)
                                ax_r.plot(var_range, full_hr, linewidth=2, color='black', label='full model')
                                for unit in drop_units:
                                    vars_r = [v for v in final_model_vars if v not in unit]
                                    cph_r = CoxPHSurvivalAnalysis(alpha=0.0)
                                    cph_r.fit(df_full_model[vars_r].to_numpy(), cox_surv_full)                                
                                    base_r = df_full_model[vars_r].median().to_dict()
                                    df_r = pd.DataFrame([base_r] * len(var_range))
                                    if combo_base in vars_r:
                                        df_r[combo_base] = var_range
                                    if combo_sq in vars_r:
                                        df_r[combo_sq] = var_range ** 2
                                    pred_r = cph_r.predict(df_r[vars_r].to_numpy())
                                    med_r = cph_r.predict(df_full_model[vars_r].median().to_frame().T.to_numpy())[0]
                                    hr_r = np.exp(np.asarray(pred_r, dtype=float) - med_r)
                                    label_r = f"- {unit[0]}" if len(unit) == 1 else f"- {unit[0]} (+ sq)"
                                    ax_r.plot(var_range, hr_r, linewidth=1.5, label=label_r)
                                ax_r.axhline(y=1, color='red', linestyle='--', linewidth=1, alpha=0.5, label='No effect')
                                ax_r.set_xlabel(combo_base, fontsize=10, fontweight='bold')
                                ax_r.set_ylabel('Hazard Ratio', fontsize=10, fontweight='bold')
                                ax_r.set_title(f'Remove-one test (full model minus one)', fontsize=11, fontweight='bold')
                                ax_r.grid(alpha=0.3)
                                ax_r.legend(fontsize=8)
                                fig_r.text(
                                    0.01, -0.12,
                                    "Full model minus one covariate; black line is the full model. "
                                    "The biggest divergence identifies the suppressing covariate.",
                                    fontsize=9, fontweight='bold', ha='left', va='top', wrap=True
                                )
                                rm_path = os.path.join(cox_results_dir, f'remove_one_effect_{combo_base}{fig_save_suffix}.png')
                                plt.tight_layout()
                                plt.savefig(rm_path, dpi=300, bbox_inches="tight")
                                plt.show()
                                plt.close()
                                print_v(f"  ✅ Saved Remove-one plot: {rm_path}")                     
                    
                    
                        except Exception as e:
                            print_v(f"⚠️ Minimal model plot failed: {e}")

        
        time_stop(combo_timer, action=combo_act, nest=1)
    else:
        print_v("⏭️ Skipping 12.6A: Combined effect plot")

## === 📦 12.7. INTERACTION EFFECTS (3D PLOTS) ===


In [ ]:
with notebook_cell_logging('12_7'):
    # Reload pipeline_config to pick up any changes made
    # to configuration and other .py scripts
    if CELL12_7:
        reload_pipeline_config()
        # Ensure histogram helper exists (safe if cell 12.7A was skipped)
        if 'plot_var_distribution' not in globals():
            def plot_var_distribution(series, label, bins=30, color='steelblue'):
                """Plot a professional histogram with summary stats."""
                if series is None:
                    print_v(f"⚠️ No data for distribution: {label}")
                    return
                s = pd.to_numeric(series, errors='coerce').dropna()
                if s.empty:
                    print_v(f"⚠️ No non-null values for distribution: {label}")
                    return
                mean_val = s.mean()
                median_val = s.median()
                p05, p95 = np.percentile(s, [5, 95])
                fig, ax = plt.subplots(figsize=(8, 5))
                ax.hist(s, bins=bins, color=color, alpha=0.75, edgecolor='white', linewidth=0.8)
                ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.5, label='Mean')
                ax.axvline(median_val, color='black', linestyle=':', linewidth=1.5, label='Median')
                ax.set_title(f"Distribution: {label}", fontsize=12, fontweight='bold')
                ax.set_xlabel(label, fontsize=10, fontweight='bold')
                ax.set_ylabel('Count', fontsize=10, fontweight='bold')
                ax.grid(alpha=0.25)
                ax.legend(fontsize=8)
                stats_text = (
                    f"n={len(s):,}\n"
                    f"mean={mean_val:.3f}\n"
                    f"median={median_val:.3f}\n"
                    f"p05={p05:.3f}\n"
                    f"p95={p95:.3f}"
                )
                ax.text(
                    0.98, 0.98, stats_text,
                    transform=ax.transAxes,
                    ha='right', va='top',
                    fontsize=8,
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
                )
                plt.tight_layout()
                plt.show()
        # === 12.7. INTERACTION EFFECTS (3D PLOTS) ===
        print_v("\n📊 12.7: Interaction effects (3D plots)...")
        int_act = "12.7: Interaction effects"
        int_timer = time_start(int_act, nest=1)
        if cph_full is None:
            print_v("⚠️ Full model required for interaction effects. Skipping.")
        else:
            # Use centralized interaction config from pipeline_config.py
            from cox_plot_helpers import plot_var_distribution
            from matplotlib.colors import LogNorm
            int_cfg = INTERACTION_CONFIG if 'INTERACTION_CONFIG' in globals() else {}
            dist_cfg = DISTRIBUTION_PLOT_CONFIG if 'DISTRIBUTION_PLOT_CONFIG' in globals() else {}
            dist_bins = dist_cfg.get('bins', 'fd')
            dist_min_bins = dist_cfg.get('min_bins', 10)
            dist_max_bins = dist_cfg.get('max_bins', 60)
            dist_color = dist_cfg.get('color', 'steelblue')
            # Load coefficient map once (for interaction sign checks)
            coef_map = {}
            coeff_files = [
                "full_model_coefficients_stdz.csv",
                "full_model_coefficients.csv",
            ]
            coeff_df = None
            for coeff_name in coeff_files:
                coeff_path = os.path.join(cox_results_dir, coeff_name)
                if os.path.exists(coeff_path):
                    coeff_df = pd.read_csv(coeff_path)
                    break
            if coeff_df is not None and {"variable", "coefficient"}.issubset(coeff_df.columns):
                coef_map = dict(zip(coeff_df["variable"], coeff_df["coefficient"]))
            if not int_cfg.get('enabled', True):
                print_v("⏭️ Interaction plots disabled in INTERACTION_CONFIG")
            else:
                plots = int_cfg.get('plots', [])
                if not plots:
                    print_v("⚠️ No interaction plots configured (INTERACTION_CONFIG['plots'] is empty)")
                else:
                    prefix = STANDARDIZE_CONFIG.get('prefix', 'z_')
                    for spec in plots:
                        name = spec.get('name', 'interaction')
                        var_x = spec.get('var_x')
                        var_y = spec.get('var_y')
                        base_x = var_x
                        base_y = var_y
                        use_z = spec.get('use_z', False)
                        var_x_label = spec.get('var_x_label') or var_x
                        var_y_label = spec.get('var_y_label') or var_y
                        # Optional: use z-score versions of the variables
                        if use_z:
                            var_x = f"{prefix}{var_x}"
                            var_y = f"{prefix}{var_y}"
                        # Detect z-scored for labels/titles (either use_z or var names already have z_ prefix from run profile)
                        is_z_scale = use_z or (str(var_x).startswith(prefix) if var_x else False) or (str(var_y).startswith(prefix) if var_y else False)
                        var_x_label_display = f"{var_x_label} (z-scored)" if is_z_scale else var_x_label
                        var_y_label_display = f"{var_y_label} (z-scored)" if is_z_scale else var_y_label
                        # Per-plot distribution overrides (optional)
                        plot_bins = spec.get('dist_bins', dist_bins)
                        plot_min_bins = spec.get('dist_min_bins', dist_min_bins)
                        plot_max_bins = spec.get('dist_max_bins', dist_max_bins)
                        plot_color = spec.get('dist_color', dist_color)
                        if not var_x or not var_y:
                            print_v(f"⚠️ Skipping interaction plot '{name}'; missing var_x/var_y")
                            continue
                        # Ensure variables exist and are in the model
                        missing = [v for v in [var_x, var_y] if v not in df_full_model.columns]
                        missing_model = [v for v in [var_x, var_y] if v not in final_model_vars]
                        if missing:
                            print_v(f"⚠️ Missing variables in df_full_model: {missing}")
                        if missing_model:
                            print_v(f"⚠️ Variables not in full model: {missing_model}")
                        if missing or missing_model:
                            continue
                        # Use 5th to 95th percentiles to avoid extreme outliers
                        x_series = df_full_model[var_x].dropna()
                        y_series = df_full_model[var_y].dropna()
                        if len(x_series) == 0 or len(y_series) == 0:
                            print_v("⚠️ Insufficient data for interaction plot (all NaNs).")
                            continue
                        # x_min, x_max = np.percentile(x_series, [5, 95])
                        # y_min, y_max = np.percentile(y_series, [5, 95])
                        pct_range = int_cfg.get('percentile_range') or [5, 95]
                        print_v(f"  Interaction plot range: {pct_range[0]}th–{pct_range[1]}th percentile (from INTERACTION_CONFIG)")
                        x_min, x_max = np.percentile(x_series, pct_range)
                        y_min, y_max = np.percentile(y_series, pct_range)
                        # Diagnostic: data range and requested percentile range (remove or set verbose=False to hide)
                        print_v(f"  Data range {var_x}: full=[{float(x_series.min()):.4f}, {float(x_series.max()):.4f}]  {pct_range[0]}-{pct_range[1]}pct=[{x_min:.4f}, {x_max:.4f}] span={x_max-x_min:.6f}")
                        print_v(f"  Data range {var_y}: full=[{float(y_series.min()):.4f}, {float(y_series.max()):.4f}]  {pct_range[0]}-{pct_range[1]}pct=[{y_min:.4f}, {y_max:.4f}] span={y_max-y_min:.6f}")
                        # If data are winsorized, narrow bands (e.g. [90,100]) can give zero range -> empty plot
                        min_range_frac = 1e-6 * (float(x_series.max()) - float(x_series.min()) or 1.0)
                        if x_max - x_min < min_range_frac:
                            x_min, x_max = float(x_series.min()), float(x_series.max())
                            print_v(f"  ℹ️ Interaction plot: {var_x} had zero range in [{pct_range[0]},{pct_range[1]}]th pct (winsorized?); using full range.")
                        if y_max - y_min < min_range_frac:
                            y_min, y_max = float(y_series.min()), float(y_series.max())
                            print_v(f"  ℹ️ Interaction plot: {var_y} had zero range in [{pct_range[0]},{pct_range[1]}]th pct (winsorized?); using full range.")
                        # Use full range for prestige_mean due to extreme skew
                        if base_x and 'prestige_mean' in base_x:
                            x_min, x_max = float(x_series.min()), float(x_series.max())
                        if base_y and 'prestige_mean' in base_y:
                            y_min, y_max = float(y_series.min()), float(y_series.max())
                        log_z = int_cfg.get('log_z', False)
                        log_xy = int_cfg.get('log_xy', False)
                        use_log_xy = log_xy and (float(x_min) > 0) and (float(y_min) > 0)
                        # If interaction coefficient is negative, plot marginal effect curve
                        term_name = None
                        for term in int_cfg.get('terms', []):
                            left = term.get('left')
                            right = term.get('right')
                            if not left or not right:
                                continue
                            base_pairs = {
                                (base_x, base_y),
                                (base_y, base_x),
                            }
                            z_pairs = {
                                (f"{prefix}{base_x}", f"{prefix}{base_y}"),
                                (f"{prefix}{base_y}", f"{prefix}{base_x}"),
                            }
                            if (left, right) in base_pairs or (left, right) in z_pairs:
                                term_name = term.get('name')
                                if term.get('zscore', False):
                                    term_name = f"{prefix}{term_name}"
                                break
                        # Use exact coefficient keys (var_x, var_y match model columns, e.g. z_* when z-scored)
                        b1 = coef_map.get(var_x)
                        b3 = coef_map.get(term_name) if term_name else None
                        if b1 is not None and b3 is not None and b3 < 0:
                            y_vals_for_slope = np.linspace(y_min, y_max, 50)
                            slopes = b1 + b3 * y_vals_for_slope
                            figm, axm = plt.subplots(figsize=(7, 4))
                            axm.plot(y_vals_for_slope, slopes, marker='o', markersize=3)
                            axm.axhline(0, color='red', linestyle='--', linewidth=1)
                            # Highlight negative region (if any)
                            neg_mask = slopes < 0
                            if np.any(neg_mask):
                                axm.fill_between(
                                    y_vals_for_slope,
                                    slopes,
                                    0,
                                    where=neg_mask,
                                    color='red',
                                    alpha=0.15,
                                    label='Negative slope',
                                )
                                axm.scatter(
                                    y_vals_for_slope[neg_mask],
                                    slopes[neg_mask],
                                    color='red',
                                    s=20,
                                    zorder=3,
                                )
                            else:
                                axm.text(
                                    0.02, 0.08,
                                    "No negative region in this range",
                                    transform=axm.transAxes,
                                    fontsize=9,
                                    color='gray',
                                )
                            zero_cross = -b1 / b3 if b3 != 0 else None
                            if zero_cross is not None and y_min <= zero_cross <= y_max:
                                axm.axvline(zero_cross, color='red', linestyle=':', linewidth=1)
                            axm.set_title(f"Marginal effect of {var_x_label_display} across {var_y_label_display}")
                            axm.set_xlabel(var_y_label_display)
                            axm.set_ylabel(f"Marginal slope (b1 + b3*{var_y_label_display})")
                            axm.grid(True, alpha=0.3)
                            plt.tight_layout()
                            save_path_m = os.path.join(
                                cox_results_dir,
                                f"interaction_marginal_{name}_{var_x}_vs_{var_y}.png"
                            )
                            plt.savefig(save_path_m, dpi=300, bbox_inches='tight')
                            plt.show()
                            plt.close(figm)
                            print_v(f"  ✅ Saved marginal slope plot: {save_path_m}")
                        x_vals = np.linspace(x_min, x_max, 25)
                        y_vals = np.linspace(y_min, y_max, 25)
                        X, Y = np.meshgrid(x_vals, y_vals)
                        # Baseline: median for all other variables
                        baseline = df_full_model[final_model_vars].median().to_dict()
                        grid_df = pd.DataFrame([baseline] * X.size)
                        grid_df[var_x] = X.ravel()
                        grid_df[var_y] = Y.ravel()
                        # Predict relative hazard ratios
                        try:
                            preds = cph_full.predict(grid_df[final_model_vars])
                            preds = np.asarray(preds, dtype=float).ravel()
                            median_pred = cph_full.predict(
                                df_full_model[final_model_vars].median().to_frame().T
                            )[0]
                            Z = np.exp(preds - median_pred).reshape(X.shape)
                            if log_z:
                                Z = np.maximum(Z, 1e-10)  # LogNorm requires positive values
                            # 1D slice: x fixed at top (super top-block), HR vs y over tip-top of pool range
                            slice_x_at = int_cfg.get('slice_x_at')
                            slice_y_range = int_cfg.get('slice_y_range') or [90, 100]
                            slice_1d = None  # (x_fixed, y_vals_1d, hr_slice) for overlay on 3D and contour
                            if slice_x_at is not None:
                                x_fixed = float(x_series.max()) if slice_x_at == 'max' else float(np.percentile(x_series, slice_x_at))
                                y_slice_min, y_slice_max = np.percentile(y_series, slice_y_range)
                                y_vals_1d = np.linspace(y_slice_min, y_slice_max, 150)
                                grid_slice = pd.DataFrame([baseline] * len(y_vals_1d))
                                grid_slice[var_x] = x_fixed
                                grid_slice[var_y] = y_vals_1d
                                preds_slice = cph_full.predict(grid_slice[final_model_vars])
                                preds_slice = np.asarray(preds_slice, dtype=float).ravel()
                                hr_slice = np.exp(preds_slice - median_pred)
                                slice_1d = (x_fixed, y_vals_1d, hr_slice)
                                fig_slice, ax_slice = plt.subplots(figsize=(9, 5))
                                ax_slice.plot(y_vals_1d, hr_slice, color='steelblue', linewidth=2)
                                ax_slice.axhline(1.0, color='red', linestyle='--', linewidth=1.5)
                                ax_slice.set_xlabel(var_y_label_display, fontsize=11, fontweight='bold')
                                ax_slice.set_ylabel('Hazard Ratio', fontsize=11, fontweight='bold')
                                ax_slice.set_title(f"Hazard ratio vs {var_y_label_display} at top {var_x_label_display} (x={x_fixed:.3f})", fontsize=13, fontweight='bold')
                                ax_slice.grid(True, alpha=0.3)
                                plt.tight_layout()
                                save_slice = os.path.join(cox_results_dir, f"interaction_slice_x_at_top_{name}_{var_x}_vs_{var_y}.png")
                                plt.savefig(save_slice, dpi=300, bbox_inches='tight')
                                plt.show()
                                plt.close(fig_slice)
                                print_v(f"  ✅ Saved 1D slice (x fixed at top): {save_slice}")
                            # 3D surface plot
                            from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
                            fig = plt.figure(figsize=(14, 10))
                            ax = fig.add_subplot(111, projection='3d')
                            surf = ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.9)
                            # Add a visible HR=1.0 line on the surface
                            fig_tmp, ax_tmp = plt.subplots()
                            hr_contour = ax_tmp.contour(X, Y, Z, levels=[1.0])
                            for seg in hr_contour.allsegs[0]:
                                z_line = np.full(seg.shape[0], 1.0 + 1e-3)
                                ax.plot(
                                    seg[:, 0],
                                    seg[:, 1],
                                    z_line,
                                    color='red',
                                    linestyle='dashed',
                                    linewidth=2.5,
                                    zorder=10,
                                )
                            plt.close(fig_tmp)
                            if slice_1d is not None:
                                x_fix, y_1d, z_1d = slice_1d
                                ax.plot(np.full_like(y_1d, x_fix), y_1d, z_1d, color='red', linewidth=3, zorder=10, label='Slice at max x')
                            ax.set_xlabel(var_x_label_display, labelpad=10, fontsize=10, fontweight='bold')
                            ax.set_ylabel(var_y_label_display, labelpad=10, fontsize=10, fontweight='bold')
                            ax.set_zlabel('log(Hazard Ratio)' if log_z else 'Hazard Ratio', labelpad=0, fontsize=10, fontweight='bold')
                            # Title: avoid duplicate z-score note (axis labels already say "(z-scored)" when is_z_scale)
                            title = f"Interaction: {var_x_label_display} vs {var_y_label_display}"
                            if pct_range != [5, 95]:
                                title += f"\n(High range: {pct_range[0]}th–{pct_range[1]}th percentile)"
                            if log_z or use_log_xy:
                                title += "\n(log scale" + (" z" if log_z else "") + (" xy" if use_log_xy else "") + ")"
                            ax.set_title(title, fontsize=18, fontweight='bold', y=1.02, pad=0)
                            # Keep axes increasing and bring the origin to the front corner
                            ax.set_xlim(x_min, x_max)
                            ax.set_ylim(y_min, y_max)
                            if log_z:
                                ax.set_zscale('log')
                            if use_log_xy:
                                ax.set_xscale('log')
                                ax.set_yscale('log')
                            ax.view_init(elev=25, azim=-135)
                            cbar = fig.colorbar(surf, shrink=0.6, aspect=10, label='log(Hazard Ratio)' if log_z else 'Hazard Ratio')
                            cbar.ax.axhline(1.0, color='red', linestyle='dotted', linewidth=2)
                            plt.tight_layout()
                            zoom_suffix = "_highrange" if pct_range != [5, 95] else ""
                            if log_z:
                                zoom_suffix += "_logz"
                            if use_log_xy:
                                zoom_suffix += "_logxy"
                            save_path = os.path.join(
                                cox_results_dir,
                                f"interaction_3d_{name}_{var_x}_vs_{var_y}{zoom_suffix}.png"
                            )
                            plt.savefig(save_path, dpi=300, bbox_inches='tight')
                            plt.show()
                            plt.close()
                            print_v(f"  ✅ Saved interaction surface: {save_path}")
                            # 2D contour plot (optional quick view)
                            fig2, ax2 = plt.subplots(figsize=(10, 8))
                            _norm = LogNorm() if log_z else None
                            contour = ax2.contourf(X, Y, Z, levels=20, cmap='viridis', norm=_norm)
                            # Add HR=1.0 contour line
                            ax2.contour(X, Y, Z, levels=[1.0], colors='red', linestyles='dashed', linewidths=1.5, norm=_norm)
                            if slice_1d is not None:
                                x_fix, y_1d, z_1d = slice_1d
                                ax2.axvline(x_fix, color='red', linestyle='-', linewidth=2, alpha=0.9, label='Slice (x at top)')
                                ax2_inset = ax2.inset_axes([0.52, 0.08, 0.45, 0.35])
                                ax2_inset.plot(y_1d, z_1d, color='darkorange', linewidth=2)
                                ax2_inset.axhline(1.0, color='gray', linestyle='--', linewidth=1)
                                _inset_color = 'white'  # visible on dark color background
                                ax2_inset.set_xlabel(var_y_label_display, fontsize=9, color=_inset_color)
                                ax2_inset.set_ylabel('HR (slice)', fontsize=9, color=_inset_color)
                                ax2_inset.set_title('At max ' + var_x_label_display, fontsize=9, color=_inset_color)
                                ax2_inset.tick_params(axis='both', colors=_inset_color)
                                ax2_inset.grid(True, alpha=0.3)
                            ax2.set_xlabel(var_x_label_display, fontsize=10, fontweight='bold')
                            ax2.set_ylabel(var_y_label_display, fontsize=10, fontweight='bold')
                            if use_log_xy:
                                ax2.set_xscale('log')
                                ax2.set_yscale('log')
                            title2 = f"Interaction Contour: {var_x_label_display} vs {var_y_label_display}"
                            if pct_range != [5, 95]:
                                title2 += f"\n(High range: {pct_range[0]}th–{pct_range[1]}th percentile)"
                            if log_z or use_log_xy:
                                title2 += "\n(log scale" + (" z" if log_z else "") + (" xy" if use_log_xy else "") + ")"
                            ax2.set_title(title2, fontsize=16, fontweight='bold', y=1.1, pad =0)
                            cbar2 = fig2.colorbar(contour, label='log(Hazard Ratio)' if log_z else 'Hazard Ratio')
                            cbar2.ax.axhline(1.0, color='red', linestyle='dotted', linewidth=2)
                            plt.tight_layout()
                            save_path2 = os.path.join(
                                cox_results_dir,
                                f"interaction_contour_{name}_{var_x}_vs_{var_y}{zoom_suffix}.png"
                            )
                            plt.savefig(save_path2, dpi=300, bbox_inches='tight')
                            plt.show()
                            plt.close()
                            print_v(f"  ✅ Saved interaction contour: {save_path2}")
                            # === Variable distributions (follow-up plots) ===
                            # Use _display labels so each plot gets the correct title (X vs Y)
                            plot_var_distribution(
                                df_full_model[var_x],
                                var_x_label_display,
                                bins=plot_bins,
                                min_bins=plot_min_bins,
                                max_bins=plot_max_bins,
                                color=plot_color,
                                output_dir=cox_results_dir,
                                filename=f"distribution_{var_x}",
                                dpi=300,
                            )
                            plot_var_distribution(
                                df_full_model[var_y],
                                var_y_label_display,
                                bins=plot_bins,
                                min_bins=plot_min_bins,
                                max_bins=plot_max_bins,
                                color=plot_color,
                                output_dir=cox_results_dir,
                                filename=f"distribution_{var_y}",
                                dpi=300,
                            )
                            # === Correlation between interaction variables ===
                            corr_df = df_full_model[[var_x, var_y]].dropna()
                            if corr_df.empty:
                                print_v(f"⚠️ No overlapping data for correlation: {var_x} vs {var_y}")
                            else:
                                corr_val = corr_df[var_x].corr(corr_df[var_y])
                                corr_text = (
                                    f"Correlation ({var_x} vs {var_y}): {corr_val:.3f}\n"
                                    f"n = {len(corr_df):,}"
                                )
                                corr_text_display = corr_text.replace("\n", " | ")
                                print_v(f"🔗 {corr_text_display}")
                                cols = [var_x, var_y]
                                corr_table = corr_df[cols].corr()
                                display(corr_table)
                                # Save the correlation table as an image
                                fig_tbl, ax_tbl = plt.subplots(figsize=(8, 2.2))
                                ax_tbl.axis('off')
                                tbl = ax_tbl.table(
                                    cellText=corr_table.round(3).values,
                                    rowLabels=corr_table.index.tolist(),
                                    colLabels=corr_table.columns.tolist(),
                                    cellLoc='center',
                                    loc='center',
                                )
                                tbl.auto_set_font_size(False)
                                tbl.set_fontsize(9)
                                tbl.scale(1.1, 1.4)
                                tbl.auto_set_column_width(col=list(range(len(corr_table.columns) + 1)))
                                table_img_name = f"{var_x}_vs_{var_y}_correlation_table.png"
                                table_img_path = os.path.join(cox_results_dir, table_img_name)
                                plt.tight_layout()
                                plt.savefig(table_img_path, dpi=300, bbox_inches='tight')
                                plt.show()
                                plt.close(fig_tbl)
                                print_v(f"  ✅ Saved correlation table: {table_img_path}")
                                # Display full model coefficients (if present)
                                coeff_files = [
                                    "full_model_coefficients_stdz.csv",
                                    "full_model_coefficients.csv",
                                ]
                                for coeff_name in coeff_files:
                                    coeff_path = os.path.join(cox_results_dir, coeff_name)
                                    if os.path.exists(coeff_path):
                                        print_v(f"📄 Showing {coeff_name}")
                                        display(pd.read_csv(coeff_path))
                        except Exception as e:
                            print_v(f"⚠️ Interaction prediction failed: {e}")
        time_stop(int_timer, action=int_act, nest=1)
    else:
        print_v("⏭️ Skipping 12.7: Interaction effects")


## === 📦 12.8. MODEL-BASED PROMOTION CURVES (FIXED X/Y) ===


In [ ]:
# === 12.8. MODEL-BASED PROMOTION CURVES (FIXED X/Y) ===
with notebook_cell_logging('12_8'):
    if CELL12_8:
        print_v("\n📊 12.8: Model-based promotion curves (fixed X/Y)...")
        mb_act = "12.8: Model-based promotion curves"
        mb_timer = time_start(mb_act, nest=1)
    
        if cph_full is None:
            print_v("⚠️ Full model required for model-based curves. Skipping.")
        else:
            var_x = var_x
            var_y = var_y
            label_x = var_x_label
            label_y = var_y_label
    
            # Ensure variables exist and are in the model
            missing = [v for v in [var_x, var_y] if v not in df_full_model.columns]
            missing_model = [v for v in [var_x, var_y] if v not in final_model_vars]
            if missing:
                print_v(f"⚠️ Missing variables in df_full_model: {missing}")
            if missing_model:
                print_v(f"⚠️ Variables not in full model: {missing_model}")
    
            if not missing and not missing_model:
                x_series = df_full_model[var_x].dropna()
                y_series = df_full_model[var_y].dropna()
    
                if len(x_series) == 0 or len(y_series) == 0:
                    print_v("⚠️ Insufficient data for model-based curves (all NaNs).")
                else:
                    # Use low/med/high quantiles for fixed values
                    x_vals = np.quantile(x_series, [0.2, 0.5, 0.8])
                    y_vals = np.quantile(y_series, [0.2, 0.5, 0.8])
                    level_labels = ["Low", "Mid", "High"]
    
                    # Time grid for model-based survival curves (use durations, not absolute stop_time)
                    try:
                        full_durations = (df_full_model['stop_time'] - df_full_model['start_time']).astype(float)
                        t_max = np.nanpercentile(full_durations, 95)    
                    except Exception:
                        try:
                            full_durations = (df_pipeline_12_01_prepared['stop_time'] - df_pipeline_12_01_prepared['start_time']).astype(float)
                            t_max = np.nanpercentile(full_durations, 95)    
                        except Exception:                
                            t_max = np.nanpercentile(df_full_model['stop_time'], 95) 
                    t_max = max(float(t_max), 1.0)
                    time_grid = np.linspace(0, t_max, 200)
    
                    # Baseline: median for all other variables
                    baseline = df_full_model[final_model_vars].median().to_dict()
    
                    fig, ax = plt.subplots(figsize=(18, 10))
                    colors = plt.cm.Set1(np.linspace(0, 1, len(x_vals) * len(y_vals)))
    
                    idx = 0
                    for i, xv in enumerate(x_vals):
                        for j, yv in enumerate(y_vals):
                            row = baseline.copy()
                            row[var_x] = float(xv)
                            row[var_y] = float(yv)
                            X = pd.DataFrame([row])[final_model_vars]
    
                            try:
                                surv = cph_full.predict_survival_function(X)
    
                                # sksurv returns list of step functions; lifelines returns a DataFrame
                                if isinstance(surv, pd.DataFrame):
                                    surv_series = surv.iloc[:, 0]
                                    t_vals = surv_series.index.values.astype(float)
                                    s_vals = surv_series.values.astype(float)
                                    grid = time_grid[(time_grid >= t_vals.min()) & (time_grid <= t_vals.max())]
                                    if len(grid) < 2:
                                        grid = t_vals
                                    surv_interp = np.interp(grid, t_vals, s_vals)
                                else:
                                    sf = surv[0]
                                    t_vals = np.asarray(sf.x,dtype=float)
                                    grid = time_grid[(time_grid >= t_vals.min()) & (time_grid <= t_vals.max())]
                                    if len(grid) < 2:
                                        grid = t_vals
                                    surv_interp = sf(grid)
    
                                promo_prob = 1 - np.asarray(surv_interp, dtype=float)
                                label = (
                                    f"{var_x} {level_labels[i]} ({xv:.2f}) | "
                                    f"{var_y} {level_labels[j]} ({yv:.2f})"
                                )
                                ax.plot(grid, promo_prob, color=colors[idx], linewidth=2, label=label)
                            except Exception as e:
                                print_v(f"⚠️ Curve failed for X={xv:.3f}, Y={yv:.3f}: {e}")
    
                            idx += 1
    
                    ax.set_title(
                        "Model-based Promotion Probability (Cox)\nFixed X/Y values (1 - S(t))",
                        fontsize=12,
                        fontweight='bold'
                    )
                    ax.set_xlabel('Days from Promotion to CPT', fontsize=axis_sz)
                    ax.set_ylabel('Promotion Probability (1 - S(t))', fontsize=axis_sz)
                    ax.grid(True, alpha=0.3)
                    ax.legend(
                        title='Fixed values',
                        bbox_to_anchor=(1.05, 1),
                        loc='upper left',
                        fontsize=lgnd_sz,
                        title_fontsize=lgnd_sz
                    )
    
                    plt.tight_layout()
                    save_path = os.path.join(
                        cox_results_dir,
                        f"model_based_promo_curves_{var_x}_vs_{var_y}.png"
                    )
                    plt.savefig(save_path, dpi=300, bbox_inches='tight')
                    plt.show()
                    plt.close()
                    print_v(f"  ✅ Saved model-based curves: {save_path}")
    
        time_stop(mb_timer, action=mb_act, nest=1)
    else:
        print_v("⏭️ Skipping 12.8: Model-based promotion curves")


# **✅ PIPELINE SUMMARY**
## **Pipeline Stages & Output Files**
1. **PHASE 1: Data Loading & Basic Preparation**
   - `df_pipeline_01_raw` - Raw snapshot data
   - `df_pipeline_02_base` - Filtered base data (clean)
   - `df_pipeline_03_base` - Filtered base data with yg
2. **PHASE 2: OER Integration**
   - `df_pipeline_04a_basic_metrics` - Individual fwd/bwd metrics
   - `df_pipeline_05_pool_means` - Pool means/minus-means/sizes (fwd/bwd)
   - `df_pipeline_06_pool_ranks` - Pool ranks/pct/z-scores (fwd/bwd)
3. **PHASE 3: Variable Creation**
   - `df_pipeline_07_time_varying` - Time-varying variables added (prestige metrics when enabled — see Cell 7 / `PRESTIGE_CONFIG` in `pipeline_config.py`)
   - `df_pipeline_08_combined` - Combined time-varying + static variables
4. **PHASE 4: Advanced Filtering**
   - `df_pipeline_09_filtered` - Filtered dataset ready for analysis
5. **PHASE 5: Cox Analysis**
   - `df_pipeline_10_cox_ready` - Base Cox-ready dataset (time-varying survival intervals)
   - `df_pipeline_10_5_cox_zscored` - Standardized + interaction dataset for modeling/plots (optional; see `CELL10_5S`)
   - `df_pipeline_11_cox_analysis` - Analysis dataset with selected columns for plotting
   - `df_pipeline_12_01_prepared` - Prepared data for Cox regression models (CELL 12)
## **Key Columns for Immediate Analysis**
- **TB Ratios**: `TB_RATIO_FWD_*`, `TB_RATIO_BWD_*`
- **Pool Metrics**: `POOL_TB_RATIO_MEAN_*`, `POOL_MINUS_MEAN_*`, `POOL_TB_RATIO_RANK_PCT_*`, `POOL_TB_RATIO_ZPOOL_*`, `POOL_SIZE_*`
- **Demographics**: `yg`, `sex`, `age_cpt`
- **OER Metrics**: All cumulative + pool metrics from `assign_oer_to_snapshots_fast()` and pool steps
- **Time-Varying Variables**: `prestige_unit`, `prestige_sum`
## **Completed Work**
✅ **CELL 7**: Unit classification logic completed (with fallback if not loading from 502)
- Creates `prestige_unit` and `prestige_sum` if not already present
- Loads UIC hierarchy data and creates unit lists
- Maps prestige metrics to all snapshots
✅ **CELL 10**: Cox data preparation completed
- Creates time-varying survival intervals with `start_time`, `stop_time`, `event` columns
- Handles competing risks (promotion, attrition, censoring)
- Preserves all time-varying covariates
✅ **CELL 11**: Cox analysis and plotting completed
- Enhanced plotting utilities ported from 513 (font scaling, reference lines, subtitles)
- Flexible configuration-based plotting system (`PLOT_CONFIG` in `pipeline_config.py`)
- Kaplan-Meier and competing risks plots; optional CIF bar panels and metadata cards
- Paper-friendly defaults with optional legacy toggles (`show_plot_titles`, `show_plot_subtitles`, legend placement, CIF bar titles/x-labels, etc.)
- OER filtering support
- Division name plotting support
✅ **CELL 12**: Cox regression models
- **12.1**: Data preparation (✅ Complete)
- **12.2**: Static model fitting (✅ Complete)
- **12.3**: Full model with time-varying covariates (✅ Complete - uses interval-level data with dynamic covariates)
- **12.4**: Model comparison and signal ratios (✅ Complete - 4-panel visualization comparing static vs full models)
- **12.5**: Competing risks analysis (✅ Complete - separate models for promotion and attrition with coefficient comparison)
- **12.6**: Partial effects plots (✅ Complete - hazard ratio curves for continuous variables, signal ratios for categorical)
- **12.6A**: Combined linear + quadratic panels (✅ Complete - minimal / full / TB-only / incremental / remove-one views for `base`+`_sq` pairs)
- **12.7**: Interaction effects (✅ Complete - 3D HR surfaces, contour + inset slice, marginal effect plots, distributions/correlation; driven by `INTERACTION_CONFIG`)
- **12.8**: Model-based promotion curves (✅ Complete - nine Cox curves at 20th/50th/80th percentiles of `var_x`×`var_y`, other covariates at medians; **uses `var_x`/`var_y` left in memory after 12.7**, typically the last interaction spec)
## **Next Steps**
1. ✅ ~~Complete CELL 10 (Cox data preparation)~~ - **COMPLETED**
2. ✅ ~~Port Cox analysis cells from 513 (CELL 11)~~ - **COMPLETED** (plotting with enhanced utilities)
3. ✅ ~~Add unit classification definitions in CELL 7~~ - **COMPLETED** (with fallback logic)
4. ✅ ~~Implement CELL 12.3 (Full model with time-varying covariates)~~ - **COMPLETED**
5. ✅ ~~Implement CELL 12.4 (Model comparison and signal ratios)~~ - **COMPLETED**
6. ✅ ~~Implement CELL 12.5 (Competing risks analysis)~~ - **COMPLETED**
7. ✅ ~~Implement CELL 12.6 (Partial effects plots)~~ - **COMPLETED**
8. ✅ ~~Implement CELL 12.7 (Interaction effects: 3D surfaces, contours, slices)~~ - **COMPLETED**
9. ✅ ~~Implement CELL 12.8 (Model-based promotion curves at fixed X/Y)~~ - **COMPLETED**
10. 🔄 **Test pipeline with sample data** - Validate end-to-end workflow
11. 🔄 **Refine filtering parameters in CELL 9** - Optimize based on analysis results
12. 🔄 **Integration testing** - Ensure upstream prep → 520 pipeline works seamlessly
13. 🔄 **Documentation** - Keep overview / summary cells aligned with `pipeline_config.py` and new cells


In [ ]:
time_stop(burrito_timer,action=burrito_act,nest=0)
